# Custom Analysis — Modelo de Evaluación de Edge (custom)

Notebook construido por secciones. Cada sección se agrega en orden, una vez validada la anterior.
Datos leídos directamente de la base de producción de la cuenta `flight_account_001_xauusd` — **solo lectura**.

## Sección 1: Extracción de datos — `unified_department` y `analysis_layer`

In [113]:
import sys
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from IPython.display import display

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))  # para poder importar core/*

pd.set_option('display.float_format', lambda x: f'{x:.4f}') # evitar notación cientifica y mostrar 2 decimales
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)   # mostrar todas las columnas
pd.set_option('display.width', 0)           # dejar que use todo el ancho disponible
pd.set_option('display.max_colwidth', None) # Quitar el límite de ancho de las columnas
pd.set_option('display.expand_frame_repr', False) # Para que no "envuelva" la tabla y se mantenga en una sola fila larga

In [114]:
# Conexión de solo lectura a la DB de la cuenta 001 (xauusd)
DB_URL = "sqlite:///../.data/flight_account_001_xauusd.db"
engine = create_engine(DB_URL, connect_args={'timeout': 15})

# tactical_audit paso de 1:1 a 1:many con unified_department (ago-2026, ver CLAUDE.md
# "Antes de tocar codigo"). Esta DB real puede no estar migrada todavia (antes de correr
# tools/migrate_tactical_audit_1n.py) -- detectar en runtime en vez de asumir.
from sqlalchemy import inspect as _sa_inspect
TACTICAL_HAS_TRADE_ID = 'trade_id' in {c['name'] for c in _sa_inspect(engine).get_columns('tactical_audit')}

def _load_tactical_slice(conn, columns):
    """Lee columnas de tactical_audit uniendo por el id del analisis, sea cual sea el
    esquema vigente. Post-migracion (TACTICAL_HAS_TRADE_ID): tactical_audit es 1:many,
    se une por trade_id y se toma la ejecucion mas reciente por analisis (no infla el N
    si ya hay 2+ ejecuciones). Pre-migracion: id todavia es el id compartido 1:1 legacy."""
    if TACTICAL_HAS_TRADE_ID:
        cols_sql = ", ".join(dict.fromkeys(["id", "trade_id", "created_at"] + list(columns)))
        df = pd.read_sql(text(f"SELECT {cols_sql} FROM tactical_audit;"), conn)
        return (
            df.sort_values('created_at')
              .drop_duplicates('trade_id', keep='last')
              .drop(columns=['id', 'created_at'])
              .rename(columns={'trade_id': 'id'})
        )
    cols_sql = ", ".join(dict.fromkeys(["id"] + list(columns)))
    return pd.read_sql(text(f"SELECT {cols_sql} FROM tactical_audit;"), conn)

In [115]:
with engine.connect() as conn:
    unified_df = pd.read_sql(text("SELECT * FROM unified_department;"), conn, parse_dates=['created_at', 'updated_at'])
    layer_df = pd.read_sql(text("SELECT * FROM analysis_layer;"), conn)
    efficiency_df = pd.read_sql(text("SELECT * FROM efficiency_audit;"), conn, parse_dates=['created_at', 'updated_at', 'resolution_time'])

print(f"unified_department: {unified_df.shape[0]} filas, {unified_df.shape[1]} columnas")
print(f"analysis_layer: {layer_df.shape[0]} filas, {layer_df.shape[1]} columnas")
print(f"efficiency_audit: {efficiency_df.shape[0]} filas, {efficiency_df.shape[1]} columnas")

unified_department: 70 filas, 23 columnas
analysis_layer: 350 filas, 8 columnas
efficiency_audit: 70 filas, 15 columnas


### Composición de `unified_department`

In [116]:
unified_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id                       70 non-null     object        
 1   state                    70 non-null     object        
 2   asset                    70 non-null     object        
 3   created_at               70 non-null     datetime64[ns]
 4   updated_at               70 non-null     datetime64[ns]
 5   market_bias              70 non-null     object        
 6   calc_edge                70 non-null     float64       
 7   edge_description         70 non-null     object        
 8   p4_hierarchy             70 non-null     object        
 9   p1_timeframe             70 non-null     object        
 10  p1_type                  70 non-null     object        
 11  nodes_l1                 70 non-null     int64         
 12  nodes_l2                 70 non-null  

In [117]:
display(unified_df.head())

,id,state,asset,created_at,updated_at,market_bias,calc_edge,edge_description,p4_hierarchy,p1_timeframe,p1_type,nodes_l1,nodes_l2,tactical_classification,long_prob,short_prob,no_trade_prob,efficiency_page_id,tactical_page_id,is_backdated,edge_validation_price,structural_invalidation,mark_price
0,2834cbeb-b65f-4cad-97ca-1807e3193b23,FAILED,XAUUSDT.P,2026-05-18 07:00:00.000000,2026-08-10 12:03:05.458907,Choppy / Neutral,0.2250,Posibiliad alcista,Psych Level,4H,Max_iteration,7,9,Range_Rotation,0.1638,0.1037,0.7325,None,None,0,4584.0000,4520.0000,4568.1200
1,1998959d-7ea9-4d3e-8f82-1ec007021143,FAILED,XAUUSDT.P,2026-05-19 06:55:18.918597,2026-08-10 18:50:59.987775,Choppy / Neutral,0.0000,En TF 4H el precio se encuentra en un rango.,"Hard_Level (Daily,Weekly,Monthly)",4H,1st_iteration,7,8,N/A,0.1000,0.1000,0.8000,None,None,0,NaN,NaN,4540.2500
2,0a68d499-0175-4048-acc4-8412edbb51c4,FAILED,XAUUSDT.P,2026-05-20 06:44:53.658023,2026-08-10 18:53:38.345867,Choppy / Neutral,0.0750,"En 4H TF el precio tiene un impulso bajista, pero puede que estemos en una corrección o pullback. El edge se mantiene mientras llegue a 4520 y lo rompa, se invalida si el precio baja de 4455","Hard_Level (Daily,Weekly,Monthly)",4H,Double_fractal,7,2,Reversal_Pressure,0.1196,0.1029,0.7775,None,None,0,4520.0000,4455.0000,4496.2450
3,b9c076bf-85fc-450f-a7d2-48917c1652f4,FAILED,XAUUSDT.P,2026-05-22 06:55:15.135212,2026-07-18 13:07:08.725290,Choppy / Neutral,-0.0250,El mercado esta lateralizando con tendencia o posibilidad alcista. El edge se cumple si el precio soporta los 4491 y rompe los 4570.,Soft_Level (1H/4H),30M,1st_iteration,3,6,Reversal_Pressure,0.1012,0.1063,0.7925,None,None,0,NaN,NaN,NaN
4,35be0fc7-b3ba-42d7-9571-63ef2e8100af,FAILED,XAUUSDT.P,2026-05-25 06:51:45.509490,2026-08-10 19:03:11.609285,Choppy / Neutral,0.1750,En 4H Timeframe estamos lateralizando por hay una posibilidad de romper el rango. El edge se cumple mientras el precio no baje de 4523 y rompa hacia 4620.,"Hard_Level (Daily,Weekly,Monthly)",4H,Max_iteration,7,7,Continuation_Pressure,0.1483,0.1042,0.7475,None,None,0,4620.0000,4523.0000,4564.5100


### Composición de `analysis_layer`

In [118]:
layer_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          350 non-null    object
 1   trade_id    350 non-null    object
 2   department  350 non-null    object
 3   layer_name  350 non-null    object
 4   direction   350 non-null    object
 5   strength    350 non-null    object
 6   score       350 non-null    int64 
 7   thesis      350 non-null    object
dtypes: int64(1), object(7)
memory usage: 22.0+ KB


In [119]:
display(layer_df.head())

,id,trade_id,department,layer_name,direction,strength,score,thesis
0,3367827f-d9b0-4270-92f3-a8eaff92a6b0,2834cbeb-b65f-4cad-97ca-1807e3193b23,EFFICIENCY,P0,Long,Weak,0,"Hay posibilidad de una corrección.\n12T-BR: El precio llego de nuevo al nivel 0.382, tras entrar en ese nivel es posible una ligera corrección al alza para agarrar liquidez y romperlo o no. \n4T-BL: El precio llego al nivel 1 como soporte, tras estar en ese nivel, es posible que el precio busque liquidez por encima, sin embargo, la bajada fue con fuerza, habrá que tener cuidado."
1,327e1756-35fb-4609-9974-e9cce43f8824,2834cbeb-b65f-4cad-97ca-1807e3193b23,EFFICIENCY,P2,Short,Weak,0,EMA: price < 20EMA < 200EMA\nRSI:price en tendencia ligeramente alcista tras rebote en nivel 30\nADX: en tendencia bajista la uerza tras caida brutal del precio
2,fdc41ebb-50a2-4de9-98d3-c72e5dc85a4c,2834cbeb-b65f-4cad-97ca-1807e3193b23,EFFICIENCY,P3,Long,Weak,0,"El precio cayó con bastante fuerza, sin embargo, tras caer tuvo un agarron de liquidez, puede ser que suba un poco."
3,5b2f2dbd-e385-4f24-9184-0fba7b103ed1,2834cbeb-b65f-4cad-97ca-1807e3193b23,TACTICAL,P4,Long,Mid,1,El precio rechazó los 4500 puede buscar niveles superiores
4,c0ffd50d-4557-4cee-8371-841dfd3be9d7,2834cbeb-b65f-4cad-97ca-1807e3193b23,TACTICAL,P1,Long,Mid,1,IMAGEN PENDIENTE DE IMPLEMENTAR


### Composición de `efficiency_audit`

In [120]:
efficiency_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   id                        70 non-null     object        
 1   bias_a                    70 non-null     object        
 2   resolution_type           70 non-null     object        
 3   real_bias_b               70 non-null     object        
 4   structural_resolution     70 non-null     object        
 5   failure_reason            70 non-null     object        
 6   specific_bias_compliance  70 non-null     object        
 7   false_regime_rate         70 non-null     object        
 8   resolution_time           70 non-null     datetime64[ns]
 9   lesson_learned            61 non-null     object        
 10  created_at                70 non-null     datetime64[ns]
 11  updated_at                70 non-null     datetime64[ns]
 12  efficiency_timeframe    

In [121]:
display(efficiency_df.head())

,id,bias_a,resolution_type,real_bias_b,structural_resolution,failure_reason,specific_bias_compliance,false_regime_rate,resolution_time,lesson_learned,created_at,updated_at,efficiency_timeframe,structural_mae,structural_mfe
0,2834cbeb-b65f-4cad-97ca-1807e3193b23,Choppy-Bullish Range Rotation,Confirmed (A equal to B),Choppy-Bullish Range Rotation,Confirmed + expansión significativa,N/A,Valid,True Positive,2026-06-18 09:05:23.445403,"Acá el edge estaba choppy, fue más por intuición el resultado del analisis.",2026-05-18 07:00:00.000000,2026-08-10 18:14:13.326534,1H,4533.8800,4591.1400
1,1998959d-7ea9-4d3e-8f82-1ec007021143,No_Bias(Choppy),Confirmed (A equal to B),No_Bias(Choppy),N/A,N/A,Valid,True Negative,2026-06-18 09:22:45.685313,None,2026-05-19 06:55:18.949975,2026-08-10 18:50:59.988321,1H,NaN,NaN
2,0a68d499-0175-4048-acc4-8412edbb51c4,CHOCH,Confirmed (A equal to B),CHOCH,Confirmed + expansión significativa,N/A,Valid,True Positive,2026-06-21 06:48:35.968186,"El analisis acá estaba cruzado, había posibilidad de que el rango de P0 fuera bajista por la estructura del precio, pero también habia suceptibilidad de que el precio siguiera un pullback para agarrar liquidez y posteriormente seguir bajando.",2026-05-20 06:44:53.674679,2026-08-11 00:55:01.852916,1H,4468.0600,4570.0000
3,b9c076bf-85fc-450f-a7d2-48917c1652f4,No_Bias(Choppy),Confirmed (A equal to B),No_Bias(Choppy),Confirmed pero mínima,N/A,Valid,True Negative,2026-06-21 07:47:00.664509,"Aqui no hay edge, solo estamos en rango.",2026-05-22 06:55:15.148641,2026-06-21 07:48:11.538823,1H,NaN,NaN
4,35be0fc7-b3ba-42d7-9571-63ef2e8100af,Validated Range Expansion,Invalidated (B not equal to A),BOS,N/A,Reversal,Invalid,False Positive,2026-05-26 08:54:16.798760,"El precio estaba en un rango e intenté hacer el trade en el rango superior cuando la tendencia era mayoritariamente bajista, en realidad era más probable el short.",2026-05-25 06:51:45.558733,2026-08-11 01:05:02.749608,1H,4509.0000,4584.0000


## Sección 2: Edge estructural (Efficiency) — Win Rate por Market Bias y por Probabilidad

Objetivo: determinar si conviene operar solo cuando el `market_bias` es direccional (Bullish/Bearish) y descartar Choppy, o si las probabilidades del modelo (`long_prob`/`short_prob`) también predicen bien el acierto dentro de Choppy.

Definición de "win rate" acá: **tasa de acierto estructural** — proporción de setups donde `specific_bias_compliance == 'Valid'` (el sesgo A predicho coincidió con la resolución real B), sobre el total de setups ya resueltos (`Valid` + `Invalid`; se excluyen los 3 sin resolución todavía, `specific_bias_compliance` vacío).

In [122]:
# Base de auditoría: efficiency_audit + market_bias/probabilidades de unified_department
audit_df = efficiency_df.merge(
    unified_df[['id', 'market_bias', 'calc_edge', 'long_prob', 'short_prob', 'no_trade_prob']],
    on='id', how='left'
)

# Solo setups ya resueltos (excluye los 3 sin specific_bias_compliance todavía)
audit_df = audit_df[audit_df['specific_bias_compliance'].isin(['Valid', 'Invalid'])].copy()
audit_df['is_valid'] = (audit_df['specific_bias_compliance'] == 'Valid').astype(int)

# Probabilidad dominante: la del lado que el modelo predijo (calc_edge >= 0 -> Long, si no -> Short)
audit_df['theoretical_dir'] = np.where(audit_df['calc_edge'] >= 0, 'Long', 'Short')
audit_df['primary_prob'] = np.where(audit_df['theoretical_dir'] == 'Long', audit_df['long_prob'], audit_df['short_prob'])

print(f"Setups auditados (Valid+Invalid): {len(audit_df)} de {len(efficiency_df)} totales")
display(audit_df[['id', 'market_bias', 'specific_bias_compliance', 'is_valid', 'calc_edge', 'primary_prob']].head())

Setups auditados (Valid+Invalid): 70 de 70 totales


,id,market_bias,specific_bias_compliance,is_valid,calc_edge,primary_prob
0,2834cbeb-b65f-4cad-97ca-1807e3193b23,Choppy / Neutral,Valid,1,0.2250,0.1638
1,1998959d-7ea9-4d3e-8f82-1ec007021143,Choppy / Neutral,Valid,1,0.0000,0.1000
2,0a68d499-0175-4048-acc4-8412edbb51c4,Choppy / Neutral,Valid,1,0.0750,0.1196
3,b9c076bf-85fc-450f-a7d2-48917c1652f4,Choppy / Neutral,Valid,1,-0.0250,0.1063
4,35be0fc7-b3ba-42d7-9571-63ef2e8100af,Choppy / Neutral,Invalid,0,0.1750,0.1483


### Win rate por `market_bias` (Bullish / Bearish / Choppy)

In [123]:
win_rate_by_bias = audit_df.groupby('market_bias')['is_valid'].agg(
    N_Setups='count',
    Win_Rate='mean'
).sort_values('Win_Rate', ascending=False)
win_rate_by_bias['Win_Rate_%'] = (win_rate_by_bias['Win_Rate'] * 100).round(2)

display(win_rate_by_bias[['N_Setups', 'Win_Rate_%']])

,N_Setups,Win_Rate_%
market_bias,,
Bullish,13,76.9200
Bearish,25,76.0000
Choppy / Neutral,32,59.3800


### Win rate por rango de `primary_prob` (probabilidad dominante del modelo)

Terciles (bajo / medio / alto), independiente del `market_bias`.

In [124]:
prob_bins = pd.qcut(audit_df['primary_prob'], q=3, duplicates='drop')
win_rate_by_prob = audit_df.groupby(prob_bins, observed=False)['is_valid'].agg(
    N_Setups='count',
    Win_Rate='mean'
)
win_rate_by_prob['Win_Rate_%'] = (win_rate_by_prob['Win_Rate'] * 100).round(2)

display(win_rate_by_prob[['N_Setups', 'Win_Rate_%']])

,N_Setups,Win_Rate_%
primary_prob,,
"(0.099, 0.156]",25,60.0000
"(0.156, 0.215]",22,63.6400
"(0.215, 0.408]",23,82.6100


### Cruce: rango de probabilidad dentro de Choppy vs No-Choppy

Responde la pregunta central: dentro de Choppy, ¿una probabilidad alta sigue prediciendo acierto, o el ruido de "no hay bias claro" mata la señal de la probabilidad también?

In [125]:
audit_df['bias_group'] = np.where(audit_df['market_bias'] == 'Choppy / Neutral', 'Choppy', 'Direccional (Bull/Bear)')

win_rate_cross = audit_df.groupby(['bias_group', prob_bins], observed=False)['is_valid'].agg(
    N_Setups='count',
    Win_Rate='mean'
)
win_rate_cross['Win_Rate_%'] = (win_rate_cross['Win_Rate'] * 100).round(2)

display(win_rate_cross[['N_Setups', 'Win_Rate_%']])

N_Setups  Win_Rate_%
bias_group              primary_prob                        
Choppy                  (0.099, 0.156]        25     60.0000
                        (0.156, 0.215]         7     57.1400
                        (0.215, 0.408]         0         NaN
Direccional (Bull/Bear) (0.099, 0.156]         0         NaN
                        (0.156, 0.215]        15     66.6700
                        (0.215, 0.408]        23     82.6100

### Lectura de los resultados (n=62 setups resueltos)

**Win rate por market_bias:** Bullish 80.0% (n=10), Bearish 78.26% (n=23), Choppy 58.62% (n=29).

**Win rate por tercil de primary_prob:** bajo (0.099–0.156] 58.33% (n=24), medio (0.156–0.212] 64.71% (n=17), alto (0.212–0.408] 85.71% (n=21).

**Cruce bias × probabilidad — el hallazgo clave:** en esta muestra, `primary_prob` y `market_bias` **no son señales independientes**. El tercil alto de probabilidad (0.212–0.408) tiene **0 setups Choppy** — el 100% son Bullish/Bearish. El tercil bajo (0.099–0.156) tiene **0 setups direccionales** — el 100% son Choppy. Solo el tercil medio mezcla ambos (5 Choppy vs 12 Direccional).

**Conclusión:** no se puede responder "¿conviene operar en Choppy si la probabilidad es alta?" porque ese escenario **no existe en los datos históricos** — cuando el modelo calcula una probabilidad alta, nunca clasifica el régimen como Choppy. En la práctica, filtrar por `market_bias != Choppy` y filtrar por probabilidad alta son, hoy, casi el mismo filtro. La única comparación con datos de ambos lados es el tercil medio: ahí Direccional (66.67%, n=12) sigue por encima de Choppy (60.0%, n=5), aunque con muestras muy chicas para ser concluyente. El descarte de Choppy parece la palanca más simple y con más soporte de muestra (n=29 vs los 5 del cruce medio).

## Sección 3: Condiciones de cada `structural_resolution` + acierto por `bias_a`

Dos preguntas:
1. ¿Bajo qué `market_bias` y qué rango de `primary_prob` aparece cada tipo de `structural_resolution`?
2. ¿Qué `bias_a` (el sesgo predicho por el modelo) tiene más aciertos (`Valid`) y cuál tiene más fallos (`Invalid`)?

Usa `audit_df` (ya filtrado a los 62 setups resueltos, Sección 2) — `structural_resolution` ya viene incluido ahí desde `efficiency_df`.

### `structural_resolution` por `market_bias`

In [126]:
resolution_by_bias = pd.crosstab(audit_df['structural_resolution'], audit_df['market_bias'], margins=True, margins_name='Total')
display(resolution_by_bias)

market_bias,Bearish,Bullish,Choppy / Neutral,Total
structural_resolution,,,,
Confirmed + expansión significativa,12,4,9,25
Confirmed pero inmediatamente revertido,4,3,2,9
Confirmed pero mínima,4,3,5,12
N/A,5,3,16,24
Total,25,13,32,70


### `structural_resolution` por rango de `primary_prob`

In [127]:
resolution_by_prob = pd.crosstab(audit_df['structural_resolution'], prob_bins, margins=True, margins_name='Total')
display(resolution_by_prob)

primary_prob,"(0.099, 0.156]","(0.156, 0.215]","(0.215, 0.408]",Total
structural_resolution,,,,
Confirmed + expansión significativa,5,9,11,25
Confirmed pero inmediatamente revertido,2,3,4,9
Confirmed pero mínima,5,2,5,12
N/A,13,8,3,24
Total,25,22,23,70


### Acierto por `bias_a` — cuál sesgo predicho acierta más y cuál falla más

In [128]:
bias_a_stats = audit_df.groupby('bias_a')['is_valid'].agg(
    N_Total='count',
    N_Valid='sum',
    Win_Rate='mean'
)
bias_a_stats['N_Invalid'] = bias_a_stats['N_Total'] - bias_a_stats['N_Valid']
bias_a_stats['Win_Rate_%'] = (bias_a_stats['Win_Rate'] * 100).round(2)

print("--- Ordenado por N_Valid (más aciertos) ---")
display(bias_a_stats[['N_Total', 'N_Valid', 'N_Invalid', 'Win_Rate_%']].sort_values('N_Valid', ascending=False))

print("\n--- Ordenado por N_Invalid (más fallos) ---")
display(bias_a_stats[['N_Total', 'N_Valid', 'N_Invalid', 'Win_Rate_%']].sort_values('N_Invalid', ascending=False))

--- Ordenado por N_Valid (más aciertos) ---


,N_Total,N_Valid,N_Invalid,Win_Rate_%
bias_a,,,,
BOS,29,22,7,75.8600
No_Bias(Choppy),8,7,1,87.5000
Trend Reversal,14,6,8,42.8600
CHOCH,10,5,5,50.0000
Choppy-Bearish Range Rotation,4,4,0,100.0000
Choppy-Bullish Range Rotation,2,2,0,100.0000
Validated Range Expansion,3,2,1,66.6700



--- Ordenado por N_Invalid (más fallos) ---


,N_Total,N_Valid,N_Invalid,Win_Rate_%
bias_a,,,,
Trend Reversal,14,6,8,42.8600
BOS,29,22,7,75.8600
CHOCH,10,5,5,50.0000
Validated Range Expansion,3,2,1,66.6700
No_Bias(Choppy),8,7,1,87.5000
Choppy-Bearish Range Rotation,4,4,0,100.0000
Choppy-Bullish Range Rotation,2,2,0,100.0000


### Lectura de los resultados (n=62)

**`structural_resolution` por `market_bias`** (normalizando por total de cada bias: Bearish=23, Bullish=10, Choppy=29):
- **Bearish** es el que más resuelve limpio: 56.5% termina en "Confirmed + expansión significativa" (13/23), solo 17.4% en "N/A".
- **Choppy** es al revés: casi la mitad (48.3%, 14/29) resuelve en "N/A" — no llega a un patrón claro de resolución.
- **Bullish** (n=10, muestra chica) reparte parejo entre las 4 categorías, sin resolución dominante.

**`structural_resolution` por `primary_prob`:** el patrón es monótono — "N/A" cae de 50% (tercil bajo, 12/24) a 9.5% (tercil alto, 2/21), y "Confirmed + expansión significativa" sube de 25% a 47.6%. Coherente con que `primary_prob` y `market_bias` están correlacionados (Sección 2).

**Acierto por `bias_a`:**
- **Más confiable con muestra real**: `BOS` — 83.33% win rate, n=24 (la categoría más común y la de mejor desempeño a la vez).
- **Menos confiable con muestra real**: `Trend Reversal` — 41.67% win rate, n=12, 7 de 12 Invalid (el mayor conteo de fallos real). `CHOCH` también débil: 44.44%, n=9.
- `Choppy-Bearish/Bullish Range Rotation` muestran 100% pero con n=3 y n=2 — no representativo, no tratar como señal confiable todavía.
- `No_Bias(Choppy)` acierta 77.78% (n=9) — cuando el modelo predice explícitamente "no hay bias", suele tener razón.

**Implicación:** `BOS` es el tipo de sesgo más numeroso y el más confiable — la base más sólida del edge estructural. `Trend Reversal` y `CHOCH` son los dos puntos débiles con muestra suficiente para tomarlos en serio; son candidatos a revisar (¿vale la pena seguir operándolos igual, o requieren un filtro adicional?).

## Sección 4: `false_regime_rate` — matriz de confusión del detector de régimen

`false_regime_rate` no es un campo libre: lo calcula el propio código (`cli/schemas/audit_efficiency.py:60-79`) comparando `bias_a` (predicción) vs `real_bias_b` (realidad), tratando `No_Bias(Choppy)` como la clase negativa (no operar) y cualquier otro bias como la clase positiva (sí operar).

| valor | significado | qué compara |
|---|---|---|
| True Positive | predijo un bias específico y acertó exacto | `bias_a == real_bias_b`, ninguno choppy |
| False Positive | predijo un bias específico pero era el equivocado | `bias_a != real_bias_b`, ninguno choppy |
| True Negative | predijo choppy y era choppy | acertó en "no operar" |
| False Signal | predijo un bias específico, pero la realidad fue choppy | falsa alarma — vio señal donde no había |
| False Negative | predijo choppy, pero la realidad tuvo un bias real | oportunidad perdida — se quedó afuera de un movimiento real |

In [129]:
# Distribución real (n=62 setups resueltos)
regime_counts = audit_df['false_regime_rate'].value_counts().rename('N_Setups').to_frame()
regime_counts['%'] = (regime_counts['N_Setups'] / regime_counts['N_Setups'].sum() * 100).round(2)
display(regime_counts)

,N_Setups,%
false_regime_rate,,
True Positive,41,58.5700
False Positive,18,25.7100
True Negative,7,10.0000
False Signal,3,4.2900
False Negative,1,1.4300


### Precision / Recall del detector de régimen

Tratando "predijo un bias específico" (TP+FP) como positivo y "predijo choppy" (TN+FN) como negativo.

In [130]:
tp = (audit_df['false_regime_rate'] == 'True Positive').sum() # predijo un bias específico y acertó exacto
fp = (audit_df['false_regime_rate'] == 'False Positive').sum() # predijo un bias específico pero era el equivocado
tn = (audit_df['false_regime_rate'] == 'True Negative').sum() #  predijo choppy y era choppy
fn = (audit_df['false_regime_rate'] == 'False Negative').sum() # predijo un bias específico, pero la realidad fue choppy
fs = (audit_df['false_regime_rate'] == 'False Signal').sum()  # predijo choppy, pero la realidad tuvo un bias real

precision = tp / (tp + fp)
recall = tp / (tp + fn)

print(f"TP={tp}  FP={fp}  TN={tn}  FN={fn}  False Signal={fs}")
print(f"Precision (acierta la dirección exacta cuando predice un bias): {precision:.2%}")
print(f"Recall (detecta el bias real cuando existe): {recall:.2%}")

TP=41  FP=18  TN=7  FN=1  False Signal=3
Precision (acierta la dirección exacta cuando predice un bias): 69.49%
Recall (detecta el bias real cuando existe): 97.62%


### `false_regime_rate` por `bias_a` — ¿dónde se concentran los False Positive?

In [131]:
regime_by_bias_a = pd.crosstab(audit_df['bias_a'], audit_df['false_regime_rate'], margins=True, margins_name='Total')
display(regime_by_bias_a)

false_regime_rate,False Negative,False Positive,False Signal,True Negative,True Positive,Total
bias_a,,,,,,
BOS,0,7,0,0,22,29
CHOCH,0,4,1,0,5,10
Choppy-Bearish Range Rotation,0,0,0,0,4,4
Choppy-Bullish Range Rotation,0,0,0,0,2,2
No_Bias(Choppy),1,0,0,7,0,8
Trend Reversal,0,6,2,0,6,14
Validated Range Expansion,0,1,0,0,2,3
Total,1,18,3,7,41,70


### `false_regime_rate` por `market_bias`

In [132]:
regime_by_market_bias = pd.crosstab(audit_df['market_bias'], audit_df['false_regime_rate'], margins=True, margins_name='Total')
display(regime_by_market_bias)

false_regime_rate,False Negative,False Positive,False Signal,True Negative,True Positive,Total
market_bias,,,,,,
Bearish,0,6,0,0,19,25
Bullish,0,3,0,0,10,13
Choppy / Neutral,1,9,3,7,12,32
Total,1,18,3,7,41,70


### `false_regime_rate` por rango de `primary_prob`

In [133]:
regime_by_prob = pd.crosstab(prob_bins, audit_df['false_regime_rate'], margins=True, margins_name='Total')
display(regime_by_prob)

false_regime_rate,False Negative,False Positive,False Signal,True Negative,True Positive,Total
primary_prob,,,,,,
"(0.099, 0.156]",1,7,2,7,8,25
"(0.156, 0.215]",0,7,1,0,14,22
"(0.215, 0.408]",0,4,0,0,19,23
Total,1,18,3,7,41,70


### Lectura de los resultados (n=62)

**Distribución:** True Positive 58.1% (36), False Positive 22.6% (14), True Negative 11.3% (7), False Signal 4.8% (3), False Negative 3.2% (2).

**Precision 72.0% / Recall 94.7%:** el sistema casi nunca se queda afuera de un movimiento real (solo 2 False Negative), pero cuando predice un bias específico, 1 de cada 4 veces (14/50) predice la dirección/tipo equivocado. El error dominante no es "quedarse quieto", es "moverse en el bias incorrecto".

**Por `bias_a` — dónde se concentran los False Positive + False Signal:**
- `Trend Reversal`: 5 FP + 2 FS = **7 de 12** (58.3%) — el peor de la muestra, consistente con el 41.67% win rate de la Sección 3.
- `CHOCH`: 4 FP + 1 FS = **5 de 9** (55.6%).
- `BOS`: 4 FP de 24 (16.7%) — el más limpio de los que tienen volumen real.
- `No_Bias(Choppy)`: sus 2 errores son ambos False Negative (se quedó afuera de un movimiento real 2 veces) — nunca produce False Positive/Signal, coherente con ser la clase negativa.

**Por `market_bias`:** todos los False Negative (2) y todos los False Signal (3) ocurren en Choppy — tiene sentido, son los únicos casos donde `real_bias_b` puede resultar `No_Bias(Choppy)`. Bearish tiene la mejor proporción TP/(TP+FP): 18/23 = 78.3%.

**Por `primary_prob`:** el tercil bajo concentra 2/2 False Negative y 2/3 False Signal — cuando la probabilidad es baja, el modelo casi no distingue entre "va a haber movimiento" y "esto es ruido". El tercil alto no tiene ni False Negative ni False Signal ni True Negative — a probabilidad alta, siempre predijo (y casi siempre acertó) un bias específico.

**Conclusión:** el punto débil no es la detección de oportunidad (recall alto), es la precisión direccional cuando sí hay señal — y ese error se concentra casi todo en `Trend Reversal` y `CHOCH`, igual que en la Sección 3.

## Sección 4.A (temporal — verificación de integridad, no forma parte del modelo)

Chequeo puntual: de todos los setups con `specific_bias_compliance == 'Valid'`, ¿el precio realmente llegó a `edge_validation_price` (unified_department) según lo que quedó registrado en `structural_mfe` (efficiency_audit)?

Metodología: `structural_mfe` es un precio crudo, no una distancia — es el mejor precio alcanzado a favor de la tesis. Dirección de la tesis = `theoretical_dir` (ya calculado en Sección 2, a partir del signo de `calc_edge`):
- Tesis **Long**: se considera alcanzado si `structural_mfe >= edge_validation_price`.
- Tesis **Short**: se considera alcanzado si `structural_mfe <= edge_validation_price`.

Se excluyen del chequeo los casos sin `edge_validation_price` o sin `structural_mfe` cargado (no se puede verificar, se reportan aparte).

In [134]:
valid_df = audit_df[audit_df['is_valid'] == 1].merge(
    unified_df[['id', 'edge_validation_price', 'created_at']].rename(columns={'created_at': 'analysis_date'}),
    on='id', how='left'
)

has_data = valid_df['edge_validation_price'].notna() & valid_df['structural_mfe'].notna()
checkable_df = valid_df[has_data].copy()
missing_df = valid_df[~has_data].copy()

checkable_df['reached_validation'] = np.where(
    checkable_df['theoretical_dir'] == 'Long',
    checkable_df['structural_mfe'] >= checkable_df['edge_validation_price'],
    checkable_df['structural_mfe'] <= checkable_df['edge_validation_price']
)

n_valid = len(valid_df)
n_checkable = len(checkable_df)
n_reached = checkable_df['reached_validation'].sum()
n_not_reached = n_checkable - n_reached

print(f"Total 'Valid': {n_valid}")
print(f"Verificables (con edge_validation_price y structural_mfe): {n_checkable}")
print(f"  Alcanzaron la validación: {n_reached}")
print(f"  NO alcanzaron la validación: {n_not_reached}")
print(f"Sin datos suficientes para verificar: {len(missing_df)}")

Total 'Valid': 48
Verificables (con edge_validation_price y structural_mfe): 42
  Alcanzaron la validación: 40
  NO alcanzaron la validación: 2
Sin datos suficientes para verificar: 6


### Detalle: casos "Valid" que no alcanzaron `edge_validation_price` (si los hay)

In [135]:
not_reached_df = checkable_df[~checkable_df['reached_validation']]
cols = ['id', 'analysis_date', 'bias_a', 'theoretical_dir', 'calc_edge', 'edge_validation_price', 'structural_mfe']

if not_reached_df.empty:
    print("Ningún caso 'Valid' verificable falló el chequeo — los 43 'Valid' con datos completos son consistentes.")
else:
    display(not_reached_df[cols])

,id,analysis_date,bias_a,theoretical_dir,calc_edge,edge_validation_price,structural_mfe
19,d55f09b2-f542-4d0e-81ab-334ada101e7d,2026-06-26 05:26:02.574408,Trend Reversal,Long,0.5500,4100.0000,4097.0000
43,68ff0f26-b77b-4468-a16f-f7570a70e144,2026-08-09 18:02:49.997402,No_Bias(Choppy),Short,-0.0250,4287.0000,4314.0000


### Detalle: casos "Valid" sin datos suficientes para verificar

In [136]:
cols_missing = ['id', 'analysis_date', 'bias_a', 'theoretical_dir', 'edge_validation_price', 'structural_mfe']
display(missing_df[cols_missing])

,id,analysis_date,bias_a,theoretical_dir,edge_validation_price,structural_mfe
1,1998959d-7ea9-4d3e-8f82-1ec007021143,2026-05-19 06:55:18.918597,No_Bias(Choppy),Long,NaN,NaN
3,b9c076bf-85fc-450f-a7d2-48917c1652f4,2026-05-22 06:55:15.135212,No_Bias(Choppy),Short,NaN,NaN
15,48937943-f9e2-4871-8c69-35e61724bf42,2026-06-19 07:47:36.256435,No_Bias(Choppy),Short,NaN,NaN
27,38d0f2bb-c47c-4d20-8bfb-2e980b4ab027,2026-07-23 05:49:18.426690,No_Bias(Choppy),Short,NaN,NaN
28,4a166533-8ade-48a2-8001-fa258d7b9e8a,2026-07-24 06:20:01.820872,No_Bias(Choppy),Long,4077.5000,NaN
38,d74b6b37-b674-4867-895d-9d13115c3cb7,2026-07-31 05:58:58.550297,No_Bias(Choppy),Short,NaN,NaN


### Lectura de los resultados (solo lectura — no se modificó ninguna DB)

De los 43 "Valid": **27 verificables** (16 sin `edge_validation_price` y/o `structural_mfe` cargado — 37% de los "Valid" no se puede auditar así). De los 27 verificables, **24 alcanzaron la validación y 3 no**:

| id | fecha | bias_a | dirección | edge_validation_price | structural_mfe | desvío |
|---|---|---|---|---|---|---|
| `d55f09b2` | 2026-06-26 | Trend Reversal | Long | 4100 | 4097 | -3 (rozó el nivel, no lo cruzó) |
| `f24b9653` | 2026-07-09 | BOS | Short | 4020 | 4094 | +74 (nunca bajó lo suficiente) |
| `68ff0f26` | 2026-08-09 | No_Bias(Choppy) | Short | 4287 | 4314 | +27 (`calc_edge=-0.025`, prácticamente 0 — la dirección teórica es ambigua acá) |

El primero (`d55f09b2`) es un desvío marginal, probablemente precisión de entrada manual. Los otros dos (`f24b9653`, `68ff0f26`) están marcados `specific_bias_compliance='Valid'` pero su propio `structural_mfe` registrado no confirma que el precio llegó al nivel de validación — vale la pena revisarlos manualmente antes de asumir que son errores de dato vs. error de cálculo.

**Patrón temporal en los 16 "sin datos suficientes":** al agregar la fecha, 10 de los 16 son del **2026-07-21 en adelante** (el resto son mayormente `No_Bias(Choppy)`, donde `edge_validation_price` no aplica por diseño). Sugiere que la carga de `structural_mfe` se volvió menos consistente en las analíticas más recientes — vale la pena confirmar si hay un cambio de flujo/hábito de registro alrededor de esa fecha.

No se tocó la DB — esto es solo el hallazgo, sección temporal de verificación.

## Sección 5: `market_bias`/probabilidades vs `structural_entry_price`, MAE y MFE

`structural_entry_price` no es una columna guardada — es un valor calculado por la propia app (`core/math_engine.py::calculate_structural_entry`, usado en `cli/main.py:1774` para la "Structural Entry Simulation"). Fórmula: `E = (TP + RR·SL) / (1 + RR)`, con `TP=edge_validation_price`, `SL=structural_invalidation`, `RR=STRUCTURAL_TARGET_RR` (2.2 en el `.env` real). Es el precio de entrada teórico para ese R:R objetivo — un pullback entre el precio de invalidación y el de validación.

Dos convenciones de dirección calculadas en `struct_df`:
- **`direction_from_bias`** (Long si `market_bias=='Bullish'`, Short si `'Bearish'`, `None` si Choppy) — la misma que usa `cli/main.py` para su propio chequeo de "¿alcanzó la entrada?". Queda disponible en el DataFrame para referencia/comparación contra la lógica de producción, pero no se usa en las tablas de esta sección.
- **`theoretical_dir`** (signo de `calc_edge`) — sí cubre Choppy, y es la que se usa en **todas** las tablas de esta sección (①-④) para poder comparar los 3 regímenes en pie de igualdad.

Con esto respondemos: (1) qué `market_bias` llega más al `structural_entry_price`, (2) MAE/MFE crudos y relativos a la entrada por bias y por probabilidad, (3) qué tan lejos está `mark_price` del `structural_entry_price` en promedio.
**Actualización:** `struct_df` ahora también trae `entry_price` y `order_filled` (`tactical_audit`, merge agregado en la construcción base) — el precio real cargado en el tactical audit, distinto de `structural_entry_price` (que sigue siendo el pullback teórico calculado arriba, sin cambios). **Ojo:** `entry_price` puede estar poblado con `order_filled=False` (intento con precio propuesto que nunca se llenó, o filas legacy con `entry_price==closing_price` como placeholder pre-rediseño del wizard) — no usar `entry_price.notna()` como proxy de "se ejecutó", filtrar siempre por `order_filled == True` explícitamente. Esta sección (①-④) no cambia sus tablas; `entry_price` se usa en las Secciones 14-17 más abajo.


In [137]:
# Base de la sección: todos los 65 análisis (no solo los resueltos), con MAE/MFE de efficiency_audit
struct_df = unified_df.merge(efficiency_df[['id', 'structural_mae', 'structural_mfe']], on='id', how='left')

with engine.connect() as conn:
    tactical_entry_df = _load_tactical_slice(conn, ["entry_price", "order_filled"])
# Merge sin filtrar por order_filled — entry_price queda disponible para TODA la muestra.
# entry_price puede estar poblado con order_filled=False (intento no llenado, o legacy) —
# cualquier análisis que represente "ejecución real" debe filtrar order_filled==True explícito.
struct_df = struct_df.merge(tactical_entry_df, on='id', how='left')

STRUCTURAL_TARGET_RR = 2.2  # os.getenv("STRUCTURAL_TARGET_RR", "2.2") en el .env real
struct_df['structural_entry_price'] = np.where(
    struct_df['edge_validation_price'].notna() & struct_df['structural_invalidation'].notna(),
    (struct_df['edge_validation_price'] + STRUCTURAL_TARGET_RR * struct_df['structural_invalidation']) / (1 + STRUCTURAL_TARGET_RR),
    np.nan
)

struct_df['direction_from_bias'] = np.select(
    [struct_df['market_bias'] == 'Bullish', struct_df['market_bias'] == 'Bearish'],
    ['Long', 'Short'],
    default=None
)
struct_df['theoretical_dir'] = np.where(struct_df['calc_edge'] >= 0, 'Long', 'Short')
struct_df['primary_prob'] = np.where(struct_df['theoretical_dir'] == 'Long', struct_df['long_prob'], struct_df['short_prob'])

print(f"Total análisis: {len(struct_df)}")
print(f"Con structural_entry_price calculable: {struct_df['structural_entry_price'].notna().sum()}")
print(f"Con structural_mae cargado: {struct_df['structural_mae'].notna().sum()}")
print(f"Con structural_mfe cargado: {struct_df['structural_mfe'].notna().sum()}")

Total análisis: 70
Con structural_entry_price calculable: 65
Con structural_mae cargado: 64
Con structural_mfe cargado: 64


### ① ¿Qué `market_bias` llega más seguido al `structural_entry_price`?

Usa `theoretical_dir` (signo de `calc_edge`) en vez de `direction_from_bias`, para poder incluir Choppy — que no tiene una dirección definida en la convención de la app (`market_bias`), pero sí tiene un `calc_edge` con signo.

In [138]:
checkable = struct_df[
    struct_df['theoretical_dir'].notna() &
    struct_df['structural_mae'].notna() &
    struct_df['structural_entry_price'].notna()
].copy()

checkable['reached_entry'] = np.where(
    checkable['theoretical_dir'] == 'Short',
    checkable['structural_mae'] >= checkable['structural_entry_price'],
    checkable['structural_mae'] <= checkable['structural_entry_price']
)

reached_by_bias = checkable.groupby('market_bias')['reached_entry'].agg(N_Verificable='count', Reached='mean')
reached_by_bias['Reached_%'] = (reached_by_bias['Reached'] * 100).round(2)

display(reached_by_bias[['N_Verificable', 'Reached_%']])

,N_Verificable,Reached_%
market_bias,,
Bearish,25,64.0000
Bullish,13,46.1500
Choppy / Neutral,26,57.6900


### ② MAE y MFE — crudos, y relativos al `structural_entry_price`

`structural_mae`/`structural_mfe` son precios crudos de XAUUSD, tomados en fechas distintas (el activo se movió de ~4000 a ~4600 en el período) — comparar sus niveles absolutos entre bias no es una medida justa. Por eso se agrega una segunda tabla: **distancia más allá de la entrada**, direccional (usa `theoretical_dir`, así que sí incluye Choppy):
- `mae_past_entry` = qué tan lejos, en contra de la tesis, se movió el precio **más allá** del `structural_entry_price` (adverso).
- `mfe_past_entry` = qué tan lejos, a favor, se movió el precio **más allá** del `structural_entry_price` (favorable).

In [139]:
print("--- MAE/MFE crudos (precio) por market_bias ---")
display(struct_df.groupby('market_bias')[['structural_mae', 'structural_mfe']].agg(['count', 'mean', 'median']).round(2))

--- MAE/MFE crudos (precio) por market_bias ---


structural_mae                     structural_mfe                    
                          count      mean    median          count      mean    median
market_bias                                                                           
Bearish                      25 4211.9100 4148.0000             25 4124.1400 4044.7500
Bullish                      13 4209.5100 4181.0000             13 4305.5200 4303.0000
Choppy / Neutral             26 4304.1300 4361.5000             26 4315.5600 4356.5000

In [140]:
struct_df['mae_past_entry'] = np.where(
    struct_df['theoretical_dir'] == 'Long',
    struct_df['structural_entry_price'] - struct_df['structural_mae'],
    struct_df['structural_mae'] - struct_df['structural_entry_price']
)
struct_df['mfe_past_entry'] = np.where(
    struct_df['theoretical_dir'] == 'Long',
    struct_df['structural_mfe'] - struct_df['structural_entry_price'],
    struct_df['structural_entry_price'] - struct_df['structural_mfe']
)

print("--- Distancia más allá del structural_entry_price (puntos), por market_bias ---")
display(struct_df.groupby('market_bias')[['mae_past_entry', 'mfe_past_entry']].agg(['count', 'mean', 'median']).round(2))

--- Distancia más allá del structural_entry_price (puntos), por market_bias ---


mae_past_entry                 mfe_past_entry                
                          count    mean  median          count    mean  median
market_bias                                                                   
Bearish                      25 20.3300  4.2500             25 67.4400 68.3100
Bullish                      13 13.4600 -3.1900             13 82.5500 62.8800
Choppy / Neutral             26 19.3800  6.6900             26 52.6900 48.5900

### ③ Lo mismo, por rango de `primary_prob`

In [212]:
struct_prob_bins = pd.qcut(struct_df['primary_prob'], q=3, duplicates='drop')

print("--- Distancia más allá del structural_entry_price, por tercil de primary_prob ---")
display(struct_df.groupby(struct_prob_bins, observed=False)[['mae_past_entry', 'mfe_past_entry']].agg(['count', 'mean','median']).round(2))

--- Distancia más allá del structural_entry_price, por tercil de primary_prob ---


mae_past_entry                mfe_past_entry                
                        count    mean median          count    mean  median
primary_prob                                                               
(0.099, 0.156]             19 27.0300 9.0300             19 52.6800 48.5000
(0.156, 0.215]             22 12.8600 0.4400             22 47.5900 57.0700
(0.215, 0.408]             23 16.9800 4.2500             23 90.4800 68.3100

### ④ Qué tan lejos está `mark_price` del `structural_entry_price`

`mark_price` es el precio de referencia al momento de revisar el análisis. Distancia con signo (positivo = mark_price por encima de la entrada) y distancia absoluta, por `market_bias`.

In [142]:
struct_df['dist_mark_entry'] = struct_df['mark_price'] - struct_df['structural_entry_price']
struct_df['dist_mark_entry_abs'] = struct_df['dist_mark_entry'].abs()

display(struct_df.groupby('market_bias')[['dist_mark_entry', 'dist_mark_entry_abs']].agg(['count', 'mean', 'median']).round(2))

dist_mark_entry                   dist_mark_entry_abs                
                           count     mean   median               count    mean  median
market_bias                                                                           
Bearish                       25 -15.4900 -14.7000                  25 16.3300 14.7000
Bullish                       13  16.1900  14.7800                  13 16.1900 14.7800
Choppy / Neutral              27   2.6000   6.5600                  27 15.7800 12.4900

### Lectura de los resultados

**① Llegada al `structural_entry_price`** (usando `theoretical_dir`, incluye Choppy): **Choppy 68.75% (n=16)**, Bullish 66.67% (n=6), Bearish 61.11% (n=18) — los tres bastante parejos, y Choppy incluso el más alto. Muestras chicas en Choppy y sobre todo Bullish, no alcanza para ser concluyente, pero descarta la idea de que Choppy tenga más problemas para llegar a la zona de entrada — el problema de Choppy, visto en Secciones 2-4, está en la resolución posterior (win rate, `false_regime_rate`), no en si el precio hace el pullback hasta ahí.

**② MAE/MFE relativos a la entrada — el hallazgo más claro de la sección:** mirando la razón favorable/adverso más allá de `structural_entry_price`:
- **Bearish**: MFE 82.69 vs MAE 17.42 → ratio **~4.75x**. El mejor de los tres, y consistente con Bearish siendo el bias con mejor `structural_resolution` en la Sección 3.
- **Choppy**: MFE 47.02 vs MAE 21.55 → ratio ~2.18x.
- **Bullish**: MFE 60.82 vs MAE 35.78 → ratio **~1.70x**, el peor — aunque Bullish llega más seguido a la entrada (①), el recorrido a favor después de entrar es proporcionalmente el más débil de los tres.

**③ Por probabilidad:** `mae_past_entry` baja monótonamente con la probabilidad (26.77 → 21.85 → 18.52) — a mayor confianza del modelo, menor sobrepaso adverso. `mfe_past_entry` no es monótono, pero el tercil alto dispara a 86.08 (ratio ~4.65x vs su propio MAE de 18.52) — muy en línea con el patrón de Bearish, y con la colinealidad prob/bias ya vista en Secciones 2 y 4.

**④ `mark_price` vs `structural_entry_price`:** en promedio, Bearish está -15.43 pts por debajo de su entrada ideal (esperando un rally para vender) y Bullish +12.76 pts por encima (esperando un pullback para comprar) — Bullish es el que en promedio queda más cerca de su zona de entrada (abs 12.76 vs 16.31 de Bearish).

**Conclusión:** el eje que más separa la calidad de un setup no es "llega o no llega a la entrada" (①, parejo entre los 3 regímenes, Choppy incluido), es el **ratio MFE/MAE una vez pasada la entrada** — ahí Bearish domina claramente y Bullish es el más débil, matizando el resultado de la Sección 2 (donde Bullish tenía el win rate más alto, pero con la muestra más chica de los tres). Y para Choppy específicamente: sí llega a la zona de entrada casi tan seguido como los otros dos, así que el filtro de "descartar Choppy" de la Sección 2 se sostiene mejor en la resolución/win rate que en el acceso geométrico al precio.

---

### Propuesta para una próxima fase

1. **Normalizar a R-múltiplos**, usando `|structural_entry_price − structural_invalidation|` como 1R (el mismo criterio de riesgo que ya define `STRUCTURAL_TARGET_RR`). Convertir `mae_past_entry`/`mfe_past_entry` a unidades de R haría estos números directamente comparables entre sí y contra el `r_multiple` realizado de `tactical_audit` — hoy están en puntos crudos de precio, que no es una unidad estable entre activos ni a través del tiempo.
2. **Usar la profundidad del pullback (`mae_past_entry`) como señal líder**, no solo descriptiva: cruzarla contra `specific_bias_compliance`/`false_regime_rate` para ver si un pullback más profundo (mayor `mae_past_entry`) predice o no un peor desenlace — sería una señal nueva, distinta de `calc_edge`, que hoy no se está aprovechando en el modelo.
3. **Medir el "timing" de la revisión**: usando `dist_mark_entry`, calcular en qué fracción de los casos `mark_price` ya cruzó `structural_entry_price` al momento del review (`updated_at`) — es decir, si la entrada teórica ya quedó atrás cuando se audita, o todavía es una zona accionable. Esto conecta directamente con Sección 4.A (ejecuciones "Valid" que no llegaron a validar el precio).

## Sección 6: las 3 fases propuestas — R-múltiplos, señal líder y timing

Ejecuta, en orden, las 3 propuestas del cierre de la Sección 5. Usa `struct_df` (ya tiene `structural_entry_price`, `theoretical_dir`, `mae_past_entry`, `mfe_past_entry`).

### Fase 1: Normalizar `mae_past_entry`/`mfe_past_entry` a R-múltiplos

`1R = |structural_entry_price − structural_invalidation|` — la misma distancia de riesgo que ya define `STRUCTURAL_TARGET_RR`. Se extrae también `tactical_audit.r_multiple` (solo esas 2 columnas, no la tabla completa) para comparar el potencial estructural contra lo realmente ejecutado.

In [143]:
struct_df['one_r'] = (struct_df['structural_entry_price'] - struct_df['structural_invalidation']).abs()
struct_df['mae_R'] = struct_df['mae_past_entry'] / struct_df['one_r']
struct_df['mfe_R'] = struct_df['mfe_past_entry'] / struct_df['one_r']

print("--- 1R (distancia entry-invalidación, en puntos) ---")
display(struct_df['one_r'].describe().round(2).to_frame().T)

print("\n--- mae_R / mfe_R por market_bias ---")
display(struct_df.groupby('market_bias')[['mae_R', 'mfe_R']].agg(['count', 'mean', 'median']).round(3))

--- 1R (distancia entry-invalidación, en puntos) ---


,count,mean,std,min,25%,50%,75%,max
one_r,65.0000,26.9500,11.2500,12.5000,18.7500,22.8100,31.5600,57.8100



--- mae_R / mfe_R por market_bias ---


mae_R                mfe_R              
                 count   mean  median count   mean median
market_bias                                              
Bearish             25 0.9090  0.0930    25 2.2140 2.6480
Bullish             13 0.6350 -0.1400    13 2.7410 2.3600
Choppy / Neutral    26 0.8110  0.2590    26 2.4340 1.7240

**Potencial estructural (R) vs resultado táctico realizado (`r_multiple`)**

In [144]:
with engine.connect() as conn:
    # order_filled ya viene en struct_df (cell 49)
    tactical_r_df = _load_tactical_slice(conn, ["r_multiple"])

realized_df = struct_df.merge(tactical_r_df, on='id', how='inner')
executed_df = realized_df[realized_df['order_filled'].fillna(True).astype(bool) & realized_df['r_multiple'].notna()]

comparison = pd.DataFrame({
    'N': [struct_df['mfe_R'].notna().sum(), struct_df['mae_R'].notna().sum(), len(executed_df)],
    'Mean': [struct_df['mfe_R'].mean(), struct_df['mae_R'].mean(), executed_df['r_multiple'].mean()],
    'Median': [struct_df['mfe_R'].median(), struct_df['mae_R'].median(), executed_df['r_multiple'].median()],
}, index=['mfe_R (potencial estructural, todos)', 'mae_R (riesgo estructural, todos)', 'r_multiple (realizado, ejecutados)'])

display(comparison.round(3))

,N,Mean,Median
"mfe_R (potencial estructural, todos)",64,2.4110,2.5200
"mae_R (riesgo estructural, todos)",64,0.8140,0.1200
"r_multiple (realizado, ejecutados)",28,0.1790,-0.4330


### Fase 2: `mae_R` (profundidad del pullback) como señal líder

Cruce contra `specific_bias_compliance` (¿un pullback más profundo anticipa un peor desenlace estructural?) y contra `false_regime_rate` (¿en qué categoría de la matriz de confusión de Sección 4 se concentra el pullback profundo?).

In [145]:
# audit_df (Sección 2) ya trae specific_bias_compliance/is_valid/false_regime_rate para los 62 resueltos.
# Se le suma mae_R desde struct_df.
signal_df = audit_df.merge(struct_df[['id', 'mae_R']], on='id', how='left')
signal_df = signal_df[signal_df['mae_R'].notna()].copy()

print(f"Setups resueltos con mae_R calculable: {len(signal_df)} de {len(audit_df)}")

print("\n--- mae_R por specific_bias_compliance ---")
display(signal_df.groupby('specific_bias_compliance')['mae_R'].agg(['count', 'mean', 'median']).round(3))

Setups resueltos con mae_R calculable: 64 de 70

--- mae_R por specific_bias_compliance ---


,count,mean,median
specific_bias_compliance,,,
Invalid,22,2.1930,1.7850
Valid,42,0.0910,-0.1250


**Win rate por tercil de `mae_R`**

In [146]:
mae_R_bins = pd.qcut(signal_df['mae_R'], q=3, duplicates='drop')
win_rate_by_mae_R = signal_df.groupby(mae_R_bins, observed=False)['is_valid'].agg(N_Setups='count', Win_Rate='mean')
win_rate_by_mae_R['Win_Rate_%'] = (win_rate_by_mae_R['Win_Rate'] * 100).round(2)

display(win_rate_by_mae_R[['N_Setups', 'Win_Rate_%']])

,N_Setups,Win_Rate_%
mae_R,,
"(-0.921, -0.182]",21,90.4800
"(-0.182, 0.715]",21,85.7100
"(0.715, 7.186]",22,22.7300


**`mae_R` por `false_regime_rate`**

In [147]:
display(signal_df.groupby('false_regime_rate')['mae_R'].agg(['count', 'mean', 'median']).round(3))

,count,mean,median
false_regime_rate,,,
False Negative,1,3.9820,3.9820
False Positive,18,2.2420,1.7850
False Signal,3,1.3070,-0.0520
True Negative,1,-0.2270,-0.2270
True Positive,41,0.0990,-0.1090


### Fase 3: timing — ¿ya cruzó `mark_price` el `structural_entry_price`?

Distinto del chequeo de la Sección 4.A (que usaba `structural_mfe`, el mejor precio alcanzado en *todo* el período hasta la resolución). Acá se usa `mark_price` — el precio de referencia en el momento puntual de la analítica/revisión — para saber si, en ese instante, la zona de entrada ya estaba en juego o todavía era un objetivo a futuro.

In [148]:
timing_df = struct_df[
    struct_df['mark_price'].notna() &
    struct_df['structural_entry_price'].notna() &
    struct_df['theoretical_dir'].notna()
].copy()

timing_df['crossed_entry'] = np.where(
    timing_df['theoretical_dir'] == 'Long',
    timing_df['mark_price'] <= timing_df['structural_entry_price'],
    timing_df['mark_price'] >= timing_df['structural_entry_price']
)

print(f"Verificables: {len(timing_df)} de {len(struct_df)}")
print(f"Ya cruzada la entrada al momento del review: {timing_df['crossed_entry'].sum()} ({timing_df['crossed_entry'].mean():.2%})")
print(f"Todavía pendiente (entrada como objetivo futuro): {(~timing_df['crossed_entry']).sum()} ({(~timing_df['crossed_entry']).mean():.2%})")

print("\n--- por market_bias ---")
crossed_by_bias = timing_df.groupby('market_bias')['crossed_entry'].agg(N='count', Crossed_Rate='mean')
crossed_by_bias['Crossed_%'] = (crossed_by_bias['Crossed_Rate'] * 100).round(2)
display(crossed_by_bias[['N', 'Crossed_%']])

Verificables: 65 de 70
Ya cruzada la entrada al momento del review: 5 (7.69%)
Todavía pendiente (entrada como objetivo futuro): 60 (92.31%)

--- por market_bias ---


,N,Crossed_%
market_bias,,
Bearish,25,8.0000
Bullish,13,0.0000
Choppy / Neutral,27,11.1100


### Lectura de los resultados de la Sección 6

**Fase 1 — R-múltiplos:** 1R promedio son ~25.6 puntos de XAUUSD (12.5 a 56.25 según el setup). En R, `mae_R`/`mfe_R` por bias confirman el ratio de la Sección 5 en unidades comparables (Bearish mfe_R/mae_R ≈ 3.47x, Bullish ≈ 1.68x). El hallazgo fuerte es la **brecha estructural vs realizado**: el potencial estructural promedio es **mfe_R = 2.48R** (lo que el mercado ofreció más allá de la entrada, en los 40 setups con datos) contra apenas **r_multiple = 0.18R** de promedio realizado en los 25 trades ejecutados — y la mediana realizada es incluso **negativa (-0.82R)**. El sistema estructural identifica movimientos grandes; la ejecución táctica captura una fracción mínima de ellos.

**Fase 2 — `mae_R` como señal líder (el hallazgo más fuerte de todo el notebook hasta ahora):**
- `mae_R` medio en `Invalid` (2.40) es **9x** el de `Valid` (0.26) — el pullback profundo antes de resolver ya distingue setups buenos de malos.
- Win rate por tercil de `mae_R`: 84.6% (bajo, típicamente ni siquiera llegó a la entrada) → **100%** (medio, entra con pullback moderado) → **21.4%** (alto, pullback profundo, 3 de cada 4 fallan). El tercio con más pullback es el peor con diferencia.
- Por `false_regime_rate`: `False Positive` tiene `mae_R` medio 2.86 vs `True Positive` 0.28 — 10x. La profundidad del pullback no solo predice `Valid`/`Invalid`, predice específicamente el tipo de error (dirección equivocada).
- **Implicación directa para el modelo**: `mae_R` alto (>~0.8R) es, con esta muestra, una señal casi tan fuerte como `P0` (63.3% win rate, la única capa con edge real según el análisis previo del proyecto) — y es una señal barata de calcular, ya con los datos existentes.

**Fase 3 — Timing:** solo **8.77% (5/57)** de los análisis tenían la entrada ya cruzada al momento del `mark_price` de referencia — el 91.23% restante todavía esperaba el pullback. Esto es estructural, no una anomalía: `structural_entry_price` está diseñado como un objetivo futuro, no un gatillo inmediato. Bullish nunca (0%) tuvo la entrada ya cruzada en esta muestra (n=11); Choppy fue el más frecuente (13.64%, n=22) — coherente con que Choppy es más ruidoso y cruza niveles con más frecuencia sin que eso implique un movimiento direccional sostenido.

**Conclusión general de la Sección 6:** de las 3 fases, la más accionable para el modelo custom es la 2 — `mae_R` (profundidad del pullback más allá del `structural_entry_price`, en R) es una señal predictiva fuerte y hasta ahora no explotada, independiente de `calc_edge`. Vale la pena considerarla como filtro o input adicional en las próximas secciones.

## Sección 7: Validación de P0-P4 con `core/edge_analysis.py`

Reusa las funciones de producción del repo (`analisis_edge`, `analizar_correlaciones`, `profit_factor_by_edge_bins`, `get_market_outcome`) — no se reimplementa nada, solo se arma el input que esperan: `analysis_layer` pivoteado de formato largo a columnas `p0_score`...`p4_score`, unido a `market_bias`/`calc_edge`/`specific_bias_compliance`.

**Extra pedido:** `analisis_edge` mide qué tan seguido cada Px acierta cuando el *sistema* acierta (`Participación`), pero no dice nada de los casos donde el sistema **falló** (`specific_bias_compliance == 'Invalid'`). Se agrega una métrica nueva — **"Rescate"**: de esos fallos, ¿cuántas veces algún Px individual sí había marcado la dirección correcta (`sign(Px) == sign(Market_Outcome)`), y terminó diluido por la suma ponderada de los demás Ps? Ejemplo textual del pedido: `market_bias=Bullish`, `compliance=Invalid` (el precio se fue Bearish) — ¿algún Px marcó Bearish ahí?

In [149]:
from core.edge_analysis import analisis_edge, analizar_correlaciones, profit_factor_by_edge_bins, get_market_outcome

# Pivot: analysis_layer (largo, layer_name in P0..P4) -> p0_score..p4_score (ancho, 1 fila por trade)
score_pivot = layer_df.pivot(index='trade_id', columns='layer_name', values='score')
score_pivot = score_pivot.rename(columns={f'P{i}': f'p{i}_score' for i in range(5)})
score_pivot = score_pivot.reset_index().rename(columns={'trade_id': 'id'})

px_df = unified_df[['id', 'market_bias', 'calc_edge']].merge(
    efficiency_df[['id', 'specific_bias_compliance']], on='id', how='left'
).merge(score_pivot, on='id', how='left')

print(f"px_df: {px_df.shape[0]} filas, {px_df.shape[1]} columnas")
display(px_df.head(3))

px_df: 70 filas, 9 columnas


,id,market_bias,calc_edge,specific_bias_compliance,p0_score,p1_score,p2_score,p3_score,p4_score
0,2834cbeb-b65f-4cad-97ca-1807e3193b23,Choppy / Neutral,0.2250,Valid,0,1,0,0,1
1,1998959d-7ea9-4d3e-8f82-1ec007021143,Choppy / Neutral,0.0000,Valid,0,1,-1,-1,0
2,0a68d499-0175-4048-acc4-8412edbb51c4,Choppy / Neutral,0.0750,Valid,-1,0,1,1,1


### `analisis_edge` — Net Profit, Profit Factor, Win Rate, Participación y Correlación por P

In [150]:
edge_results = analisis_edge(px_df)
display(edge_results)

,Parametro,Net Profit,Profit Factor,Win Rate (%),Participacion (Wins %),Correlacion
0,P0,7,1.4700,59.5000,45.8000,0.4200
1,P2,0,1.0000,50.0000,35.4000,0.4300
2,P3,0,1.0000,50.0000,41.7000,0.3400
3,P1,-2,0.9300,48.1000,54.2000,0.3900
4,P4,-6,0.7800,43.8000,43.8000,0.3000


### `analizar_correlaciones` — ¿los Ps son redundantes entre sí? (Spearman)

In [151]:
display(analizar_correlaciones(px_df))

,p0_score,p1_score,p2_score,p3_score,p4_score
p0_score,1.0000,0.3499,0.3918,0.5557,0.3186
p1_score,0.3499,1.0000,0.3001,0.4740,0.6973
p2_score,0.3918,0.3001,1.0000,0.3733,0.2612
p3_score,0.5557,0.4740,0.3733,1.0000,0.4268
p4_score,0.3186,0.6973,0.2612,0.4268,1.0000


### `profit_factor_by_edge_bins` — win rate por magnitud de `calc_edge`

In [152]:
display(profit_factor_by_edge_bins(px_df))

,edge_range,n_setups,win_rate
0,"(-0.001, 0.2]",25,100.0000
1,"(0.2, 0.375]",22,77.3000
2,"(0.375, 0.825]",23,82.6000


### Métrica nueva: "Rescate" — ¿algún Px acertó cuando el sistema falló?

Solo tiene sentido para `market_bias` Bullish/Bearish con `compliance='Invalid'` (Choppy no tiene una dirección real definida en `get_market_outcome`, siempre da `Market_Outcome=0`). Por cada Px: de esos fallos del sistema, ¿en qué % ese Px individual sí tenía el signo correcto?

In [153]:
px_df['Market_Outcome'] = px_df.apply(get_market_outcome, axis=1)

invalid_df = px_df[
    (px_df['specific_bias_compliance'] == 'Invalid') &
    (px_df['market_bias'].isin(['Bullish', 'Bearish']))
].copy()

score_cols = ['p0_score', 'p1_score', 'p2_score', 'p3_score', 'p4_score']
rescue_rows = []
for col in score_cols:
    active = invalid_df[invalid_df[col] != 0]
    rescued = active[np.sign(active[col]) == np.sign(active['Market_Outcome'])]
    rescue_rows.append({
        'Parametro': col.split('_')[0].upper(),
        'N_Fallos_Sistema': len(invalid_df),
        'N_Activo': len(active),
        'N_Rescatado': len(rescued),
        'Rescate_%': round(len(rescued) / len(invalid_df) * 100, 1) if len(invalid_df) else 0.0,
    })

rescue_df = pd.DataFrame(rescue_rows).sort_values('Rescate_%', ascending=False).reset_index(drop=True)
print(f"Fallos del sistema evaluables (Bullish/Bearish + Invalid): {len(invalid_df)} de 19 Invalid totales (12 son Choppy, no evaluables)")
display(rescue_df)

Fallos del sistema evaluables (Bullish/Bearish + Invalid): 9 de 19 Invalid totales (12 son Choppy, no evaluables)


,Parametro,N_Fallos_Sistema,N_Activo,N_Rescatado,Rescate_%
0,P2,9,3,1,11.1000
1,P0,9,5,0,0.0000
2,P1,9,8,0,0.0000
3,P3,9,7,0,0.0000
4,P4,9,7,0,0.0000


### Detalle: filas donde algún Px rescató la dirección correcta

In [154]:
def rescuers(row):
    return [c.split('_')[0].upper() for c in score_cols if row[c] != 0 and np.sign(row[c]) == np.sign(row['Market_Outcome'])]

invalid_df['rescatado_por'] = invalid_df.apply(rescuers, axis=1)
detail_cols = ['id', 'market_bias', 'specific_bias_compliance', 'Market_Outcome'] + score_cols + ['rescatado_por']

display(invalid_df[invalid_df['rescatado_por'].apply(len) > 0][detail_cols])

,id,market_bias,specific_bias_compliance,Market_Outcome,p0_score,p1_score,p2_score,p3_score,p4_score,rescatado_por
58,7b9ea3f2-810b-47d1-ac70-bb717e9302ca,Bearish,Invalid,1,-1,-2,1,-2,-1,[P2]


### Lectura de los resultados (n=62 resueltos)

**`analisis_edge`:** `P0` domina en Profit Factor (1.73) y Win Rate (63.3%) — confirma lo que ya sabíamos del proyecto (P0 es la única capa con edge direccional real). `P1` tiene la Participación más alta (58.1%, aparece más seguido en los aciertos del sistema) pero Win Rate de moneda al aire (51.0%) — está presente, no aporta. `P2`, `P3`, `P4` están todos por debajo de 50% Win Rate y Profit Factor <1 — restan al compuesto, coincide con el análisis previo del proyecto.

**`analizar_correlaciones` (Spearman entre Ps):** la correlación más alta es P1-P4 (0.68), seguida de P0-P3 (0.54) — hay redundancia parcial, ningún par es totalmente independiente, pero tampoco hay un par que sea prácticamente el mismo parámetro (todas <0.7).

**`profit_factor_by_edge_bins` — ⚠️ resultado engañoso, hay que leerlo con cuidado:** el bin de menor `|calc_edge|` muestra 100% win rate (n=24) — a primera vista parecería que "edge chico = más confiable", pero es un artefacto: **los 24 casos de ese bin son 100% Choppy**, y `profit_factor_by_edge_bins` (código de producción, sin tocar) calcula `bias_sign=0` para Choppy y `Market_Outcome=0` cuando `market_bias` no es Bullish/Bearish — `0 == 0` se cuenta como acierto trivialmente. No es que Choppy prediga bien, es que la función no tiene una noción de "Choppy correcto" distinta de "Choppy" a secas. Los bins 2 y 3 (76.5% y 85.7%, con Bullish/Bearish reales) son los únicos comparables entre sí.

**Métrica "Rescate":** de los 19 `Invalid` totales, solo 7 son evaluables (Bullish/Bearish; los otros 12 son Choppy, sin `Market_Outcome` definido). De esos 7 fallos del sistema, **1 solo caso fue "rescatable"**: `7b9ea3f2` (`Bearish`/`Invalid`, el mercado se fue alcista — `Market_Outcome=+1` — y `p2_score=+1` sí marcó esa dirección, mientras `p0_score=-1`, `p1_score=-2`, `p3_score=-2`, `p4_score=-1` empujaban fuerte hacia Bearish y ganaron la ponderación). Con n=7 no alcanza para generalizar, pero el patrón puntual es consistente con lo ya visto: cuando el sistema falla, es porque P1/P3/P4 (los más débiles) arrastran la suma en contra del parámetro que sí tenía razón.

## Sección 8: Profundización — consenso, P0 por bias_a, y re-ponderación

Los 3 puntos propuestos al cierre de la Sección 7, cada uno por separado con su propia conclusión.

### Punto 1: Consenso entre Ps — ¿más Ps de acuerdo = más confiable?

`consensus_count` = cuántos de los 5 Ps (con score≠0) coinciden en signo con la dirección final (`sign(calc_edge)`). Se cruza contra `is_valid` (`specific_bias_compliance=='Valid'`).

In [155]:
px_df['final_sign'] = np.sign(px_df['calc_edge'])

def consensus(row):
    return sum(1 for c in score_cols if row[c] != 0 and np.sign(row[c]) == row['final_sign'])

px_df['consensus_count'] = px_df.apply(consensus, axis=1)
px_df['is_valid'] = (px_df['specific_bias_compliance'] == 'Valid').astype(int)

resolved_df = px_df[px_df['specific_bias_compliance'].notna()].copy()
win_rate_by_consensus = resolved_df.groupby('consensus_count')['is_valid'].agg(N='count', Win_Rate='mean')
win_rate_by_consensus['Win_Rate_%'] = (win_rate_by_consensus['Win_Rate'] * 100).round(1)

display(win_rate_by_consensus[['N', 'Win_Rate_%']])

,N,Win_Rate_%
consensus_count,,
0,1,100.0000
1,8,50.0000
2,19,57.9000
3,23,69.6000
4,15,80.0000
5,4,100.0000


**Conclusión Punto 1:** patrón monótono y limpio — Win Rate sube de 50% (consensus=1, n=8) a 56.3% (2, n=16) a 70% (3, n=20) a 84.6% (4, n=13) a 100% (5, n=4). `consensus_count` es una señal fuerte y no explotada: mientras más Ps apuntan en la misma dirección que el resultado final, más confiable es el setup — independiente de cuál P específico esté opinando. (consensus=0, n=1, se ignora por muestra insuficiente.)

### Punto 2: ¿El edge de P0 se sostiene parejo entre `bias_a`, o está concentrado en uno solo?

In [156]:
active_p0 = resolved_df[resolved_df['p0_score'] != 0].copy()
active_p0 = active_p0.merge(efficiency_df[['id', 'bias_a']], on='id', how='left')
active_p0['p0_correct'] = (np.sign(active_p0['p0_score']) == active_p0['Market_Outcome']).astype(int)

p0_by_bias_a = active_p0.groupby('bias_a')['p0_correct'].agg(N='count', Win_Rate='mean').sort_values('N', ascending=False)
p0_by_bias_a['Win_Rate_%'] = (p0_by_bias_a['Win_Rate'] * 100).round(1)

display(p0_by_bias_a[['N', 'Win_Rate_%']])

,N,Win_Rate_%
bias_a,,
BOS,19,73.7000
CHOCH,8,50.0000
No_Bias(Choppy),4,0.0000
Trend Reversal,4,50.0000
Choppy-Bearish Range Rotation,1,100.0000
Validated Range Expansion,1,100.0000


**Conclusión Punto 2:** el edge de P0 (63.3% global, Sección 7) NO es parejo — está concentrado en `BOS` (85.7%, n=14, el más numeroso y encima el mejor). En `CHOCH` cae a 42.9% (n=7, peor que moneda al aire) y en `No_Bias(Choppy)` a 0% (n=4). `Trend Reversal` da 66.7% pero n=3, no alcanza. Implicación: el "P0 confiable" del análisis global es en realidad "P0 confiable cuando `bias_a=='BOS'`" — usarlo igual en CHOCH sería sobreestimar su confiabilidad ahí.

### Punto 3: Simulación — ¿mejoraría la precisión direccional con otros pesos?

Se recalcula el signo de `calc_edge` con distintos juegos de pesos (los actuales son P0=0.30, P1=0.25, P2=0.15, P3=0.10, P4=0.20, `core/math_engine.py::calculate_edge_score`) y se compara contra `Market_Outcome` real — solo en los 33 setups con `market_bias` Bullish/Bearish y resueltos (donde `Market_Outcome` no es 0, para no contaminar con el caso trivial de Choppy visto en la Sección 7).

In [157]:
eval_df = px_df[px_df['Market_Outcome'] != 0].copy()
print(f"n evaluable: {len(eval_df)}")

weight_sets = {
    'Actual (0.30,0.25,0.15,0.10,0.20)': (0.30, 0.25, 0.15, 0.10, 0.20),
    'P0+P1 solo (0.55,0.45,0,0,0)': (0.55, 0.45, 0, 0, 0),
    'P0 solo (1,0,0,0,0)': (1, 0, 0, 0, 0),
    'P0 dominante (0.7,0.3,0,0,0)': (0.7, 0.3, 0, 0, 0),
}

sim_results = []
for name, (w0, w1, w2, w3, w4) in weight_sets.items():
    sim = (w0*eval_df['p0_score'] + w1*eval_df['p1_score'] + w2*eval_df['p2_score']
           + w3*eval_df['p3_score'] + w4*eval_df['p4_score'])
    sim_sign = np.sign(sim)
    correct = (sim_sign == eval_df['Market_Outcome'])
    sim_results.append({
        'Pesos': name,
        'N': len(eval_df),
        'Aciertos': int(correct.sum()),
        'Accuracy_%': round(correct.mean() * 100, 1),
        'Indeterminados (sim=0)': int((sim_sign == 0).sum()),
    })

display(pd.DataFrame(sim_results))

n evaluable: 38


,Pesos,N,Aciertos,Accuracy_%,Indeterminados (sim=0)
0,"Actual (0.30,0.25,0.15,0.10,0.20)",38,29,76.3000,0
1,"P0+P1 solo (0.55,0.45,0,0,0)",38,29,76.3000,0
2,"P0 solo (1,0,0,0,0)",38,22,57.9000,11
3,"P0 dominante (0.7,0.3,0,0,0)",38,29,76.3000,0


**Conclusión Punto 3:** los pesos actuales (78.8%, 26/33) y "P0+P1 solo" dan **exactamente la misma accuracy** (78.8%) — en esta muestra, P2/P3/P4 no están rescatando ni arruinando ningún signo, son pasajeros neutros en el resultado final (no en el valor de `calc_edge`, que sí cambia, sino en su signo, que es lo único que importa para la dirección). "P0 dominante" (0.7/0.3) también empata en 78.8%. **"P0 solo" es la peor** (57.6%) porque deja **11 de 33 casos indeterminados** (P0=0, sin opinión) — sin P1 de respaldo, el sistema se queda mudo en un tercio de los casos reales. Conclusión práctica: simplificar a P0+P1 no perdería nada de precisión con los datos actuales, pero tampoco hay evidencia de que cambiar los pesos actuales mejore el resultado — el problema no está en la ponderación de P2-P4, está en que están presentes.

## Sección 9: Segmentando la thesis de P0 por temporalidad (regex)

El texto de `analysis_layer.thesis` para `layer_name=='P0'` sigue un patrón informal: `"{temporalidad}-B{L|R}"` (ej. `"1DT-BR"`, `"4HT-BL"`) — `L`=Bullish, `R`=Bearish, por cada una de las 4 temporalidades que usa el analista (1DT, 12H, 4H, 1H).

**Antes de asumir el patrón, se verificó contra el texto real** (65/65 thesis de P0 tienen al menos 1 match) y se encontró que la notación **no es consistente** — mismo timeframe, distintas abreviaturas según cuándo se escribió el análisis:

| temporalidad | variantes encontradas en el texto | conteo |
|---|---|---|
| 1DT | `1DT` | 61 |
| 12H | `12HT` (62), `12H` (2), `12T` (2) | 66 |
| 4H | `4HT` (61), `4H` (2), `4T` (2) | 65 |
| 1H | `1HT` (57), `1H` (2) | 59 |
| (otro) | `15M` (1) — fuera de las 4 temporalidades pedidas, se excluye | — |

Se normalizan todas las variantes a las 4 etiquetas canónicas. Se verificó que ningún thesis tiene la misma temporalidad mencionada dos veces con direcciones distintas (0 conflictos en 65 filas) — es seguro tomar la primera mención.

In [158]:
import re

TF_PATTERN = re.compile(r'(\d+[A-Za-z]*)\s*-\s*B([LR])', re.IGNORECASE)
TFS = ['1DT', '12H', '4H', '1H']

def normalize_tf(tf):
    tf = tf.upper()
    if tf == '1DT':
        return '1DT'
    if tf in ('12HT', '12H', '12T'):
        return '12H'
    if tf in ('4HT', '4H', '4T'):
        return '4H'
    if tf in ('1HT', '1H', '1T'):
        return '1H'
    return None  # ej. 15M, fuera de las 4 pedidas

layer_p0 = layer_df[layer_df['layer_name'] == 'P0'][['trade_id', 'score', 'thesis']].copy()

records = []
for _, r in layer_p0.iterrows():
    found = {}
    for tf_raw, direction in TF_PATTERN.findall(r['thesis']):
        tf = normalize_tf(tf_raw)
        if tf and tf not in found:
            found[tf] = 1 if direction.upper() == 'L' else -1
    rec = {'id': r['trade_id'], 'p0_score': r['score']}
    for tf in TFS:
        rec[f'tf_{tf}'] = found.get(tf, np.nan)
    records.append(rec)

tf_df = pd.DataFrame(records)
print("Cobertura por temporalidad (de 65 thesis de P0):")
display(tf_df[[f'tf_{tf}' for tf in TFS]].notna().sum().to_frame('N_con_match'))

Cobertura por temporalidad (de 65 thesis de P0):


,N_con_match
tf_1DT,66
tf_12H,70
tf_4H,70
tf_1H,64


### Convergencia: ¿la temporalidad coincide con la dirección final del propio P0?

In [159]:
p0_active_tf = tf_df[tf_df['p0_score'] != 0].copy()
p0_active_tf['p0_dir'] = np.sign(p0_active_tf['p0_score'])

conv_rows = []
for tf in TFS:
    sub = p0_active_tf[p0_active_tf[f'tf_{tf}'].notna()]
    match = (sub[f'tf_{tf}'] == sub['p0_dir']).mean()
    conv_rows.append({'Temporalidad': tf, 'N': len(sub), 'Convergencia_%': round(match * 100, 1)})

display(pd.DataFrame(conv_rows))

,Temporalidad,N,Convergencia_%
0,1DT,35,80.0000
1,12H,37,81.1000
2,4H,37,81.1000
3,1H,35,62.9000


### Win Rate: ¿qué temporalidad, tomada como señal propia, acierta más contra el resultado real?

In [160]:
tf_full = tf_df.merge(unified_df[['id', 'market_bias', 'calc_edge']], on='id', how='left')
tf_full = tf_full.merge(efficiency_df[['id', 'specific_bias_compliance']], on='id', how='left')
tf_full = tf_full[tf_full['specific_bias_compliance'].notna()].copy()
tf_full['Market_Outcome'] = tf_full.apply(get_market_outcome, axis=1)

tf_eval = tf_full[tf_full['Market_Outcome'] != 0]
print(f"n evaluable (market_bias Bullish/Bearish, resuelto): {len(tf_eval)}")

wr_rows = []
for tf in TFS:
    sub = tf_eval[tf_eval[f'tf_{tf}'].notna()]
    correct = (sub[f'tf_{tf}'] == sub['Market_Outcome'])
    wr_rows.append({'Temporalidad': tf, 'N': len(sub), 'Win_Rate_%': round(correct.mean() * 100, 1)})

display(pd.DataFrame(wr_rows).sort_values('Win_Rate_%', ascending=False))

n evaluable (market_bias Bullish/Bearish, resuelto): 38


,Temporalidad,N,Win_Rate_%
3,1H,36,66.7000
0,1DT,38,60.5000
1,12H,38,60.5000
2,4H,38,60.5000


### Lectura de los resultados

**Convergencia con la dirección final de P0:** 1DT (80.6%, n=31), 12H (81.8%, n=33) y 4H (81.8%, n=33) están muy alineadas con lo que P0 termina diciendo — pero **1H converge mucho menos (61.3%, n=31)**. El timeframe más bajo es, dentro del propio texto de P0, el que más seguido "dice algo distinto" al veredicto final.

**Win Rate contra el resultado real — el hallazgo no es el esperado:** 1DT, 12H y 4H empatan en 60.6% (20/33) — y en esta muestra de 33 evaluables, **las tres son literalmente idénticas fila por fila** (mismo signo en las 33), es decir, no aportan información independiente entre sí, son una sola señal repetida 3 veces con nombres distintos (verificado directo contra el texto, no es un bug de parseo — sí hay 9 filas del total de 65 donde disienten, pero ninguna cae dentro de este subconjunto de 33 evaluables). **`1H` es la única temporalidad realmente distinta, y es la que mejor acierta: 67.7% (21/31)** — a pesar de ser la que menos converge con el veredicto final de P0 (61.3% arriba).

**Conclusión:** hay una desconexión real — la temporalidad que más se ignora al formar el score final de P0 (`1H`) es la que mejor predice el resultado real, mientras que 1DT/12H/4H, que dominan el consenso interno, no aportan nada más allá de repetir la misma señal tres veces. Es la misma dinámica que ya vimos con `P2` en la Sección 7 (Métrica "Rescate") y con `mae_R` en la Sección 6: hay señal de valor que el sistema actual está sub-ponderando o ignorando dentro de su propio proceso de decisión.

## Sección 10: Profundizando en 1H

Tres preguntas: (1) cuando `1H` discrepa de las otras 3 temporalidades de su propia thesis, ¿tenía razón? (2) ¿el acierto de `1H` también está concentrado en un `bias_a` específico, como pasó con P0? (3) cara a cara contra el propio `p0_score` — ¿`1H` es mejor que el juicio final de P0, o solo mejor que las otras temporalidades?

In [161]:
def majority_htf(row):
    vals = [v for v in [row['tf_1DT'], row['tf_12H'], row['tf_4H']] if pd.notna(v)]
    if not vals:
        return np.nan
    return 1 if sum(vals) > 0 else (-1 if sum(vals) < 0 else 0)

tf_full['htf_majority'] = tf_full.apply(majority_htf, axis=1)

has_both = tf_full['tf_1H'].notna() & tf_full['htf_majority'].notna()
tf_disagree = tf_full[has_both & (tf_full['tf_1H'] != tf_full['htf_majority'])]
tf_agree = tf_full[has_both & (tf_full['tf_1H'] == tf_full['htf_majority'])]

ev_dis = tf_disagree[tf_disagree['Market_Outcome'] != 0]
ev_agr = tf_agree[tf_agree['Market_Outcome'] != 0]

print(f"Cuando 1H DISCREPA de la mayoría 1DT/12H/4H (n={len(ev_dis)}):")
print(f"  1H acierta: {(ev_dis['tf_1H'] == ev_dis['Market_Outcome']).mean():.1%}")
print(f"  La mayoría 1DT/12H/4H acierta: {(ev_dis['htf_majority'] == ev_dis['Market_Outcome']).mean():.1%}")
print(f"\nCuando 1H COINCIDE con la mayoría (n={len(ev_agr)}):")
print(f"  Acierto conjunto: {(ev_agr['tf_1H'] == ev_agr['Market_Outcome']).mean():.1%}")

Cuando 1H DISCREPA de la mayoría 1DT/12H/4H (n=9):
  1H acierta: 66.7%
  La mayoría 1DT/12H/4H acierta: 33.3%

Cuando 1H COINCIDE con la mayoría (n=27):
  Acierto conjunto: 66.7%


### 1H por `bias_a` — ¿el patrón se repite?

In [162]:
sub_1h = tf_eval[tf_eval['tf_1H'].notna()].merge(efficiency_df[['id', 'bias_a']], on='id', how='left').copy()
sub_1h['correct'] = (sub_1h['tf_1H'] == sub_1h['Market_Outcome']).astype(int)

wr_1h_bias_a = sub_1h.groupby('bias_a')['correct'].agg(N='count', Win_Rate='mean').sort_values('N', ascending=False)
wr_1h_bias_a['Win_Rate_%'] = (wr_1h_bias_a['Win_Rate'] * 100).round(1)
display(wr_1h_bias_a[['N', 'Win_Rate_%']])

,N,Win_Rate_%
bias_a,,
BOS,21,81.0000
CHOCH,8,37.5000
Trend Reversal,5,60.0000
Choppy-Bearish Range Rotation,1,100.0000
Validated Range Expansion,1,0.0000


### 1H vs `p0_score` — cara a cara, en el mismo subconjunto

In [163]:
head_to_head = tf_eval[(tf_eval['p0_score'] != 0) & (tf_eval['tf_1H'].notna())].copy()
head_to_head['p0_dir'] = np.sign(head_to_head['p0_score'])

print(f"n = {len(head_to_head)}")
print(f"P0 (score final) accuracy: {(head_to_head['p0_dir'] == head_to_head['Market_Outcome']).mean():.1%}")
print(f"1H (solo) accuracy: {(head_to_head['tf_1H'] == head_to_head['Market_Outcome']).mean():.1%}")

agree_p0 = head_to_head[head_to_head['p0_dir'] == head_to_head['tf_1H']]
disagree_p0 = head_to_head[head_to_head['p0_dir'] != head_to_head['tf_1H']]
print(f"\nP0 y 1H coinciden en {len(agree_p0)}/{len(head_to_head)} casos -> accuracy conjunta: {(agree_p0['p0_dir'] == agree_p0['Market_Outcome']).mean():.1%}")
print(f"P0 y 1H discrepan en {len(disagree_p0)}/{len(head_to_head)} casos:")
print(f"  P0 acierta: {(disagree_p0['p0_dir'] == disagree_p0['Market_Outcome']).mean():.1%}")
print(f"  1H acierta: {(disagree_p0['tf_1H'] == disagree_p0['Market_Outcome']).mean():.1%}")

n = 25
P0 (score final) accuracy: 80.0%
1H (solo) accuracy: 64.0%

P0 y 1H coinciden en 15/25 casos -> accuracy conjunta: 86.7%
P0 y 1H discrepan en 10/25 casos:
  P0 acierta: 70.0%
  1H acierta: 30.0%


### Lectura de los resultados — matiza el hallazgo de la Sección 9

**(1) 1H vs las otras 3 temporalidades:** cuando `1H` discrepa de la mayoría `1DT`/`12H`/`4H` (n=9), **1H acierta 66.7% contra solo 33.3% de la mayoría** — confirma la Sección 9: 1H sí aporta información que las otras 3 (colineales entre sí) no tienen. Cuando coinciden (n=22), el acierto conjunto es 68.2%, la línea base normal.

**(2) 1H por `bias_a`:** mismo patrón que P0 — BOS 82.4% (n=17), CHOCH 42.9% (n=7), Trend Reversal 60% (n=5). El acierto de 1H también está concentrado en BOS, no es una señal universal.

**(3) El matiz importante — 1H vs el propio `p0_score` (no solo contra las otras temporalidades):** acá la historia cambia. En el mismo subconjunto (n=20), **P0 acierta 85% y 1H solo 65%** — P0 sigue siendo mejor que 1H tomado solo. Y cuando específicamente discrepan entre sí (n=8), **P0 sigue ganando (75% vs 25% de 1H)**. Es decir: 1H le gana a 1DT/12H/4H (que son una sola señal repetida 3 veces), pero **no le gana al juicio final de P0** — el `p0_score` ya parece incorporar más contexto que el tag literal de 1H solo.

**Conclusión revisada:** el hallazgo de la Sección 9 sigue siendo válido en su forma original (1H > 1DT/12H/4H), pero no implica que 1H deba reemplazar o corregir a P0 — P0 sigue siendo la mejor señal disponible, incluso cuando 1H no está de acuerdo con él. El valor práctico de 1H es más limitado de lo que parecía: sirve como diagnóstico de que 1DT/12H/4H son redundantes entre sí, no como una señal para pisar el juicio de P0.

## Sección 11: ¿Llega o no llega a `structural_entry_price`? Y el "sweet spot" de entrada

Tres preguntas: (1) cuántas veces el precio llega vs no llega a la entrada teórica, por `market_bias`; (2) lo mismo, por tercil de `primary_prob`; (3) el "sweet spot" — dentro de todo el rango posible (desde "nunca llega" hasta "la atraviesa con fuerza"), ¿qué tramo de `mae_R` da el mejor Win Rate?

Reusa `checkable` (Sección 5 — setups con `theoretical_dir`, `structural_mae` y `structural_entry_price` disponibles) y le suma `mae_R` (Sección 6).

In [164]:
checkable = checkable.merge(struct_df[['id', 'mae_R']], on='id', how='left')

reach_by_bias = checkable.groupby(['market_bias', 'reached_entry']).size().unstack(fill_value=0)
reach_by_bias = reach_by_bias.rename(columns={False: 'No_Llega', True: 'Llega'})
reach_by_bias['Llega_%'] = (reach_by_bias['Llega'] / (reach_by_bias['Llega'] + reach_by_bias['No_Llega']) * 100).round(1)

display(reach_by_bias)

reached_entry,No_Llega,Llega,Llega_%
market_bias,,,
Bearish,9,16,64.0000
Bullish,7,6,46.2000
Choppy / Neutral,11,15,57.7000


### Por tercil de `primary_prob`

In [165]:
entry_prob_bins = pd.qcut(checkable['primary_prob'], q=3, duplicates='drop')

reach_by_prob = checkable.groupby([entry_prob_bins, 'reached_entry'], observed=False).size().unstack(fill_value=0)
reach_by_prob = reach_by_prob.rename(columns={False: 'No_Llega', True: 'Llega'})
reach_by_prob['Llega_%'] = (reach_by_prob['Llega'] / (reach_by_prob['Llega'] + reach_by_prob['No_Llega']) * 100).round(1)

display(reach_by_prob)

reached_entry,No_Llega,Llega,Llega_%
primary_prob,,,
"(0.105, 0.164]",11,14,56.0000
"(0.164, 0.233]",8,11,57.9000
"(0.233, 0.408]",8,12,60.0000


### El "sweet spot" — Win Rate por tramo fino de `mae_R`

Se restringe a los setups resueltos (`specific_bias_compliance` no nulo). Bins más finos que el tercil de la Sección 6, para ubicar el punto exacto donde el Win Rate se cae.

In [166]:
entry_resolved = checkable.merge(efficiency_df[['id', 'specific_bias_compliance']], on='id', how='left')
entry_resolved = entry_resolved[entry_resolved['specific_bias_compliance'].notna()].copy()
entry_resolved['is_valid'] = (entry_resolved['specific_bias_compliance'] == 'Valid').astype(int)

bin_edges = [-10, -0.01, 0.15, 0.35, 0.6, 0.9, 1.3, 10]
bin_labels = ['No llega (<0)', '0 a 0.15R', '0.15 a 0.35R', '0.35 a 0.6R', '0.6 a 0.9R', '0.9 a 1.3R', '>1.3R']
entry_resolved['mae_R_fino'] = pd.cut(entry_resolved['mae_R'], bins=bin_edges, labels=bin_labels)

sweet_spot = entry_resolved.groupby(['mae_R_fino','market_bias'], observed=False)['is_valid'].agg(N='count', Win_Rate='mean')
sweet_spot['Win_Rate_%'] = (sweet_spot['Win_Rate'] * 100).round(1)

print(f"n total resuelto con mae_R: {len(entry_resolved)}")
display(sweet_spot[['N', 'Win_Rate_%']])

n total resuelto con mae_R: 64


N  Win_Rate_%
mae_R_fino    market_bias                     
No llega (<0) Bearish            9    100.0000
              Bullish            7    100.0000
              Choppy / Neutral  10     70.0000
0 a 0.15R     Bearish            5    100.0000
              Bullish            0         NaN
              Choppy / Neutral   2    100.0000
0.15 a 0.35R  Bearish            1    100.0000
              Bullish            1    100.0000
              Choppy / Neutral   2     50.0000
0.35 a 0.6R   Bearish            1    100.0000
              Bullish            0         NaN
              Choppy / Neutral   3     66.7000
0.6 a 0.9R    Bearish            2    100.0000
              Bullish            2    100.0000
              Choppy / Neutral   0         NaN
0.9 a 1.3R    Bearish            1    100.0000
              Bullish            1      0.0000
              Choppy / Neutral   3      0.0000
>1.3R         Bearish            6      0.0000
              Bullish            2      0.0000
              Choppy / Neutral   6     16.7000

In [167]:
sweet_spot_filtered_bias = entry_resolved.groupby(['mae_R_fino','direction_from_bias'], observed=False)['is_valid'].agg(N='count', Win_Rate='mean')
sweet_spot_filtered_bias['Win_Rate_%'] = (sweet_spot_filtered_bias['Win_Rate'] * 100).round(1)

print(f"N Bullish/Bearish resuelto con mae_R: {(sweet_spot_filtered_bias['N'].sum())}")
display(sweet_spot_filtered_bias[['N', 'Win_Rate_%']])

N Bullish/Bearish resuelto con mae_R: 38


N  Win_Rate_%
mae_R_fino    direction_from_bias               
No llega (<0) Long                 7    100.0000
              Short                9    100.0000
0 a 0.15R     Long                 0         NaN
              Short                5    100.0000
0.15 a 0.35R  Long                 1    100.0000
              Short                1    100.0000
0.35 a 0.6R   Long                 0         NaN
              Short                1    100.0000
0.6 a 0.9R    Long                 2    100.0000
              Short                2    100.0000
0.9 a 1.3R    Long                 1      0.0000
              Short                1    100.0000
>1.3R         Long                 2      0.0000
              Short                6      0.0000

### Lectura de los resultados

**Llega vs no llega, por `market_bias`:** las 3 categorías están parejas (Bearish 61.1%, Bullish 66.7%, Choppy 68.8% de las veces llega) — coincide con la Sección 5, no hay un bias que tenga más problema geométrico para alcanzar su propia entrada.

**Por tercil de probabilidad:** no es monótono — el tercil bajo llega 66.7%, el medio 71.4% (el más alto), el alto solo 54.5%. La probabilidad del modelo no predice si el precio hará el pullback o no; son cosas independientes.

**El "sweet spot" — el hallazgo central de la sección:** no es una ventana angosta, es un **techo**. Desde "nunca llega a la entrada" (`mae_R<0`) hasta "~0.9R más allá de la entrada", el Win Rate se mantiene alto y estable:
- No llega: 85.7% (n=14)
- 0 a 0.9R de sobrepaso: 100% en los 4 sub-tramos (n=5, 3, 3, 2 respectivamente)

Y ahí hay un **quiebre marcado justo en ~0.9R**:
- 0.9 a 1.3R: cae a 25.0% (n=4)
- Más de 1.3R: 11.1% (n=9)

**Implicación práctica — corrige la intuición típica:** no hace falta que el precio "toque justo" tu entrada para que el trade sea bueno — de hecho, ni siquiera llegar (85.7%) es casi tan bueno como llegar limpio. Lo que de verdad importa es **no dejar que el sobrepaso pase de ~0.9-1.0R** — ese es el techo real, no un rango ideal estrecho. Es consistente con el corte de ~0.8R que ya habíamos visto en tercios en la Sección 6, ahora con más resolución. Ojo con el tamaño de muestra en los tramos finos (2-9 casos cada uno) — el quiebre es consistente en dos granularidades distintas (tercios y estos 7 tramos), pero no es un número que deba tratarse como definitivo con esta cantidad de datos.

## Sección 12: Tabla de referencia — MAE y MFE estructural esperado por Bias, ICD y Probabilidad

Objetivo: dado un `market_bias`, una magnitud de `calc_edge` (ICD) o un rango de `primary_prob`, tener una referencia de cuánto MAE/MFE esperar — media y mediana, crudo (puntos de precio) y normalizado a R.

**Aclaración importante antes de la tabla, para no repetir el error de la Sección 5:** `structural_mae`/`structural_mfe` crudos son precios absolutos de XAUUSD en fechas distintas (~4000 a ~4600 en el período) — la media/mediana cruda solo sirve como referencia de "en qué zona de precio suele moverse", no es comparable como magnitud de movimiento entre bias/fechas. Para eso está la versión en R (`mae_R`/`mfe_R`, Sección 6) al lado — esa sí es la comparable y accionable.

In [168]:
RR = 2.2
ref_df = unified_df.merge(efficiency_df[['id', 'structural_mae', 'structural_mfe']], on='id', how='left')
ref_df['structural_entry_price'] = np.where(
    ref_df['edge_validation_price'].notna() & ref_df['structural_invalidation'].notna(),
    (ref_df['edge_validation_price'] + RR * ref_df['structural_invalidation']) / (1 + RR),
    np.nan
)
ref_df['theoretical_dir'] = np.where(ref_df['calc_edge'] >= 0, 'Long', 'Short')
ref_df['primary_prob'] = np.where(ref_df['theoretical_dir'] == 'Long', ref_df['long_prob'], ref_df['short_prob'])
ref_df['one_r'] = (ref_df['structural_entry_price'] - ref_df['structural_invalidation']).abs()
ref_df['mae_past_entry'] = np.where(ref_df['theoretical_dir'] == 'Long', ref_df['structural_entry_price'] - ref_df['structural_mae'], ref_df['structural_mae'] - ref_df['structural_entry_price'])
ref_df['mfe_past_entry'] = np.where(ref_df['theoretical_dir'] == 'Long', ref_df['structural_mfe'] - ref_df['structural_entry_price'], ref_df['structural_entry_price'] - ref_df['structural_mfe'])
ref_df['mae_R'] = ref_df['mae_past_entry'] / ref_df['one_r']
ref_df['mfe_R'] = ref_df['mfe_past_entry'] / ref_df['one_r']
ref_df['edge_mag'] = ref_df['calc_edge'].abs()

print("--- Por market_bias ---")
display(ref_df.groupby('market_bias')[['structural_mae', 'structural_mfe', 'mae_R', 'mfe_R']].agg(['count', 'mean', 'median']).round(2))

--- Por market_bias ---


structural_mae                     structural_mfe                     mae_R                mfe_R              
                          count      mean    median          count      mean    median count   mean  median count   mean median
market_bias                                                                                                                    
Bearish                      25 4211.9100 4148.0000             25 4124.1400 4044.7500    25 0.9100  0.0900    25 2.2100 2.6500
Bullish                      13 4209.5100 4181.0000             13 4305.5200 4303.0000    13 0.6300 -0.1400    13 2.7400 2.3600
Choppy / Neutral             26 4304.1300 4361.5000             26 4315.5600 4356.5000    26 0.8100  0.2600    26 2.4300 1.7200

### Por magnitud de `calc_edge` (ICD)

In [169]:
edge_bins = pd.qcut(ref_df['edge_mag'], q=3, duplicates='drop')
display(ref_df.groupby(edge_bins, observed=False)[['structural_mae', 'structural_mfe', 'mae_R', 'mfe_R']].agg(['count', 'mean', 'median']).round(2))

structural_mae                     structural_mfe                     mae_R               mfe_R              
                        count      mean    median          count      mean    median count   mean median count   mean median
edge_mag                                                                                                                    
(-0.001, 0.2]              19 4299.7700 4361.0000             19 4304.5000 4353.0000    19 1.1500 0.4400    19 2.4500 1.6000
(0.2, 0.375]               22 4229.1500 4174.4800             22 4229.3000 4200.5800    22 0.6000 0.0200    22 1.9800 2.3800
(0.375, 0.825]             23 4225.7300 4181.0000             23 4193.4600 4115.0000    23 0.7400 0.0800    23 2.7900 2.7900

### Por rango de `primary_prob`

In [170]:
ref_prob_bins = pd.qcut(ref_df['primary_prob'], q=3, duplicates='drop')
display(ref_df.groupby(ref_prob_bins, observed=False)[['structural_mae', 'structural_mfe', 'mae_R', 'mfe_R']].agg(['count', 'mean', 'median']).round(2))

# Verificación: ¿los bins de calc_edge y de primary_prob son la misma partición de filas?
print(f"¿Mismos bins fila por fila?: {(edge_bins.cat.codes == ref_prob_bins.cat.codes).all()}")

structural_mae                     structural_mfe                     mae_R               mfe_R              
                        count      mean    median          count      mean    median count   mean median count   mean median
primary_prob                                                                                                                
(0.099, 0.156]             19 4299.7700 4361.0000             19 4304.5000 4353.0000    19 1.1500 0.4400    19 2.4500 1.6000
(0.156, 0.215]             22 4229.1500 4174.4800             22 4229.3000 4200.5800    22 0.6000 0.0200    22 1.9800 2.3800
(0.215, 0.408]             23 4225.7300 4181.0000             23 4193.4600 4115.0000    23 0.7400 0.0800    23 2.7900 2.7900

¿Mismos bins fila por fila?: True


### Lectura de los resultados y cómo usarlos

**Los bins por magnitud de `|calc_edge|` y por `primary_prob` son la MISMA partición de filas** (verificado arriba, coincide exacto) — confirma otra vez la colinealidad ICD/probabilidad ya vista en la Sección 2. No hace falta mirar las dos tablas por separado, dicen lo mismo.

**Tabla de referencia (mediana, más robusta que la media con esta muestra):**

| Segmento | mae_R (mediana) | mfe_R (mediana) |
|---|---|---|
| Bearish | 0.08 | 3.05 |
| Bullish | 0.48 | 2.31 |
| Choppy | 0.40 | 1.69 |
| ICD/prob bajo (tercil 1) | 0.79 | 1.69 |
| ICD/prob medio (tercil 2) | 0.51 | 1.90 |
| ICD/prob alto (tercil 3) | 0.08 | 2.92 |

**Patrón:** a mayor `|calc_edge|`/`primary_prob`, menor `mae_R` esperado (0.79→0.51→0.08) y mayor `mfe_R` esperado (1.69→1.90→2.92) — coherente con todo lo visto hasta ahora: más confianza del modelo, movimiento más limpio.

**Importante — esto NO es luz verde para "entrar a más trades":** esta tabla te dice *cuánto esperar* si el setup se da, no *si conviene tomarlo*. La Sección 11 ya mostró que el Win Rate se desploma en cuanto `mae_R` supera ~0.9-1.0R, sin importar el bias o la probabilidad. Usar esta tabla junto con ese techo: por ejemplo, en Bearish la mediana de `mae_R` es 0.08 (muy por debajo del techo de 1R) — eso es lo que hace a Bearish un bias más seguro en la práctica, no que "tenga mejor probabilidad". Si vas a usar esto para decidir tomar más trades, el criterio debería ser "el bias/probabilidad de este setup históricamente se queda cómodo por debajo de 1R de `mae_R`", no "el MFE esperado es grande, así que vale la pena arriesgarse".

## Sección 14: Frecuencia de alcance/cruce de cada nivel de precio, por `market_bias` × ICD

Objetivo (Goal A del pedido original): para cada combinación de `market_bias` y tercil de magnitud de `calc_edge` (ICD, misma convención que la Sección 12), ¿con qué frecuencia el precio llega a cada uno de los 4 niveles de referencia — `mark_price`, `structural_entry_price`, el TP estructural (`edge_validation_price`), y el `entry_price` real (solo trades con `order_filled == True`)?

Las tablas ①-③ reusan la lógica direccional ya usada en Secciones 5/6/11 (`>=` para Short, `<=` para Long, según `theoretical_dir`); la ④ es nueva — compara `entry_price` real contra `structural_entry_price` para saber si el fill quedó mejor o peor que el pullback ideal.

In [171]:
struct_df['edge_mag_tercile'] = pd.qcut(struct_df['calc_edge'].abs(), q=3, duplicates='drop')

def _reach_rate_table(df, cond_col):
    out = df.groupby(['market_bias', 'edge_mag_tercile'], observed=False)[cond_col].agg(N_Verificable='count', Reached='mean')
    out['Reached_%'] = (out['Reached'] * 100).round(1)
    return out[['N_Verificable', 'Reached_%']].reset_index()

print(f"Total análisis: {len(struct_df)}")
display(struct_df.groupby('edge_mag_tercile', observed=False).size().rename('N').to_frame())

Total análisis: 70


,N
edge_mag_tercile,
"(-0.001, 0.2]",25
"(0.2, 0.375]",22
"(0.375, 0.825]",23


### Ampliando la muestra — `all_entries_df` (TODOS los registros con `entry_price`, incluye `order_filled=False`)

`tactical_audit.entry_price` es la entrada de precisión que el trader busca deliberadamente (para no tener que conformarse con `structural_entry_price`), y se carga en el wizard **independientemente de si la orden se llegó a llenar** — restringir a `order_filled==True` descarta justo los casos donde apuntó a un nivel y el mercado no se lo dio. `all_entries_df` reusa `struct_df` pero sin ese filtro, excluyendo únicamente los placeholders legacy (`entry_price==closing_price` y `entry_time==exit_time` simultáneos — documentados en `jupyter/p2_order_filled_reached_entry.md`, nunca fueron un target real). Se usa en los apartados `.B` de las Secciones 14, 15, 16 y 17.

In [172]:
with engine.connect() as conn:
    entry_extra_df = _load_tactical_slice(conn, ["entry_time", "exit_time", "closing_price"])

all_entries_df = struct_df.merge(entry_extra_df, on='id', how='left')
is_legacy_placeholder = (
    all_entries_df['closing_price'].notna()
    & (all_entries_df['entry_price'] == all_entries_df['closing_price'])
    & (all_entries_df['entry_time'] == all_entries_df['exit_time'])
)
# entry_price==0 es un placeholder de DB (NOT NULL, no un precio real -- XAUUSD nunca vale 0),
# del flujo "abortar por gates" que nunca pide entry_price. .notna() no lo detecta porque 0 no es NaN.
is_zero_placeholder = all_entries_df['entry_price'] == 0
all_entries_df = all_entries_df[
    all_entries_df['entry_price'].notna() & ~is_legacy_placeholder & ~is_zero_placeholder
].copy()

print(f"Placeholders legacy excluidos: {int(is_legacy_placeholder.sum())}")
print(f"Placeholders entry_price==0 excluidos: {int(is_zero_placeholder.sum())}")
print(f"Total en all_entries_df: {len(all_entries_df)} (order_filled=True: {int((all_entries_df['order_filled'] == True).sum())}, "
      f"order_filled=False: {int((all_entries_df['order_filled'] == False).sum())})")

Placeholders legacy excluidos: 13
Placeholders entry_price==0 excluidos: 9
Total en all_entries_df: 30 (order_filled=True: 28, order_filled=False: 2)


### ① `mark_price` cruzó `structural_entry_price` (generaliza Sección 6, Fase 3)

In [173]:
timing_df = struct_df[
    struct_df['mark_price'].notna() & struct_df['structural_entry_price'].notna() & struct_df['theoretical_dir'].notna()
].copy()
timing_df['crossed_entry'] = np.where(
    timing_df['theoretical_dir'] == 'Long',
    timing_df['mark_price'] <= timing_df['structural_entry_price'],
    timing_df['mark_price'] >= timing_df['structural_entry_price'],
)
display(_reach_rate_table(timing_df, 'crossed_entry'))

,market_bias,edge_mag_tercile,N_Verificable,Reached_%
0,Bearish,"(-0.001, 0.2]",0,NaN
1,Bearish,"(0.2, 0.375]",10,0.0000
2,Bearish,"(0.375, 0.825]",15,13.3000
3,Bullish,"(-0.001, 0.2]",0,NaN
4,Bullish,"(0.2, 0.375]",5,0.0000
5,Bullish,"(0.375, 0.825]",8,0.0000
6,Choppy / Neutral,"(-0.001, 0.2]",20,10.0000
7,Choppy / Neutral,"(0.2, 0.375]",7,14.3000
8,Choppy / Neutral,"(0.375, 0.825]",0,NaN


### ② `structural_mae` alcanzó `structural_entry_price` (generaliza Secciones 5/11)

In [174]:
checkable_mae = struct_df[
    struct_df['theoretical_dir'].notna() & struct_df['structural_mae'].notna() & struct_df['structural_entry_price'].notna()
].copy()
checkable_mae['reached_entry'] = np.where(
    checkable_mae['theoretical_dir'] == 'Short',
    checkable_mae['structural_mae'] >= checkable_mae['structural_entry_price'],
    checkable_mae['structural_mae'] <= checkable_mae['structural_entry_price'],
)
display(_reach_rate_table(checkable_mae, 'reached_entry'))

,market_bias,edge_mag_tercile,N_Verificable,Reached_%
0,Bearish,"(-0.001, 0.2]",0,NaN
1,Bearish,"(0.2, 0.375]",10,70.0000
2,Bearish,"(0.375, 0.825]",15,60.0000
3,Bullish,"(-0.001, 0.2]",0,NaN
4,Bullish,"(0.2, 0.375]",5,40.0000
5,Bullish,"(0.375, 0.825]",8,50.0000
6,Choppy / Neutral,"(-0.001, 0.2]",19,68.4000
7,Choppy / Neutral,"(0.2, 0.375]",7,28.6000
8,Choppy / Neutral,"(0.375, 0.825]",0,NaN


### ③ `structural_mfe` alcanzó `edge_validation_price` (TP) — generaliza Sección 4.A, sin restringir a `Valid`

In [175]:
checkable_mfe = struct_df[
    struct_df['theoretical_dir'].notna() & struct_df['structural_mfe'].notna() & struct_df['edge_validation_price'].notna()
].copy()
checkable_mfe['reached_tp'] = np.where(
    checkable_mfe['theoretical_dir'] == 'Long',
    checkable_mfe['structural_mfe'] >= checkable_mfe['edge_validation_price'],
    checkable_mfe['structural_mfe'] <= checkable_mfe['edge_validation_price'],
)
display(_reach_rate_table(checkable_mfe, 'reached_tp'))

,market_bias,edge_mag_tercile,N_Verificable,Reached_%
0,Bearish,"(-0.001, 0.2]",0,NaN
1,Bearish,"(0.2, 0.375]",10,60.0000
2,Bearish,"(0.375, 0.825]",15,86.7000
3,Bullish,"(-0.001, 0.2]",0,NaN
4,Bullish,"(0.2, 0.375]",5,80.0000
5,Bullish,"(0.375, 0.825]",8,62.5000
6,Choppy / Neutral,"(-0.001, 0.2]",19,42.1000
7,Choppy / Neutral,"(0.2, 0.375]",7,57.1000
8,Choppy / Neutral,"(0.375, 0.825]",0,NaN


### ④ Nuevo — `entry_price` real vs. `structural_entry_price`: ¿el fill quedó peor que el pullback ideal?

Solo trades con `order_filled == True` (no basta `entry_price.notna()` — ver nota de la Sección 5). `entry_worse_than_structural=True` significa: Long entró a un precio >= al pullback ideal (pagó de más, no esperó el pullback completo); Short entró a un precio <= al pullback ideal (vendió más barato de lo ideal).

In [176]:
checkable_fill = struct_df[
    (struct_df['order_filled'] == True)
    & struct_df['theoretical_dir'].notna() & struct_df['entry_price'].notna() & struct_df['structural_entry_price'].notna()
].copy()
checkable_fill['entry_worse_than_structural'] = np.where(
    checkable_fill['theoretical_dir'] == 'Long',
    checkable_fill['entry_price'] >= checkable_fill['structural_entry_price'],
    checkable_fill['entry_price'] <= checkable_fill['structural_entry_price'],
)
display(_reach_rate_table(checkable_fill, 'entry_worse_than_structural').rename(columns={'Reached_%': 'Entro_Peor_%'}))

,market_bias,edge_mag_tercile,N_Verificable,Entro_Peor_%
0,Bearish,"(-0.001, 0.2]",0,NaN
1,Bearish,"(0.2, 0.375]",5,80.0000
2,Bearish,"(0.375, 0.825]",8,62.5000
3,Bullish,"(-0.001, 0.2]",0,NaN
4,Bullish,"(0.2, 0.375]",1,0.0000
5,Bullish,"(0.375, 0.825]",3,66.7000
6,Choppy / Neutral,"(-0.001, 0.2]",9,55.6000
7,Choppy / Neutral,"(0.2, 0.375]",2,100.0000
8,Choppy / Neutral,"(0.375, 0.825]",0,NaN


### ④.B — mismo cálculo, sobre `all_entries_df` (incluye `order_filled=False`)

In [177]:
checkable_fill_all = all_entries_df[
    all_entries_df['theoretical_dir'].notna() & all_entries_df['entry_price'].notna() & all_entries_df['structural_entry_price'].notna()
].copy()
checkable_fill_all['entry_worse_than_structural'] = np.where(
    checkable_fill_all['theoretical_dir'] == 'Long',
    checkable_fill_all['entry_price'] >= checkable_fill_all['structural_entry_price'],
    checkable_fill_all['entry_price'] <= checkable_fill_all['structural_entry_price'],
)
display(_reach_rate_table(checkable_fill_all, 'entry_worse_than_structural').rename(columns={'Reached_%': 'Entro_Peor_%'}))

,market_bias,edge_mag_tercile,N_Verificable,Entro_Peor_%
0,Bearish,"(-0.001, 0.2]",0,NaN
1,Bearish,"(0.2, 0.375]",7,71.4000
2,Bearish,"(0.375, 0.825]",8,62.5000
3,Bullish,"(-0.001, 0.2]",0,NaN
4,Bullish,"(0.2, 0.375]",1,0.0000
5,Bullish,"(0.375, 0.825]",3,66.7000
6,Choppy / Neutral,"(-0.001, 0.2]",9,55.6000
7,Choppy / Neutral,"(0.2, 0.375]",2,100.0000
8,Choppy / Neutral,"(0.375, 0.825]",0,NaN


### Lectura de los resultados

**① `mark_price` rara vez ya cruzó `structural_entry_price`** al momento del análisis (0-14% en casi todos los segmentos) — consistente con que la mayoría de los setups todavía están esperando el pullback cuando se hace el análisis, no llegando tarde.

**② `structural_mae` alcanza `structural_entry_price`** con más frecuencia en Bearish (60-70%) que en Bullish (40-66.7%) o Choppy (28.6-68.8%) — coincide con el hallazgo de la Sección 12 (Bearish tiene el `mae_R` mediano más bajo, movimientos más limpios hacia la entrada).

**③ `structural_mfe` alcanza el TP (`edge_validation_price`)** con tasas altas en varios segmentos (hasta 86.7% en Bearish ICD alto) — el potencial estructural sigue siendo generoso, en línea con la Sección 6.

**④ Nuevo — el fill real suele quedar peor que el pullback ideal:** en la mayoría de los segmentos con datos, `entry_price` real entró peor que `structural_entry_price` más del 55% de las veces (hasta 80-100% en algunos), y solo en el segmento Bullish ICD bajo (n=1) el fill fue mejor. **Esto es la primera evidencia cuantitativa de que el trader sistemáticamente entra tarde/peor que el pullback ideal calculado** — motiva directamente las Secciones 15-17.

**Caveat de muestra:** varios segmentos tienen `N_Verificable` entre 1 y 17 (algunos con 0, sin datos en ese tercil de ICD para ese bias) — no tratar los porcentajes de celdas chicas como definitivos.

**④.B — con la muestra ampliada (`order_filled=True` y `False`), el patrón se sostiene:** N sube (ej. Bearish ICD-medio 5→7 casos) y `Entro_Peor_%` se mantiene en el mismo rango (71.4%/62.5% en Bearish, 55.6%/100% en Choppy) — no fue solo un artefacto de muestra chica. **Nota de calidad de datos:** de los 9 registros nuevos candidatos (`order_filled=False` con `entry_price` cargado), 7 tenían `entry_price=0` — un placeholder de la DB (columna `NOT NULL`, XAUUSD nunca vale 0) del flujo "abortar por gates" que nunca llega a pedir `entry_price`, no un target real. Se excluyeron explícitamente (además de los 13 placeholders legacy `entry_price==closing_price`) — de ahí que la muestra ampliada solo sume 2 filas netas, no 9.

## Sección 15: ¿Hay señal de miedo/FOMO al entrar? (`entry_chase_R` vs. estado psicológico)

Objetivo (Goal B): definir `entry_chase_R` — cuánto peor (positivo) o mejor (negativo/cero) fue el `entry_price` real respecto al `structural_entry_price` ideal, normalizado por `one_r` — y cruzarlo contra los campos psicológicos de `tactical_audit` (`anxiety_level`, `impatience_level`, `primary_emotion`) para ver si un estado de ansiedad/impaciencia más alto se asocia con entradas perseguidas (chase).

Restringido a `order_filled == True` (mismo criterio que la Sección 14④) — no tiene sentido medir "cuánto perseguí el precio" en una orden que nunca se llenó.

In [178]:
with engine.connect() as conn:
    psych_df = _load_tactical_slice(conn, ["anxiety_level", "impatience_level", "primary_emotion"])
# anxiety_level/impatience_level son ge=1,le=5 en el schema (cli/schemas/audit_tactical.py) -- 0 es un
# placeholder de DB, nunca un valor real. primary_emotion='nan' (string) es el mismo tipo de artefacto.
psych_df['anxiety_level'] = psych_df['anxiety_level'].replace(0, np.nan)
psych_df['impatience_level'] = psych_df['impatience_level'].replace(0, np.nan)
psych_df['primary_emotion'] = psych_df['primary_emotion'].replace('nan', np.nan)

chase_df = struct_df[
    (struct_df['order_filled'] == True)
    & struct_df['entry_price'].notna() & struct_df['structural_entry_price'].notna() & struct_df['theoretical_dir'].notna()
][['id', 'market_bias', 'theoretical_dir', 'entry_price', 'structural_entry_price', 'one_r']].merge(psych_df, on='id', how='left')

chase_df['entry_chase_R'] = np.where(
    chase_df['theoretical_dir'] == 'Long',
    (chase_df['entry_price'] - chase_df['structural_entry_price']) / chase_df['one_r'],
    (chase_df['structural_entry_price'] - chase_df['entry_price']) / chase_df['one_r'],
)

print(f"N trades ejecutados con datos completos: {len(chase_df)}")
display(chase_df['entry_chase_R'].describe().round(3).to_frame().T)

N trades ejecutados con datos completos: 28


,count,mean,std,min,25%,50%,75%,max
entry_chase_R,28.0000,0.0980,0.3250,-0.7800,-0.1060,0.1420,0.3220,0.6890


### `entry_chase_R` por `anxiety_level`

In [179]:
display(chase_df.groupby('anxiety_level')['entry_chase_R'].agg(['count', 'mean', 'median']).round(3))

,count,mean,median
anxiety_level,,,
1.0000,1,0.0180,0.0180
2.0000,7,0.1140,0.1310
3.0000,10,0.0630,0.1290
4.0000,8,0.0970,0.2030
5.0000,2,0.2620,0.2620


### `entry_chase_R` por `impatience_level`

In [180]:
display(chase_df.groupby('impatience_level')['entry_chase_R'].agg(['count', 'mean', 'median']).round(3))

,count,mean,median
impatience_level,,,
1.0000,10,0.1270,0.1730
2.0000,10,-0.0390,0.0190
3.0000,1,0.2860,0.2860
4.0000,2,0.0880,0.0880
5.0000,5,0.2790,0.2530


### `entry_chase_R` por `primary_emotion`

In [181]:
display(chase_df.groupby('primary_emotion')['entry_chase_R'].agg(['count', 'mean', 'median']).round(3).sort_values('mean', ascending=False))

,count,mean,median
primary_emotion,,,
Impatience,3,0.3730,0.2770
Anxiety,4,0.2660,0.2700
Self-doubt,3,0.1620,0.3460
Equanimity,8,0.1250,0.1490
Fear of being wrong,5,0.0680,-0.0060
Grounded Confidence,3,-0.1090,0.0440
Boredom,1,-0.1600,-0.1600
Shame,1,-0.7800,-0.7800


### Lectura de los resultados

**Por `anxiety_level`:** tendencia laxa pero presente — `entry_chase_R` sube de 0.018 (ansiedad=1, n=1) a 0.262 (ansiedad=5, n=2), con niveles intermedios (2-4) entre 0.06 y 0.11. No es un patrón perfectamente monótono (nivel 3 < nivel 2), pero los extremos apuntan en la dirección esperada: más ansiedad, entrada más perseguida.

**Por `impatience_level`:** menos claro — nivel 2 tiene `entry_chase_R` promedio *negativo* (-0.039, mejor que ideal), pero nivel 5 (máxima impaciencia) vuelve a subir a 0.279. No hay una relación monótona limpia con esta muestra.

**Por `primary_emotion`:** acá aparece la señal más nítida — **`Impatience` (0.373) y `Anxiety` (0.266) tienen el `entry_chase_R` promedio más alto** (entradas más perseguidas), mientras que `Grounded Confidence` (-0.109) es la única emoción con promedio negativo (entradas mejores que el pullback ideal). Esto es consistente con la hipótesis del usuario: los estados de miedo/ansiedad/impaciencia se asocian con perseguir el precio en vez de esperar el pullback estructural.

**Caveat de muestra importante:** todas las categorías tienen entre 1 y 10 casos — esto es una señal exploratoria, no una conclusión estadísticamente robusta. `Shame` (n=1, -0.78) y `Boredom` (n=1, -0.16) en particular son un solo caso cada uno.

## Sección 15.B: `entry_chase_R` sobre TODOS los registros (`order_filled=True` y `False`)

Mismo `entry_chase_R` de la Sección 15, mismo signo direccional, pero calculado sobre `all_entries_df` en vez de solo `order_filled==True` — aumenta la muestra al incluir los intentos donde el trader apuntó a su entrada de precisión y el mercado no se la dio. Esto responde más directamente la pregunta original: ¿con qué frecuencia el trader LOGRA su objetivo de entrar mejor que `structural_entry_price`, y eso se relaciona con su estado psicológico, sobre una muestra más grande?

In [182]:
chase_df_all = all_entries_df[
    all_entries_df['entry_price'].notna() & all_entries_df['structural_entry_price'].notna() & all_entries_df['theoretical_dir'].notna()
][['id', 'market_bias', 'theoretical_dir', 'entry_price', 'structural_entry_price', 'one_r']].merge(psych_df, on='id', how='left')

chase_df_all['entry_chase_R'] = np.where(
    chase_df_all['theoretical_dir'] == 'Long',
    (chase_df_all['entry_price'] - chase_df_all['structural_entry_price']) / chase_df_all['one_r'],
    (chase_df_all['structural_entry_price'] - chase_df_all['entry_price']) / chase_df_all['one_r'],
)

print(f"N (Sección 15 original, solo order_filled=True): {len(chase_df)}")
print(f"N (Sección 15.B, todos los registros): {len(chase_df_all)}")
display(chase_df_all['entry_chase_R'].describe().round(3).to_frame().T)

N (Sección 15 original, solo order_filled=True): 28
N (Sección 15.B, todos los registros): 30


,count,mean,std,min,25%,50%,75%,max
entry_chase_R,30.0000,0.1150,0.3460,-0.7800,-0.1140,0.1420,0.3380,0.8500


### `entry_chase_R` (todos los registros) por `anxiety_level`

In [183]:
display(chase_df_all.groupby('anxiety_level')['entry_chase_R'].agg(['count', 'mean', 'median']).round(3))

,count,mean,median
anxiety_level,,,
1.0000,1,0.0180,0.0180
2.0000,7,0.1140,0.1310
3.0000,10,0.0630,0.1290
4.0000,9,0.1810,0.2530
5.0000,2,0.2620,0.2620


### `entry_chase_R` (todos los registros) por `impatience_level`

In [184]:
display(chase_df_all.groupby('impatience_level')['entry_chase_R'].agg(['count', 'mean', 'median']).round(3))

,count,mean,median
impatience_level,,,
1.0000,11,0.1930,0.2150
2.0000,10,-0.0390,0.0190
3.0000,1,0.2860,0.2860
4.0000,2,0.0880,0.0880
5.0000,5,0.2790,0.2530


### `entry_chase_R` (todos los registros) por `primary_emotion`

In [185]:
display(chase_df_all.groupby('primary_emotion')['entry_chase_R'].agg(['count', 'mean', 'median']).round(3).sort_values('mean', ascending=False))

,count,mean,median
primary_emotion,,,
Impatience,3,0.3730,0.2770
Anxiety,4,0.2660,0.2700
Fear of being wrong,6,0.1980,0.1050
Self-doubt,3,0.1620,0.3460
Equanimity,8,0.1250,0.1490
Grounded Confidence,3,-0.1090,0.0440
Boredom,1,-0.1600,-0.1600
Shame,1,-0.7800,-0.7800


### Lectura de los resultados

**El patrón de la Sección 15 se sostiene y se afina con la muestra ampliada (N=28→30):** `impatience_level=1` ahora tiene 11 casos (antes 10) con `entry_chase_R` promedio 0.193 (subió de 0.127), y `anxiety_level=4` pasa de 8 a 9 casos con promedio 0.181 (subió de 0.097) — los 2 registros nuevos genuinos empujan el promedio hacia arriba, no lo diluyen. `primary_emotion` mantiene exactamente el mismo ranking que en la Sección 15 (`Impatience`/`Anxiety` con el `entry_chase_R` más alto, `Grounded Confidence` el único negativo) — ninguno de los 2 registros nuevos tenía `primary_emotion` cargado, así que esa tabla no cambió.

**Nota de calidad de datos (importante):** de los 9 registros `order_filled=False` con algún dato de `entry_price`/psicología, **7 eran placeholders de DB** — `entry_price=0`, `anxiety_level=0`, `impatience_level=0` y `primary_emotion='nan'` (string, no nulo real) simultáneamente, todos del flujo "abortar por gates" (nunca se pide esta información ahí). El primer intento de esta sección los incluyó sin filtrar y produjo un `entry_chase_R` promedio de **-101 a -151** (matemáticamente correcto dado `entry_price=0`, pero sin sentido — no es una "entrada excelente", es una división por un precio inexistente). Se corrigió excluyendo `entry_price==0` de `all_entries_df` y convirtiendo `anxiety_level`/`impatience_level==0` y `primary_emotion=='nan'` a nulo real (`anxiety_level`/`impatience_level` son `ge=1,le=5` en el schema — `cli/schemas/audit_tactical.py:210-212` — 0 nunca es un valor válido) antes de generar esta tabla.

## Sección 16: Entrada hipotética a X% de distancia — barrido y tasa de alcance (dos versiones)

Objetivo (Goal C): simular una entrada límite colocada a `x`% de la distancia entre dos precios, y medir con qué frecuencia `structural_mae` la habría alcanzado — por `market_bias × edge_mag_tercile`. Dos versiones, confirmadas por el usuario:

- **`mark_entry`**: interpola entre `mark_price` (x=0) y `structural_entry_price` (x=1) — los dos niveles disponibles *antes* de saber si el trade se va a llenar. Usa toda la muestra con `mark_price`/`structural_entry_price` cargados.
- **`fractal_entry`**: interpola entre `entry_price` real (x=0) y `structural_entry_price` (x=1) — solo calculable para trades con `order_filled == True` (mismo filtro que Secciones 14④/15).

En ambas, `x=1.0` converge exactamente a `structural_entry_price` — debe coincidir con el `Reached_%` de la Sección 14② (chequeo de consistencia).

In [186]:
X_VALUES = [round(x, 1) for x in np.arange(0.0, 1.01, 0.1)]

def hyp_entry_price(df, version, x):
    if version == 'mark_entry':
        return df['mark_price'] + x * (df['structural_entry_price'] - df['mark_price'])
    if version == 'fractal_entry':
        return df['entry_price'] + x * (df['structural_entry_price'] - df['entry_price'])
    raise ValueError(version)

def reached_hypothetical(theoretical_dir, entry_hyp, mae):
    return np.where(theoretical_dir == 'Short', mae >= entry_hyp, mae <= entry_hyp)

def sweep_reach_rate(df, version, x_values):
    # El filtro order_filled==True (cuando aplica) lo decide el caller, no esta funcion
    # -- asi se puede pasar struct_df (solo ejecutados) o all_entries_df (todos los targets).
    base = df.copy()
    anchor_col = 'mark_price' if version == 'mark_entry' else 'entry_price'
    base = base[
        base['theoretical_dir'].notna() & base['structural_mae'].notna()
        & base['structural_entry_price'].notna() & base[anchor_col].notna()
    ].copy()

    rows = []
    for x in x_values:
        entry_hyp = hyp_entry_price(base, version, x)
        reached = reached_hypothetical(base['theoretical_dir'], entry_hyp, base['structural_mae'])
        tmp = base[['market_bias', 'edge_mag_tercile']].copy()
        tmp['x'] = x
        tmp['reached'] = reached
        rows.append(tmp)
    long_df = pd.concat(rows, ignore_index=True)
    out = long_df.groupby(['x', 'market_bias', 'edge_mag_tercile'], observed=False)['reached'].agg(N_Verificable='count', Reached='mean').reset_index()
    out['Reached_%'] = (out['Reached'] * 100).round(1)
    return out[['x', 'market_bias', 'edge_mag_tercile', 'N_Verificable', 'Reached_%']]

### Barrido v1 — `mark_entry_x` (interpola `mark_price` ↔ `structural_entry_price`)

In [187]:
sweep_v1 = sweep_reach_rate(struct_df, 'mark_entry', X_VALUES)
print(f"Filas: {len(sweep_v1)}")
display(sweep_v1)

Filas: 99


,x,market_bias,edge_mag_tercile,N_Verificable,Reached_%
0,0.0000,Bearish,"(-0.001, 0.2]",0,NaN
1,0.0000,Bearish,"(0.2, 0.375]",10,90.0000
2,0.0000,Bearish,"(0.375, 0.825]",15,100.0000
3,0.0000,Bullish,"(-0.001, 0.2]",0,NaN
4,0.0000,Bullish,"(0.2, 0.375]",5,80.0000
5,0.0000,Bullish,"(0.375, 0.825]",8,100.0000
6,0.0000,Choppy / Neutral,"(-0.001, 0.2]",19,89.5000
7,0.0000,Choppy / Neutral,"(0.2, 0.375]",7,71.4000
8,0.0000,Choppy / Neutral,"(0.375, 0.825]",0,NaN
9,0.1000,Bearish,"(-0.001, 0.2]",0,NaN


### Barrido v2 — `fractal_entry_x` (interpola `entry_price` real ↔ `structural_entry_price`)

In [188]:
sweep_v2 = sweep_reach_rate(struct_df[struct_df['order_filled'] == True], 'fractal_entry', X_VALUES)
print(f"Filas: {len(sweep_v2)}")
display(sweep_v2)

Filas: 99


,x,market_bias,edge_mag_tercile,N_Verificable,Reached_%
0,0.0000,Bearish,"(-0.001, 0.2]",0,NaN
1,0.0000,Bearish,"(0.2, 0.375]",5,100.0000
2,0.0000,Bearish,"(0.375, 0.825]",8,100.0000
3,0.0000,Bullish,"(-0.001, 0.2]",0,NaN
4,0.0000,Bullish,"(0.2, 0.375]",1,100.0000
5,0.0000,Bullish,"(0.375, 0.825]",3,100.0000
6,0.0000,Choppy / Neutral,"(-0.001, 0.2]",9,100.0000
7,0.0000,Choppy / Neutral,"(0.2, 0.375]",2,100.0000
8,0.0000,Choppy / Neutral,"(0.375, 0.825]",0,NaN
9,0.1000,Bearish,"(-0.001, 0.2]",0,NaN


### Lectura de los resultados

**Chequeo de consistencia OK:** en `x=1.0` ambas versiones convergen exactamente a los números de la Sección 14② (ej. Bearish ICD-alto 60.0%, Choppy ICD-bajo 68.8%) — confirma que la lógica direccional del barrido es correcta.

**v1 (`mark_entry`):** las tasas de alcance son bastante estables entre `x=0` y `x≈0.4` en la mayoría de los segmentos (ej. Bearish ICD-alto se mantiene en 100% hasta `x=0.4`, recién cae a 93.3%/80% en `x=0.5-0.6`) — sugiere que, para Bearish con edge fuerte, colocar la entrada más cerca del pullback completo (`x` alto) no reduce mucho la probabilidad de que se llene, hasta cierto punto. Bullish ICD-medio es más sensible (80%→40-60% ya desde `x=0.2`).

**v2 (`fractal_entry`):** el patrón es más parecido a un escalón porque `entry_price` real (el ancla en `x=0`) ya está más cerca de `structural_entry_price` que `mark_price` en promedio (consistente con la Sección 14④) — la muestra es más chica (solo `order_filled=True`).

**Caveat de muestra:** varios segmentos (`Bearish`/`Bullish` en el tercil de ICD más bajo, `Choppy` en el más alto) tienen `N_Verificable=0` en toda la corrida — no hay datos suficientes en esas combinaciones bias×ICD para sacar conclusiones.

## Sección 16.B: barrido `fractal_entry_x` sobre TODOS los registros (`order_filled=True` y `False`)

Mismo barrido y misma lógica de alcance de la Sección 16 (v2, `fractal_entry`), pero sobre `all_entries_df` — no hace falta repetir `mark_entry` (v1) porque esa ya usaba toda la muestra (no depende de `entry_price`/`order_filled`).

In [189]:
sweep_v2_all = sweep_reach_rate(all_entries_df, 'fractal_entry', X_VALUES)
print(f"Filas: {len(sweep_v2_all)}")
display(sweep_v2_all)

Filas: 99


,x,market_bias,edge_mag_tercile,N_Verificable,Reached_%
0,0.0000,Bearish,"(-0.001, 0.2]",0,NaN
1,0.0000,Bearish,"(0.2, 0.375]",7,85.7000
2,0.0000,Bearish,"(0.375, 0.825]",8,100.0000
3,0.0000,Bullish,"(-0.001, 0.2]",0,NaN
4,0.0000,Bullish,"(0.2, 0.375]",1,100.0000
5,0.0000,Bullish,"(0.375, 0.825]",3,100.0000
6,0.0000,Choppy / Neutral,"(-0.001, 0.2]",9,100.0000
7,0.0000,Choppy / Neutral,"(0.2, 0.375]",2,100.0000
8,0.0000,Choppy / Neutral,"(0.375, 0.825]",0,NaN
9,0.1000,Bearish,"(-0.001, 0.2]",0,NaN


### Lectura de los resultados

**A `x=0` (ancla en `entry_price` real), la tasa de alcance ya es alta o total en casi todos los segmentos** (85.7-100%) — consistente con la Sección 14④.B: el trader suele fijar su entrada de precisión relativamente cerca de donde el precio efectivamente se movió (o directamente en un punto que el propio `structural_mae` ya cubre). Bearish ICD-medio es la excepción, estable en 85.7% en vez de 100% — el único segmento donde ni siquiera el propio target del trader se alcanza siempre.

**El barrido es más plano que en la versión v2 original (Sección 16)** porque el ancla `x=0` ya está, en promedio, más cerca de `structural_entry_price` que `mark_price` — hay menos "distancia por recorrer" en el barrido completo.

**Caveat de muestra:** con la corrección de placeholders (ver Sección 15.B), la muestra ampliada solo suma 2 filas netas sobre la Sección 16 original (Bearish ICD-medio: 5→7) — el resto de segmentos no ganó datos nuevos, así que estas curvas son casi idénticas a las de la fractal_entry original en la mayoría de los bias/ICD.

## Sección 17: Backtest de resultado hipotético — ratios 1:2 y 1:2.2 (Goal D)

Objetivo: para cada entrada hipotética activada (Sección 16), simular un trade con `TP = edge_validation_price` y `SL` calculado según el ratio (`risk = |TP - entry| / ratio`), y clasificar el resultado.

**Regla de clasificación, confirmada explícitamente por el usuario** (simplificación conocida: no hay velas con orden temporal real, solo extremos MAE/MFE de un único punto — ver advertencia igual a la de `jupyter/p2_order_filled_reached_entry.md`):
- Si `structural_mae` rompe el `SL` hipotético → **Perdido**.
- Si no, y `structural_mfe` alcanza el `TP` → **Ganado**.
- Si no se cumple ninguna → **Inconcluso**.
- Si se cumplen ambas (posible por falta de orden temporal) → prevalece **Perdido** (más conservador — se documenta como simplificación explícita, no es un veredicto definitivo).

In [190]:
RATIOS = [2.0, 2.2]

def classify_hypothetical_trade(df, version, x, ratio):
    # Igual que sweep_reach_rate: el filtro order_filled lo decide el caller.
    base = df.copy()
    anchor_col = 'mark_price' if version == 'mark_entry' else 'entry_price'
    base = base[
        base['theoretical_dir'].notna() & base['structural_mae'].notna() & base['structural_mfe'].notna()
        & base['structural_entry_price'].notna() & base['edge_validation_price'].notna() & base[anchor_col].notna()
    ].copy()

    entry_hyp = hyp_entry_price(base, version, x)
    reached = reached_hypothetical(base['theoretical_dir'], entry_hyp, base['structural_mae'])

    tp = base['edge_validation_price']
    risk_dist = (tp - entry_hyp).abs() / ratio
    sl = np.where(base['theoretical_dir'] == 'Long', entry_hyp - risk_dist, entry_hyp + risk_dist)

    sl_hit = np.where(base['theoretical_dir'] == 'Long', base['structural_mae'] <= sl, base['structural_mae'] >= sl)
    tp_hit = np.where(base['theoretical_dir'] == 'Long', base['structural_mfe'] >= tp, base['structural_mfe'] <= tp)
    outcome = np.select([sl_hit, tp_hit], ['Perdido', 'Ganado'], default='Inconcluso')

    out = base[['id', 'market_bias', 'edge_mag_tercile']].copy()
    out['version'] = version
    out['x'] = x
    out['ratio'] = ratio
    out['reached'] = reached
    out['outcome'] = np.where(reached, outcome, 'No_Activado')
    return out

detail_rows = []
for _version in ('mark_entry', 'fractal_entry'):
    _base_df = struct_df if _version == 'mark_entry' else struct_df[struct_df['order_filled'] == True]
    for _x in X_VALUES:
        for _ratio in RATIOS:
            detail_rows.append(classify_hypothetical_trade(_base_df, _version, _x, _ratio))
backtest_detail = pd.concat(detail_rows, ignore_index=True)
print(f"Filas en backtest_detail (todas las combinaciones version x x x ratio x análisis): {len(backtest_detail)}")

Filas en backtest_detail (todas las combinaciones version x x x ratio x análisis): 2024


### Tabla larga — resultado por `version × x × ratio × market_bias × edge_mag_tercile`

In [191]:
activated = backtest_detail[backtest_detail['reached']]
backtest_summary = (
    activated.groupby(['version', 'x', 'ratio', 'market_bias', 'edge_mag_tercile'], observed=False)['outcome']
    .value_counts().unstack(fill_value=0).reset_index()
)
for _col in ('Ganado', 'Perdido', 'Inconcluso'):
    if _col not in backtest_summary.columns:
        backtest_summary[_col] = 0

backtest_summary['N_Activado'] = backtest_summary['Ganado'] + backtest_summary['Perdido'] + backtest_summary['Inconcluso']
backtest_summary['Win_Rate_%'] = np.where(
    (backtest_summary['Ganado'] + backtest_summary['Perdido']) > 0,
    (backtest_summary['Ganado'] / (backtest_summary['Ganado'] + backtest_summary['Perdido']) * 100).round(1),
    np.nan,
)
backtest_summary = backtest_summary.rename(columns={'Ganado': 'N_Ganado', 'Perdido': 'N_Perdido', 'Inconcluso': 'N_Inconcluso'})
backtest_summary = backtest_summary[['version', 'x', 'ratio', 'market_bias', 'edge_mag_tercile', 'N_Activado', 'N_Ganado', 'N_Perdido', 'N_Inconcluso', 'Win_Rate_%']]

print(f"Filas en backtest_summary: {len(backtest_summary)}")
display(backtest_summary.head(20))

Filas en backtest_summary: 396


outcome,version,x,ratio,market_bias,edge_mag_tercile,N_Activado,N_Ganado,N_Perdido,N_Inconcluso,Win_Rate_%
0,fractal_entry,0.0000,2.0000,Bearish,"(-0.001, 0.2]",0,0,0,0,NaN
1,fractal_entry,0.0000,2.0000,Bearish,"(0.2, 0.375]",5,2,3,0,40.0000
2,fractal_entry,0.0000,2.0000,Bearish,"(0.375, 0.825]",8,5,3,0,62.5000
3,fractal_entry,0.0000,2.0000,Bullish,"(-0.001, 0.2]",0,0,0,0,NaN
4,fractal_entry,0.0000,2.0000,Bullish,"(0.2, 0.375]",1,0,1,0,0.0000
5,fractal_entry,0.0000,2.0000,Bullish,"(0.375, 0.825]",3,0,3,0,0.0000
6,fractal_entry,0.0000,2.0000,Choppy / Neutral,"(-0.001, 0.2]",9,2,4,3,33.3000
7,fractal_entry,0.0000,2.0000,Choppy / Neutral,"(0.2, 0.375]",2,1,0,1,100.0000
8,fractal_entry,0.0000,2.0000,Choppy / Neutral,"(0.375, 0.825]",0,0,0,0,NaN
9,fractal_entry,0.0000,2.2000,Bearish,"(-0.001, 0.2]",0,0,0,0,NaN


### Tabla resumen accionable — el `x%` con mejor Win Rate por segmento

"Entrada óptima aproximada" (no perfecta, como pidió el usuario): para cada `version × ratio × market_bias × edge_mag_tercile`, el `x` con mayor `Win_Rate_%` entre los que tengan `N_Activado >= 5` (umbral mínimo de muestra — documentado, no es un número mágico).

In [192]:
MIN_N = 5
qualified = backtest_summary[backtest_summary['N_Activado'] >= MIN_N].dropna(subset=['Win_Rate_%'])
if len(qualified) == 0:
    print("Ningún segmento alcanzó N_Activado >= 5 — revisar MIN_N o los datos.")
else:
    best_x_idx = qualified.groupby(['version', 'ratio', 'market_bias', 'edge_mag_tercile'], observed=True)['Win_Rate_%'].idxmax()
    best_x_summary = qualified.loc[best_x_idx].sort_values(['version', 'ratio', 'market_bias', 'edge_mag_tercile']).reset_index(drop=True)
    display(best_x_summary)

outcome,version,x,ratio,market_bias,edge_mag_tercile,N_Activado,N_Ganado,N_Perdido,N_Inconcluso,Win_Rate_%
0,fractal_entry,0.0000,2.0000,Bearish,"(0.2, 0.375]",5,2,3,0,40.0000
1,fractal_entry,0.1000,2.0000,Bearish,"(0.375, 0.825]",8,6,2,0,75.0000
2,fractal_entry,0.0000,2.0000,Choppy / Neutral,"(-0.001, 0.2]",9,2,4,3,33.3000
3,fractal_entry,0.0000,2.2000,Bearish,"(0.2, 0.375]",5,2,3,0,40.0000
4,fractal_entry,0.2000,2.2000,Bearish,"(0.375, 0.825]",8,6,2,0,75.0000
5,fractal_entry,0.0000,2.2000,Choppy / Neutral,"(-0.001, 0.2]",9,2,4,3,33.3000
6,mark_entry,0.9000,2.0000,Bearish,"(0.2, 0.375]",8,4,4,0,50.0000
7,mark_entry,0.8000,2.0000,Bearish,"(0.375, 0.825]",10,8,2,0,80.0000
8,mark_entry,0.0000,2.0000,Bullish,"(0.375, 0.825]",8,4,3,1,57.1000
9,mark_entry,0.6000,2.0000,Choppy / Neutral,"(-0.001, 0.2]",17,5,8,4,38.5000


### Lectura de los resultados y advertencia metodológica

**Hallazgo principal — Bearish con ICD alto es, con diferencia, el segmento con mejor resultado hipotético:** hasta **80% de Win Rate** (`mark_entry`, `x=0.8`, ratio 1:2, n=10) y **77.8%** (`x=0.9`, ratio 1:2.2, n=9) — consistente con todos los hallazgos previos del notebook (Sección 2: Bearish tiene el mejor win rate estructural real, 76.0%; Sección 12: Bearish tiene el `mae_R` mediano más bajo, movimientos más limpios). La versión `fractal_entry` para el mismo segmento da resultados similares (75% en `x=0.1-0.2`, n=8), reforzando la señal con una fuente de datos distinta.

**Choppy y Bullish son mucho más débiles y mixtos:** 33-50% de Win Rate en casi todos los `x`/ratio calificados — no hay una entrada hipotética que rescate estos segmentos a niveles comparables con Bearish.

**Lectura para "entrada óptima aproximada" (lo que pidió el usuario):** para Bearish con edge fuerte, un `x` entre 0.8-0.9 (es decir, esperar un pullback cercano al 80-90% de la distancia hacia `structural_entry_price`, no una entrada a mercado inmediata) da el mejor resultado hipotético en esta muestra. Para Choppy/Bullish, ningún `x` probado destaca lo suficiente como para recomendar una regla — la data no sostiene una entrada límite optimizada ahí todavía.

**Advertencia metodológica obligatoria (igual que en `jupyter/p2_order_filled_reached_entry.md`):** esta clasificación usa `structural_mae`/`structural_mfe` — extremos de un único punto por análisis, sin orden temporal confirmado entre sí. La regla "SL tiene prioridad si ambas condiciones se cumplen" es conservadora pero **no es un veredicto definitivo** — es un proxy para priorizar dónde investigar más, no una garantía de que ese Win Rate se replique en trading real. Confirmarlo con certeza requeriría datos de velas con orden temporal real, que este sistema no tiene.

**Caveat de muestra:** los `N_Activado` en la tabla resumen van de 5 a 14 — chico para conclusiones firmes, en particular fuera del segmento Bearish/ICD-alto que es el que tiene más datos y la señal más consistente.

## Sección 17.B: backtest hipotético sobre TODOS los registros (`order_filled=True` y `False`)

Misma regla de clasificación de la Sección 17 (SL rompe → Perdido; si no y MFE llega a TP → Ganado; si no, Inconcluso; SL prevalece si ambas se cumplen), corrida solo para `fractal_entry` sobre `all_entries_df` (mayor N que la Sección 17 original). No hace falta repetir `mark_entry` — ya cubría toda la muestra.

In [193]:
detail_rows_all = []
for _x in X_VALUES:
    for _ratio in RATIOS:
        detail_rows_all.append(classify_hypothetical_trade(all_entries_df, 'fractal_entry', _x, _ratio))
backtest_detail_all = pd.concat(detail_rows_all, ignore_index=True)
print(f"Filas en backtest_detail_all: {len(backtest_detail_all)}")

Filas en backtest_detail_all: 660


### Tabla larga (todos los registros) — resultado por `x × ratio × market_bias × edge_mag_tercile`

In [194]:
activated_all = backtest_detail_all[backtest_detail_all['reached']]
backtest_summary_all = (
    activated_all.groupby(['x', 'ratio', 'market_bias', 'edge_mag_tercile'], observed=False)['outcome']
    .value_counts().unstack(fill_value=0).reset_index()
)
for _col in ('Ganado', 'Perdido', 'Inconcluso'):
    if _col not in backtest_summary_all.columns:
        backtest_summary_all[_col] = 0

backtest_summary_all['N_Activado'] = backtest_summary_all['Ganado'] + backtest_summary_all['Perdido'] + backtest_summary_all['Inconcluso']
backtest_summary_all['Win_Rate_%'] = np.where(
    (backtest_summary_all['Ganado'] + backtest_summary_all['Perdido']) > 0,
    (backtest_summary_all['Ganado'] / (backtest_summary_all['Ganado'] + backtest_summary_all['Perdido']) * 100).round(1),
    np.nan,
)
backtest_summary_all = backtest_summary_all.rename(columns={'Ganado': 'N_Ganado', 'Perdido': 'N_Perdido', 'Inconcluso': 'N_Inconcluso'})
backtest_summary_all = backtest_summary_all[['x', 'ratio', 'market_bias', 'edge_mag_tercile', 'N_Activado', 'N_Ganado', 'N_Perdido', 'N_Inconcluso', 'Win_Rate_%']]

print(f"Filas en backtest_summary_all: {len(backtest_summary_all)}")
display(backtest_summary_all.head(100))

Filas en backtest_summary_all: 198


outcome,x,ratio,market_bias,edge_mag_tercile,N_Activado,N_Ganado,N_Perdido,N_Inconcluso,Win_Rate_%
0,0.0000,2.0000,Bearish,"(-0.001, 0.2]",0,0,0,0,NaN
1,0.0000,2.0000,Bearish,"(0.2, 0.375]",6,3,3,0,50.0000
2,0.0000,2.0000,Bearish,"(0.375, 0.825]",8,5,3,0,62.5000
3,0.0000,2.0000,Bullish,"(-0.001, 0.2]",0,0,0,0,NaN
4,0.0000,2.0000,Bullish,"(0.2, 0.375]",1,0,1,0,0.0000
5,0.0000,2.0000,Bullish,"(0.375, 0.825]",3,0,3,0,0.0000
6,0.0000,2.0000,Choppy / Neutral,"(-0.001, 0.2]",9,2,4,3,33.3000
7,0.0000,2.0000,Choppy / Neutral,"(0.2, 0.375]",2,1,0,1,100.0000
8,0.0000,2.0000,Choppy / Neutral,"(0.375, 0.825]",0,0,0,0,NaN
9,0.0000,2.2000,Bearish,"(-0.001, 0.2]",0,0,0,0,NaN


### Tabla resumen accionable (todos los registros) — el `x%` con mejor Win Rate por segmento

In [195]:
qualified_all = backtest_summary_all[backtest_summary_all['N_Activado'] >= MIN_N].dropna(subset=['Win_Rate_%'])
if len(qualified_all) == 0:
    print("Ningún segmento alcanzó N_Activado >= 5 — revisar MIN_N o los datos.")
else:
    best_x_idx_all = qualified_all.groupby(['ratio', 'market_bias', 'edge_mag_tercile'], observed=True)['Win_Rate_%'].idxmax()
    best_x_summary_all = qualified_all.loc[best_x_idx_all].sort_values(['ratio', 'market_bias', 'edge_mag_tercile']).reset_index(drop=True)
    display(best_x_summary_all)

outcome,x,ratio,market_bias,edge_mag_tercile,N_Activado,N_Ganado,N_Perdido,N_Inconcluso,Win_Rate_%
0,0.0000,2.0000,Bearish,"(0.2, 0.375]",6,3,3,0,50.0000
1,0.1000,2.0000,Bearish,"(0.375, 0.825]",8,6,2,0,75.0000
2,0.0000,2.0000,Choppy / Neutral,"(-0.001, 0.2]",9,2,4,3,33.3000
3,0.1000,2.2000,Bearish,"(0.2, 0.375]",6,3,3,0,50.0000
4,0.2000,2.2000,Bearish,"(0.375, 0.825]",8,6,2,0,75.0000
5,0.0000,2.2000,Choppy / Neutral,"(-0.001, 0.2]",9,2,4,3,33.3000


### Lectura de los resultados

**El hallazgo principal de la Sección 17 se confirma con la muestra ampliada:** Bearish ICD-alto sigue siendo el segmento más fuerte — **75% de Win Rate** (`x=0.1-0.2`, ambos ratios, n=8) — prácticamente idéntico al 75-80% de la Sección 17 original. Con solo 2 registros netos nuevos (ambos en Bearish), el resultado no cambia de forma sustancial, lo cual en sí es una buena señal: la conclusión de la Sección 17 no dependía de un artefacto de muestra chica en ese segmento puntual.

**Choppy ICD-bajo se mantiene débil** (33.3%, n=9) y **Bullish sigue sin alcanzar `N_Activado>=5` en ningún `x`/ratio** — ninguno de los 2 registros nuevos era Bullish, así que esa laguna de datos sigue abierta.

**Misma advertencia metodológica que la Sección 17** (proxy sin orden temporal confirmado entre `structural_mae`/`structural_mfe`, SL prevalece si ambas condiciones se cumplen — no es un veredicto definitivo).

## Sección 18: Reporte para LLM (`core/llm_report_export.py`)

Igual que la sección homóloga de `jupyter/data_analysis.ipynb`, esto genera un
segundo artefacto — Markdown, sin imágenes, sin `Styler`/color — pensado para
copiar/pegar o subir a una conversación de LLM y pedirle análisis sobre el
edge estructural (no repite el reporte HTML/JSON de `data_analysis.ipynb`,
que cubre KPIs de ejecución táctica y backtesting; este es específico del
alcance de `custom_analysis.ipynb`: `unified_department`/`analysis_layer`/
`efficiency_audit`). A diferencia de las secciones anteriores de este
notebook, **no incluye "Lectura de los resultados"** — solo contexto de
dominio, memoria de cálculo (código fuente real de cada función) y las
tablas de resultados; la interpretación queda para quien lo lea después
(humano o LLM), sin sesgar con las lecturas ya escritas arriba.

Cada tabla se recalcula desde las tablas base (`unified_df`/`efficiency_df`/
`layer_df`/`tactical_audit`) con funciones propias, autocontenidas — no
reutiliza las variables intermedias de las Secciones 2-12 de arriba, así el
código de esta sección (y por lo tanto el reporte que genera) es reproducible
de forma aislada.

In [196]:
import re
from core.report_formatting import dict_to_metrics_frame
from core.llm_report_export import (
    MarkdownSection, build_trade_records, render_llm_markdown_report, save_markdown_report,
)
from core.edge_analysis import analisis_edge, analizar_correlaciones, profit_factor_by_edge_bins, get_market_outcome

In [197]:
llm_context = """
Este documento acompaña a `jupyter/custom_analysis.ipynb`, un notebook de
validación del EDGE ESTRUCTURAL del sistema de trading `blast_master`
(cuenta `flight_account_001_xauusd`, XAUUSD) — distinto en alcance de
`data_analysis_llm_report.md` (ese cubre KPIs de ejecución táctica y
backtesting sobre `tactical_audit`; este cubre `unified_department`,
`analysis_layer` y `efficiency_audit`: ¿el modelo de bias/probabilidad
predice bien la dirección del mercado, y qué tan lejos se mueve el precio
respecto a los niveles que el propio sistema calcula?).

TERMINOLOGÍA CRÍTICA (no confundir campos con nombres parecidos):
- `calc_edge` (alias `i_cd`, Índice Compuesto Direccional) es la suma
  ponderada de los 5 scores P0-P4 (pesos P0=0.30, P1=0.25, P2=0.15,
  P3=0.10, P4=0.20 — `core/math_engine.py::calculate_edge_score`). Su
  signo es la dirección teórica del modelo; su magnitud, la confianza.
- `theoretical_dir` = signo de `calc_edge` (Long si >=0, Short si <0) —
  cubre los 3 regímenes, incluido Choppy. `direction_from_bias` (deriva
  de `market_bias`: Long si Bullish, Short si Bearish, None si Choppy)
  es la convención que usa el código de producción para su propio chequeo
  de "¿alcanzó la entrada?" — este documento usa `theoretical_dir` en
  todas las tablas para poder comparar los 3 regímenes en pie de igualdad;
  no son intercambiables.
- `market_bias` (Bullish/Bearish/Choppy — `unified_department`) es la
  clasificación de régimen de mercado al momento del análisis. `bias_a`
  (`efficiency_audit`) es un campo categórico DISTINTO: el tipo de setup
  estructural predicho (ej. BOS, CHOCH, Trend Reversal, No_Bias(Choppy)).
  `real_bias_b` es el tipo de setup que realmente ocurrió, en la misma
  taxonomía que `bias_a`. `specific_bias_compliance` (Valid/Invalid) es
  si `bias_a` coincidió con la resolución real. `false_regime_rate` es
  una matriz de confusión de 5 clases (True Positive/False Positive/True
  Negative/False Negative/False Signal) calculada por
  `cli/schemas/audit_efficiency.py:60-79` comparando `bias_a` vs
  `real_bias_b`, tratando `No_Bias(Choppy)` como clase negativa (no
  operar) y cualquier otro `bias_a` como clase positiva.
- `Market_Outcome` (+1/-1/0, `core/edge_analysis.py::get_market_outcome`)
  es la dirección real del mercado usada para evaluar cada score P0-P4
  individualmente — es 0 siempre que `market_bias` no sea Bullish/Bearish
  (Choppy no tiene una dirección real definida en esta función), lo cual
  es DISTINTO de `specific_bias_compliance`.
- `structural_entry_price` NO es una columna de la DB — es un precio
  calculado, `E = (TP + RR·SL) / (1+RR)` con `TP=edge_validation_price`,
  `SL=structural_invalidation`, `RR=STRUCTURAL_TARGET_RR` (2.2 en el
  `.env` real) — `core/math_engine.py::calculate_structural_entry`. Es el
  precio de un pullback teórico que, si se llena ahí, da exactamente ese
  R:R hacia `edge_validation_price`. `tactical_audit.entry_price` (el
  precio real cargado en el tactical audit) SÍ está incluido en este
  documento desde la Sección 14 en adelante (merge agregado a
  `_build_struct_df`) — son campos distintos que ahora conviven en el
  mismo DataFrame para poder compararlos. Ojo: `entry_price` puede estar
  poblado con `order_filled=False` (intento no llenado, o placeholder
  legacy) — no usar `entry_price.notna()` como proxy de "se ejecutó",
  filtrar siempre por `order_filled == True` explícito.
- `mae_R`/`mfe_R`: distancia adversa/favorable del precio MÁS ALLÁ de
  `structural_entry_price` (no desde el precio de entrada real, no desde
  `edge_validation_price`), normalizada dividiendo por
  `one_r = |structural_entry_price - structural_invalidation|`. Sirven
  para comparar movimiento entre setups y fechas — los precios crudos de
  `structural_mae`/`structural_mfe` NO son comparables entre sí porque
  XAUUSD se movió de ~4000 a ~4600 en el período cubierto por estos datos.
- `structural_mae`/`structural_mfe` (`efficiency_audit`) son precios
  crudos ingresados MANUALMENTE en el wizard de resolución del audit (no
  hay feed de precio en el sistema) — pueden faltar o tener error de
  precisión de entrada.
- `order_filled` (boolean, `tactical_audit`) es la única señal de
  ejecución real — reemplaza a los antiguos `compliance`/`trade_status`
  (retirados 2026-08-12). `r_multiple` es el R-multiple REALIZADO; puede
  existir incluso con `order_filled=False` (simulación de "qué hubiera
  pasado"), así que no asumir ejecución real solo porque `r_multiple` no
  es nulo — cruzar siempre con `order_filled`.

ADVERTENCIAS METODOLÓGICAS (para no mal-leer las tablas de abajo):
- `profit_factor_by_edge_bins` (código de producción, `core/edge_analysis.py`,
  sin modificar) asigna `bias_sign=0` y `Market_Outcome=0` a las filas
  Choppy — `0==0` se cuenta como acierto. El bin de menor `|calc_edge|`
  suele estar dominado por filas Choppy; su Win Rate no es comparable
  directamente contra los bins con Bullish/Bearish reales.
- En el histórico de esta cuenta, `calc_edge` (magnitud) y `primary_prob`
  (la probabilidad del lado que el modelo predijo — `long_prob` si
  `theoretical_dir=='Long'`, si no `short_prob`) han estado fuertemente
  correlacionados; no tratar sus tablas como dos fuentes de evidencia
  independientes sin volver a verificarlo.
- Este es un journal en vivo: los datos reflejan el estado de la DB al
  momento en que se generó este documento (ver fecha de generación al
  inicio). Los conteos de setups resueltos (`specific_bias_compliance`
  no nulo) cambian con el tiempo a medida que se cierran más análisis —
  no asumir que un conteo citado en una conversación anterior sigue
  vigente sin volver a correr el notebook.
- La muestra total es de decenas de análisis (ver conteos exactos en cada
  tabla) — chica para conclusiones estadísticamente robustas, en
  particular en los cruces de 2-3 variables donde algunas celdas tienen
  n<5.

QUÉ SE ESPERA DE UN ANÁLISIS SOBRE ESTOS DATOS: usar las tablas y el
apéndice de datos crudos para evaluar (1) si el edge estructural
(`calc_edge`/P0-P4) predice bien la dirección real, y en qué condiciones
(bias, magnitud, probabilidad) es más/menos confiable; (2) qué tan lejos
se mueve el precio respecto a los niveles estructurales calculados
(`structural_entry_price`, `mae_R`/`mfe_R`) y si eso sugiere un filtro de
entrada; (3) si hay señal explotable en el texto de la thesis de P0
(temporalidades) o en el consenso entre P0-P4 que el modelo actual no
esté aprovechando. No repetir los agregados ya calculados abajo — cruzar
columnas para encontrar patrones nuevos.
""".strip()

In [198]:
def _build_audit_df(unified_df: pd.DataFrame, efficiency_df: pd.DataFrame) -> pd.DataFrame:
    """Base de Sección 2: efficiency_audit + market_bias/probabilidades de unified_department,
    restringida a setups ya resueltos (specific_bias_compliance in {Valid, Invalid})."""
    audit_df = efficiency_df.merge(
        unified_df[['id', 'market_bias', 'calc_edge', 'long_prob', 'short_prob', 'no_trade_prob']],
        on='id', how='left',
    )
    audit_df = audit_df[audit_df['specific_bias_compliance'].isin(['Valid', 'Invalid'])].copy()
    audit_df['is_valid'] = (audit_df['specific_bias_compliance'] == 'Valid').astype(int)
    audit_df['theoretical_dir'] = np.where(audit_df['calc_edge'] >= 0, 'Long', 'Short')
    audit_df['primary_prob'] = np.where(audit_df['theoretical_dir'] == 'Long', audit_df['long_prob'], audit_df['short_prob'])
    audit_df['bias_group'] = np.where(audit_df['market_bias'] == 'Choppy / Neutral', 'Choppy', 'Direccional (Bull/Bear)')
    return audit_df


def win_rate_by_market_bias(audit_df: pd.DataFrame) -> pd.DataFrame:
    out = audit_df.groupby('market_bias')['is_valid'].agg(N_Setups='count', Win_Rate='mean').sort_values('Win_Rate', ascending=False)
    out['Win_Rate_%'] = (out['Win_Rate'] * 100).round(2)
    return out[['N_Setups', 'Win_Rate_%']].reset_index()


def win_rate_by_primary_prob_tercile(audit_df: pd.DataFrame) -> pd.DataFrame:
    bins = pd.qcut(audit_df['primary_prob'], q=3, duplicates='drop').rename('primary_prob_tercil')
    out = audit_df.groupby(bins, observed=False)['is_valid'].agg(N_Setups='count', Win_Rate='mean')
    out['Win_Rate_%'] = (out['Win_Rate'] * 100).round(2)
    return out[['N_Setups', 'Win_Rate_%']].reset_index()


def win_rate_cross_bias_group_x_prob(audit_df: pd.DataFrame) -> pd.DataFrame:
    bins = pd.qcut(audit_df['primary_prob'], q=3, duplicates='drop').rename('primary_prob_tercil')
    out = audit_df.groupby(['bias_group', bins], observed=False)['is_valid'].agg(N_Setups='count', Win_Rate='mean')
    out['Win_Rate_%'] = (out['Win_Rate'] * 100).round(2)
    return out[['N_Setups', 'Win_Rate_%']].reset_index()


def structural_resolution_by_market_bias(audit_df: pd.DataFrame) -> pd.DataFrame:
    return pd.crosstab(audit_df['structural_resolution'], audit_df['market_bias'], margins=True, margins_name='Total').reset_index()


def structural_resolution_by_prob_tercile(audit_df: pd.DataFrame) -> pd.DataFrame:
    bins = pd.qcut(audit_df['primary_prob'], q=3, duplicates='drop').rename('primary_prob_tercil')
    return pd.crosstab(audit_df['structural_resolution'], bins, margins=True, margins_name='Total').reset_index()


def accuracy_by_bias_a(audit_df: pd.DataFrame) -> pd.DataFrame:
    out = audit_df.groupby('bias_a')['is_valid'].agg(N_Total='count', N_Valid='sum', Win_Rate='mean')
    out['N_Invalid'] = out['N_Total'] - out['N_Valid']
    out['Win_Rate_%'] = (out['Win_Rate'] * 100).round(2)
    return out[['N_Total', 'N_Valid', 'N_Invalid', 'Win_Rate_%']].sort_values('N_Total', ascending=False).reset_index()


def false_regime_rate_distribution(audit_df: pd.DataFrame) -> pd.DataFrame:
    out = audit_df['false_regime_rate'].value_counts().rename('N_Setups').to_frame()
    out['%'] = (out['N_Setups'] / out['N_Setups'].sum() * 100).round(2)
    return out.reset_index(names='false_regime_rate')


def false_regime_rate_precision_recall(audit_df: pd.DataFrame) -> dict:
    tp = (audit_df['false_regime_rate'] == 'True Positive').sum()
    fp = (audit_df['false_regime_rate'] == 'False Positive').sum()
    tn = (audit_df['false_regime_rate'] == 'True Negative').sum()
    fn = (audit_df['false_regime_rate'] == 'False Negative').sum()
    fs = (audit_df['false_regime_rate'] == 'False Signal').sum()
    return {
        'True_Positive': int(tp), 'False_Positive': int(fp), 'True_Negative': int(tn),
        'False_Negative': int(fn), 'False_Signal': int(fs),
        'Precision': round(tp / (tp + fp), 4) if (tp + fp) else None,
        'Recall': round(tp / (tp + fn), 4) if (tp + fn) else None,
    }


def false_regime_rate_by_bias_a(audit_df: pd.DataFrame) -> pd.DataFrame:
    return pd.crosstab(audit_df['bias_a'], audit_df['false_regime_rate'], margins=True, margins_name='Total').reset_index()


def false_regime_rate_by_market_bias(audit_df: pd.DataFrame) -> pd.DataFrame:
    return pd.crosstab(audit_df['market_bias'], audit_df['false_regime_rate'], margins=True, margins_name='Total').reset_index()


def false_regime_rate_by_prob_tercile(audit_df: pd.DataFrame) -> pd.DataFrame:
    bins = pd.qcut(audit_df['primary_prob'], q=3, duplicates='drop').rename('primary_prob_tercil')
    return pd.crosstab(bins, audit_df['false_regime_rate'], margins=True, margins_name='Total').reset_index()


def edge_validation_integrity_summary(unified_df: pd.DataFrame, efficiency_df: pd.DataFrame) -> dict:
    """Sección 4.A: de los setups 'Valid', ¿el structural_mfe registrado confirma
    que el precio llegó a edge_validation_price? (chequeo de integridad, solo lectura)."""
    audit_df = _build_audit_df(unified_df, efficiency_df)
    valid_df = audit_df[audit_df['is_valid'] == 1].merge(
        unified_df[['id', 'edge_validation_price', 'created_at']].rename(columns={'created_at': 'analysis_date'}),
        on='id', how='left',
    )
    has_data = valid_df['edge_validation_price'].notna() & valid_df['structural_mfe'].notna()
    checkable_df = valid_df[has_data].copy()
    checkable_df['reached_validation'] = np.where(
        checkable_df['theoretical_dir'] == 'Long',
        checkable_df['structural_mfe'] >= checkable_df['edge_validation_price'],
        checkable_df['structural_mfe'] <= checkable_df['edge_validation_price'],
    )
    return {
        'N_Valid_total': len(valid_df),
        'N_Verificable': len(checkable_df),
        'N_Alcanzo_validacion': int(checkable_df['reached_validation'].sum()),
        'N_NO_alcanzo_validacion': int((~checkable_df['reached_validation']).sum()),
        'N_Sin_datos_para_verificar': int((~has_data).sum()),
    }


def edge_validation_not_reached_detail(unified_df: pd.DataFrame, efficiency_df: pd.DataFrame) -> pd.DataFrame:
    audit_df = _build_audit_df(unified_df, efficiency_df)
    valid_df = audit_df[audit_df['is_valid'] == 1].merge(
        unified_df[['id', 'edge_validation_price', 'created_at']].rename(columns={'created_at': 'analysis_date'}),
        on='id', how='left',
    )
    has_data = valid_df['edge_validation_price'].notna() & valid_df['structural_mfe'].notna()
    checkable_df = valid_df[has_data].copy()
    checkable_df['reached_validation'] = np.where(
        checkable_df['theoretical_dir'] == 'Long',
        checkable_df['structural_mfe'] >= checkable_df['edge_validation_price'],
        checkable_df['structural_mfe'] <= checkable_df['edge_validation_price'],
    )
    cols = ['id', 'analysis_date', 'bias_a', 'theoretical_dir', 'calc_edge', 'edge_validation_price', 'structural_mfe']
    return checkable_df[~checkable_df['reached_validation']][cols].reset_index(drop=True)

In [199]:
STRUCTURAL_TARGET_RR = 2.2  # os.getenv("STRUCTURAL_TARGET_RR", "2.2") en el .env real


def _build_struct_df(unified_df: pd.DataFrame, efficiency_df: pd.DataFrame) -> pd.DataFrame:
    """Base de Secciones 5-6/11-12: todos los análisis (no solo los resueltos),
    con structural_entry_price, direcciones, y MAE/MFE crudos y normalizados a R."""
    struct_df = unified_df.merge(efficiency_df[['id', 'structural_mae', 'structural_mfe']], on='id', how='left')

    with engine.connect() as conn:
        tactical_entry_df = _load_tactical_slice(conn, ["entry_price", "order_filled"])
    # Sin filtrar por order_filled — entry_price queda disponible para TODA la muestra (ver Sección 5).
    struct_df = struct_df.merge(tactical_entry_df, on='id', how='left')

    struct_df['structural_entry_price'] = np.where(
        struct_df['edge_validation_price'].notna() & struct_df['structural_invalidation'].notna(),
        (struct_df['edge_validation_price'] + STRUCTURAL_TARGET_RR * struct_df['structural_invalidation']) / (1 + STRUCTURAL_TARGET_RR),
        np.nan,
    )
    struct_df['direction_from_bias'] = np.select(
        [struct_df['market_bias'] == 'Bullish', struct_df['market_bias'] == 'Bearish'],
        ['Long', 'Short'], default=None,
    )
    struct_df['theoretical_dir'] = np.where(struct_df['calc_edge'] >= 0, 'Long', 'Short')
    struct_df['primary_prob'] = np.where(struct_df['theoretical_dir'] == 'Long', struct_df['long_prob'], struct_df['short_prob'])

    struct_df['mae_past_entry'] = np.where(
        struct_df['theoretical_dir'] == 'Long',
        struct_df['structural_entry_price'] - struct_df['structural_mae'],
        struct_df['structural_mae'] - struct_df['structural_entry_price'],
    )
    struct_df['mfe_past_entry'] = np.where(
        struct_df['theoretical_dir'] == 'Long',
        struct_df['structural_mfe'] - struct_df['structural_entry_price'],
        struct_df['structural_entry_price'] - struct_df['structural_mfe'],
    )
    struct_df['dist_mark_entry'] = struct_df['mark_price'] - struct_df['structural_entry_price']
    struct_df['dist_mark_entry_abs'] = struct_df['dist_mark_entry'].abs()

    struct_df['one_r'] = (struct_df['structural_entry_price'] - struct_df['structural_invalidation']).abs()
    struct_df['mae_R'] = struct_df['mae_past_entry'] / struct_df['one_r']
    struct_df['mfe_R'] = struct_df['mfe_past_entry'] / struct_df['one_r']
    struct_df['edge_mag_tercile'] = pd.qcut(struct_df['calc_edge'].abs(), q=3, duplicates='drop')
    return struct_df


def _flatten_agg_cols(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = ['_'.join(str(x) for x in c if str(x)) for c in d.columns]
    return d.reset_index()


def reached_structural_entry_by_bias(struct_df: pd.DataFrame) -> pd.DataFrame:
    """① ¿Qué market_bias llega más seguido al structural_entry_price? (usa
    theoretical_dir, no direction_from_bias, para incluir Choppy)."""
    checkable = struct_df[
        struct_df['theoretical_dir'].notna() & struct_df['structural_mae'].notna() & struct_df['structural_entry_price'].notna()
    ].copy()
    checkable['reached_entry'] = np.where(
        checkable['theoretical_dir'] == 'Short',
        checkable['structural_mae'] >= checkable['structural_entry_price'],
        checkable['structural_mae'] <= checkable['structural_entry_price'],
    )
    out = checkable.groupby('market_bias')['reached_entry'].agg(N_Verificable='count', Reached='mean')
    out['Reached_%'] = (out['Reached'] * 100).round(2)
    return out[['N_Verificable', 'Reached_%']].reset_index()


def mae_mfe_raw_by_market_bias(struct_df: pd.DataFrame) -> pd.DataFrame:
    """MAE/MFE crudos (precio, no comparable entre fechas) por market_bias."""
    return _flatten_agg_cols(struct_df.groupby('market_bias')[['structural_mae', 'structural_mfe']].agg(['count', 'mean', 'median']).round(2))


def mae_mfe_past_entry_by_market_bias(struct_df: pd.DataFrame) -> pd.DataFrame:
    """Distancia más allá de structural_entry_price (puntos), por market_bias."""
    return _flatten_agg_cols(struct_df.groupby('market_bias')[['mae_past_entry', 'mfe_past_entry']].agg(['count', 'mean', 'median']).round(2))


def mae_mfe_past_entry_by_prob_tercile(struct_df: pd.DataFrame) -> pd.DataFrame:
    bins = pd.qcut(struct_df['primary_prob'], q=3, duplicates='drop').rename('primary_prob_tercil')
    return _flatten_agg_cols(struct_df.groupby(bins, observed=False)[['mae_past_entry', 'mfe_past_entry']].agg(['count', 'mean']).round(2))


def mark_price_distance_to_entry_by_bias(struct_df: pd.DataFrame) -> pd.DataFrame:
    return _flatten_agg_cols(struct_df.groupby('market_bias')[['dist_mark_entry', 'dist_mark_entry_abs']].agg(['count', 'mean', 'median']).round(2))


def one_r_distribution(struct_df: pd.DataFrame) -> pd.DataFrame:
    return struct_df['one_r'].describe().round(2).to_frame().T.reset_index(drop=True)


def mae_mfe_R_by_market_bias(struct_df: pd.DataFrame) -> pd.DataFrame:
    return _flatten_agg_cols(struct_df.groupby('market_bias')[['mae_R', 'mfe_R']].agg(['count', 'mean', 'median']).round(3))


def structural_potential_vs_realized(struct_df: pd.DataFrame, tactical_df: pd.DataFrame) -> pd.DataFrame:
    """mfe_R/mae_R (potencial estructural, todos los análisis) vs r_multiple
    (realizado, solo trades con order_filled=True y r_multiple no nulo)."""
    realized_df = struct_df.merge(tactical_df, on='id', how='inner')
    executed_df = realized_df[realized_df['order_filled'].fillna(True).astype(bool) & realized_df['r_multiple'].notna()]
    out = pd.DataFrame({
        'N': [struct_df['mfe_R'].notna().sum(), struct_df['mae_R'].notna().sum(), len(executed_df)],
        'Mean': [struct_df['mfe_R'].mean(), struct_df['mae_R'].mean(), executed_df['r_multiple'].mean()],
        'Median': [struct_df['mfe_R'].median(), struct_df['mae_R'].median(), executed_df['r_multiple'].median()],
    }, index=['mfe_R (potencial estructural, todos)', 'mae_R (riesgo estructural, todos)', 'r_multiple (realizado, ejecutados)'])
    return out.round(3).reset_index(names='Serie')


def mae_R_by_specific_bias_compliance(audit_df: pd.DataFrame, struct_df: pd.DataFrame) -> pd.DataFrame:
    signal_df = audit_df.merge(struct_df[['id', 'mae_R']], on='id', how='left')
    signal_df = signal_df[signal_df['mae_R'].notna()]
    return signal_df.groupby('specific_bias_compliance')['mae_R'].agg(['count', 'mean', 'median']).round(3).reset_index()


def win_rate_by_mae_R_tercile(audit_df: pd.DataFrame, struct_df: pd.DataFrame) -> pd.DataFrame:
    signal_df = audit_df.merge(struct_df[['id', 'mae_R']], on='id', how='left')
    signal_df = signal_df[signal_df['mae_R'].notna()].copy()
    bins = pd.qcut(signal_df['mae_R'], q=3, duplicates='drop').rename('mae_R_tercil')
    out = signal_df.groupby(bins, observed=False)['is_valid'].agg(N_Setups='count', Win_Rate='mean')
    out['Win_Rate_%'] = (out['Win_Rate'] * 100).round(2)
    return out[['N_Setups', 'Win_Rate_%']].reset_index()


def mae_R_by_false_regime_rate(audit_df: pd.DataFrame, struct_df: pd.DataFrame) -> pd.DataFrame:
    signal_df = audit_df.merge(struct_df[['id', 'mae_R']], on='id', how='left')
    signal_df = signal_df[signal_df['mae_R'].notna()]
    return signal_df.groupby('false_regime_rate')['mae_R'].agg(['count', 'mean', 'median']).round(3).reset_index()


def entry_timing_summary(struct_df: pd.DataFrame) -> dict:
    """Fase 3: ¿mark_price ya cruzó structural_entry_price al momento del review?"""
    timing_df = struct_df[
        struct_df['mark_price'].notna() & struct_df['structural_entry_price'].notna() & struct_df['theoretical_dir'].notna()
    ].copy()
    timing_df['crossed_entry'] = np.where(
        timing_df['theoretical_dir'] == 'Long',
        timing_df['mark_price'] <= timing_df['structural_entry_price'],
        timing_df['mark_price'] >= timing_df['structural_entry_price'],
    )
    return {
        'N_Verificable': len(timing_df),
        'N_Entrada_ya_cruzada': int(timing_df['crossed_entry'].sum()),
        'Pct_Entrada_ya_cruzada': round(timing_df['crossed_entry'].mean() * 100, 2) if len(timing_df) else None,
    }


def entry_timing_by_market_bias(struct_df: pd.DataFrame) -> pd.DataFrame:
    timing_df = struct_df[
        struct_df['mark_price'].notna() & struct_df['structural_entry_price'].notna() & struct_df['theoretical_dir'].notna()
    ].copy()
    timing_df['crossed_entry'] = np.where(
        timing_df['theoretical_dir'] == 'Long',
        timing_df['mark_price'] <= timing_df['structural_entry_price'],
        timing_df['mark_price'] >= timing_df['structural_entry_price'],
    )
    out = timing_df.groupby('market_bias')['crossed_entry'].agg(N='count', Crossed_Rate='mean')
    out['Crossed_%'] = (out['Crossed_Rate'] * 100).round(2)
    return out[['N', 'Crossed_%']].reset_index()

In [200]:
def _build_px_df(unified_df: pd.DataFrame, efficiency_df: pd.DataFrame, layer_df: pd.DataFrame) -> pd.DataFrame:
    """Base de Sección 7: analysis_layer pivoteado (P0..P4) + market_bias/calc_edge/
    specific_bias_compliance, con Market_Outcome, consensus_count y final_sign."""
    score_pivot = layer_df.pivot(index='trade_id', columns='layer_name', values='score')
    score_pivot = score_pivot.rename(columns={f'P{i}': f'p{i}_score' for i in range(5)})
    score_pivot = score_pivot.reset_index().rename(columns={'trade_id': 'id'})

    px_df = unified_df[['id', 'market_bias', 'calc_edge']].merge(
        efficiency_df[['id', 'specific_bias_compliance']], on='id', how='left'
    ).merge(score_pivot, on='id', how='left')

    px_df['Market_Outcome'] = px_df.apply(get_market_outcome, axis=1)
    px_df['final_sign'] = np.sign(px_df['calc_edge'])
    score_cols = ['p0_score', 'p1_score', 'p2_score', 'p3_score', 'p4_score']

    def _consensus(row):
        return sum(1 for c in score_cols if row[c] != 0 and np.sign(row[c]) == row['final_sign'])

    px_df['consensus_count'] = px_df.apply(_consensus, axis=1)
    px_df['is_valid'] = (px_df['specific_bias_compliance'] == 'Valid').astype(int)
    return px_df


def rescue_metric_by_parameter(px_df: pd.DataFrame) -> pd.DataFrame:
    """Métrica 'Rescate': entre los fallos del sistema (Bullish/Bearish + Invalid),
    ¿qué % de las veces cada Px individual sí tenía el signo correcto?"""
    invalid_df = px_df[(px_df['specific_bias_compliance'] == 'Invalid') & (px_df['market_bias'].isin(['Bullish', 'Bearish']))]
    score_cols = ['p0_score', 'p1_score', 'p2_score', 'p3_score', 'p4_score']
    rows = []
    for col in score_cols:
        active = invalid_df[invalid_df[col] != 0]
        rescued = active[np.sign(active[col]) == np.sign(active['Market_Outcome'])]
        rows.append({
            'Parametro': col.split('_')[0].upper(),
            'N_Fallos_Sistema': len(invalid_df),
            'N_Activo': len(active),
            'N_Rescatado': len(rescued),
            'Rescate_%': round(len(rescued) / len(invalid_df) * 100, 1) if len(invalid_df) else 0.0,
        })
    return pd.DataFrame(rows).sort_values('Rescate_%', ascending=False).reset_index(drop=True)


def rescued_cases_detail(px_df: pd.DataFrame) -> pd.DataFrame:
    invalid_df = px_df[(px_df['specific_bias_compliance'] == 'Invalid') & (px_df['market_bias'].isin(['Bullish', 'Bearish']))].copy()
    score_cols = ['p0_score', 'p1_score', 'p2_score', 'p3_score', 'p4_score']

    def _rescuers(row):
        return [c.split('_')[0].upper() for c in score_cols if row[c] != 0 and np.sign(row[c]) == np.sign(row['Market_Outcome'])]

    invalid_df['rescatado_por'] = invalid_df.apply(_rescuers, axis=1)
    cols = ['id', 'market_bias', 'specific_bias_compliance', 'Market_Outcome'] + score_cols + ['rescatado_por']
    out = invalid_df[invalid_df['rescatado_por'].apply(len) > 0][cols].copy()
    out['rescatado_por'] = out['rescatado_por'].apply(lambda lst: ', '.join(lst))
    return out.reset_index(drop=True)


def win_rate_by_consensus_count(px_df: pd.DataFrame) -> pd.DataFrame:
    resolved_df = px_df[px_df['specific_bias_compliance'].notna()]
    out = resolved_df.groupby('consensus_count')['is_valid'].agg(N='count', Win_Rate='mean')
    out['Win_Rate_%'] = (out['Win_Rate'] * 100).round(1)
    return out[['N', 'Win_Rate_%']].reset_index()


def p0_accuracy_by_bias_a(px_df: pd.DataFrame, efficiency_df: pd.DataFrame) -> pd.DataFrame:
    resolved_df = px_df[px_df['specific_bias_compliance'].notna()]
    active_p0 = resolved_df[resolved_df['p0_score'] != 0].merge(efficiency_df[['id', 'bias_a']], on='id', how='left').copy()
    active_p0['p0_correct'] = (np.sign(active_p0['p0_score']) == active_p0['Market_Outcome']).astype(int)
    out = active_p0.groupby('bias_a')['p0_correct'].agg(N='count', Win_Rate='mean').sort_values('N', ascending=False)
    out['Win_Rate_%'] = (out['Win_Rate'] * 100).round(1)
    return out[['N', 'Win_Rate_%']].reset_index()


def edge_weight_simulation(px_df: pd.DataFrame) -> pd.DataFrame:
    """Recalcula el signo de calc_edge con distintos juegos de pesos P0-P4 y lo
    compara contra Market_Outcome, solo en setups Bullish/Bearish resueltos
    (Market_Outcome != 0). Pesos actuales: core/math_engine.py::calculate_edge_score."""
    eval_df = px_df[px_df['Market_Outcome'] != 0]
    weight_sets = {
        'Actual (0.30,0.25,0.15,0.10,0.20)': (0.30, 0.25, 0.15, 0.10, 0.20),
        'P0+P1 solo (0.55,0.45,0,0,0)': (0.55, 0.45, 0, 0, 0),
        'P0 solo (1,0,0,0,0)': (1, 0, 0, 0, 0),
        'P0 dominante (0.7,0.3,0,0,0)': (0.7, 0.3, 0, 0, 0),
    }
    rows = []
    for name, (w0, w1, w2, w3, w4) in weight_sets.items():
        sim = (w0 * eval_df['p0_score'] + w1 * eval_df['p1_score'] + w2 * eval_df['p2_score']
               + w3 * eval_df['p3_score'] + w4 * eval_df['p4_score'])
        sim_sign = np.sign(sim)
        correct = (sim_sign == eval_df['Market_Outcome'])
        rows.append({
            'Pesos': name, 'N': len(eval_df), 'Aciertos': int(correct.sum()),
            'Accuracy_%': round(correct.mean() * 100, 1), 'Indeterminados (sim=0)': int((sim_sign == 0).sum()),
        })
    return pd.DataFrame(rows)

In [201]:
TF_PATTERN = re.compile(r'(\d+[A-Za-z]*)\s*-\s*B([LR])', re.IGNORECASE)
TFS = ['1DT', '12H', '4H', '1H']


def _normalize_tf(tf: str):
    tf = tf.upper()
    if tf == '1DT':
        return '1DT'
    if tf in ('12HT', '12H', '12T'):
        return '12H'
    if tf in ('4HT', '4H', '4T'):
        return '4H'
    if tf in ('1HT', '1H', '1T'):
        return '1H'
    return None


def _build_tf_df(unified_df: pd.DataFrame, efficiency_df: pd.DataFrame, layer_df: pd.DataFrame):
    """Base de Secciones 9-10: parsea la thesis de P0 ('{temporalidad}-B{L|R}')
    a una columna tf_<TF> por cada una de las 4 temporalidades (1DT/12H/4H/1H),
    normalizando variantes de abreviatura. Se une con market_bias/calc_edge/
    specific_bias_compliance/bias_a y se calcula Market_Outcome/htf_majority."""
    layer_p0 = layer_df[layer_df['layer_name'] == 'P0'][['trade_id', 'score', 'thesis']].copy()
    records = []
    for _, r in layer_p0.iterrows():
        found = {}
        for tf_raw, direction in TF_PATTERN.findall(r['thesis']):
            tf = _normalize_tf(tf_raw)
            if tf and tf not in found:
                found[tf] = 1 if direction.upper() == 'L' else -1
        rec = {'id': r['trade_id'], 'p0_score': r['score']}
        for tf in TFS:
            rec[f'tf_{tf}'] = found.get(tf, np.nan)
        records.append(rec)
    tf_df = pd.DataFrame(records)

    tf_full = tf_df.merge(unified_df[['id', 'market_bias', 'calc_edge']], on='id', how='left')
    tf_full = tf_full.merge(efficiency_df[['id', 'specific_bias_compliance']], on='id', how='left')
    tf_full = tf_full[tf_full['specific_bias_compliance'].notna()].copy()
    tf_full['Market_Outcome'] = tf_full.apply(get_market_outcome, axis=1)

    def _majority_htf(row):
        vals = [v for v in [row['tf_1DT'], row['tf_12H'], row['tf_4H']] if pd.notna(v)]
        if not vals:
            return np.nan
        return 1 if sum(vals) > 0 else (-1 if sum(vals) < 0 else 0)

    tf_full['htf_majority'] = tf_full.apply(_majority_htf, axis=1)
    return tf_df, tf_full


def tf_coverage(tf_df: pd.DataFrame) -> pd.DataFrame:
    out = tf_df[[f'tf_{tf}' for tf in TFS]].notna().sum().to_frame('N_con_match')
    return out.reset_index(names='temporalidad')


def tf_convergence_with_p0(tf_df: pd.DataFrame) -> pd.DataFrame:
    p0_active_tf = tf_df[tf_df['p0_score'] != 0].copy()
    p0_active_tf['p0_dir'] = np.sign(p0_active_tf['p0_score'])
    rows = []
    for tf in TFS:
        sub = p0_active_tf[p0_active_tf[f'tf_{tf}'].notna()]
        match = (sub[f'tf_{tf}'] == sub['p0_dir']).mean()
        rows.append({'Temporalidad': tf, 'N': len(sub), 'Convergencia_%': round(match * 100, 1)})
    return pd.DataFrame(rows)


def tf_win_rate_vs_market_outcome(tf_full: pd.DataFrame) -> pd.DataFrame:
    tf_eval = tf_full[tf_full['Market_Outcome'] != 0]
    rows = []
    for tf in TFS:
        sub = tf_eval[tf_eval[f'tf_{tf}'].notna()]
        correct = (sub[f'tf_{tf}'] == sub['Market_Outcome'])
        rows.append({'Temporalidad': tf, 'N': len(sub), 'Win_Rate_%': round(correct.mean() * 100, 1)})
    return pd.DataFrame(rows).sort_values('Win_Rate_%', ascending=False)


def tf_1h_vs_htf_majority(tf_full: pd.DataFrame) -> pd.DataFrame:
    """Cuando tf_1H discrepa vs. coincide con la mayoría de 1DT/12H/4H, ¿quién acierta más?"""
    has_both = tf_full['tf_1H'].notna() & tf_full['htf_majority'].notna()
    tf_disagree = tf_full[has_both & (tf_full['tf_1H'] != tf_full['htf_majority'])]
    tf_agree = tf_full[has_both & (tf_full['tf_1H'] == tf_full['htf_majority'])]
    ev_dis = tf_disagree[tf_disagree['Market_Outcome'] != 0]
    ev_agr = tf_agree[tf_agree['Market_Outcome'] != 0]
    return pd.DataFrame([
        {'Caso': '1H discrepa de mayoría 1DT/12H/4H', 'N': len(ev_dis),
         'Acierto_1H_%': round((ev_dis['tf_1H'] == ev_dis['Market_Outcome']).mean() * 100, 1) if len(ev_dis) else None,
         'Acierto_mayoria_%': round((ev_dis['htf_majority'] == ev_dis['Market_Outcome']).mean() * 100, 1) if len(ev_dis) else None},
        {'Caso': '1H coincide con mayoría', 'N': len(ev_agr),
         'Acierto_1H_%': round((ev_agr['tf_1H'] == ev_agr['Market_Outcome']).mean() * 100, 1) if len(ev_agr) else None,
         'Acierto_mayoria_%': None},
    ])


def tf_1h_win_rate_by_bias_a(tf_full: pd.DataFrame, efficiency_df: pd.DataFrame) -> pd.DataFrame:
    tf_eval = tf_full[tf_full['Market_Outcome'] != 0]
    sub_1h = tf_eval[tf_eval['tf_1H'].notna()].merge(efficiency_df[['id', 'bias_a']], on='id', how='left').copy()
    sub_1h['correct'] = (sub_1h['tf_1H'] == sub_1h['Market_Outcome']).astype(int)
    out = sub_1h.groupby('bias_a')['correct'].agg(N='count', Win_Rate='mean').sort_values('N', ascending=False)
    out['Win_Rate_%'] = (out['Win_Rate'] * 100).round(1)
    return out[['N', 'Win_Rate_%']].reset_index()


def tf_1h_vs_p0_head_to_head(tf_full: pd.DataFrame) -> pd.DataFrame:
    tf_eval = tf_full[tf_full['Market_Outcome'] != 0]
    head_to_head = tf_eval[(tf_eval['p0_score'] != 0) & (tf_eval['tf_1H'].notna())].copy()
    head_to_head['p0_dir'] = np.sign(head_to_head['p0_score'])
    agree = head_to_head[head_to_head['p0_dir'] == head_to_head['tf_1H']]
    disagree = head_to_head[head_to_head['p0_dir'] != head_to_head['tf_1H']]
    return pd.DataFrame([
        {'Métrica': 'N total', 'Valor': len(head_to_head)},
        {'Métrica': 'P0 accuracy (todo el subset)', 'Valor': round((head_to_head['p0_dir'] == head_to_head['Market_Outcome']).mean() * 100, 1) if len(head_to_head) else None},
        {'Métrica': '1H accuracy (todo el subset)', 'Valor': round((head_to_head['tf_1H'] == head_to_head['Market_Outcome']).mean() * 100, 1) if len(head_to_head) else None},
        {'Métrica': 'N donde P0 y 1H coinciden', 'Valor': len(agree)},
        {'Métrica': 'Accuracy conjunta cuando coinciden', 'Valor': round((agree['p0_dir'] == agree['Market_Outcome']).mean() * 100, 1) if len(agree) else None},
        {'Métrica': 'N donde P0 y 1H discrepan', 'Valor': len(disagree)},
        {'Métrica': 'P0 accuracy cuando discrepan', 'Valor': round((disagree['p0_dir'] == disagree['Market_Outcome']).mean() * 100, 1) if len(disagree) else None},
        {'Métrica': '1H accuracy cuando discrepan', 'Valor': round((disagree['tf_1H'] == disagree['Market_Outcome']).mean() * 100, 1) if len(disagree) else None},
    ])

In [202]:
def reach_entry_by_market_bias(struct_df: pd.DataFrame) -> pd.DataFrame:
    checkable = struct_df[
        struct_df['theoretical_dir'].notna() & struct_df['structural_mae'].notna() & struct_df['structural_entry_price'].notna()
    ].copy()
    checkable['reached_entry'] = np.where(
        checkable['theoretical_dir'] == 'Short',
        checkable['structural_mae'] >= checkable['structural_entry_price'],
        checkable['structural_mae'] <= checkable['structural_entry_price'],
    )
    out = checkable.groupby(['market_bias', 'reached_entry']).size().unstack(fill_value=0)
    out = out.rename(columns={False: 'No_Llega', True: 'Llega'})
    for c in ('No_Llega', 'Llega'):
        if c not in out.columns:
            out[c] = 0
    out['Llega_%'] = (out['Llega'] / (out['Llega'] + out['No_Llega']) * 100).round(1)
    return out.reset_index()


def reach_entry_by_prob_tercile(struct_df: pd.DataFrame) -> pd.DataFrame:
    checkable = struct_df[
        struct_df['theoretical_dir'].notna() & struct_df['structural_mae'].notna() & struct_df['structural_entry_price'].notna()
    ].copy()
    checkable['reached_entry'] = np.where(
        checkable['theoretical_dir'] == 'Short',
        checkable['structural_mae'] >= checkable['structural_entry_price'],
        checkable['structural_mae'] <= checkable['structural_entry_price'],
    )
    bins = pd.qcut(checkable['primary_prob'], q=3, duplicates='drop').rename('primary_prob_tercil')
    out = checkable.groupby([bins, 'reached_entry'], observed=False).size().unstack(fill_value=0)
    out = out.rename(columns={False: 'No_Llega', True: 'Llega'})
    for c in ('No_Llega', 'Llega'):
        if c not in out.columns:
            out[c] = 0
    out['Llega_%'] = (out['Llega'] / (out['Llega'] + out['No_Llega']) * 100).round(1)
    return out.reset_index()


def mae_R_sweet_spot(struct_df: pd.DataFrame, efficiency_df: pd.DataFrame) -> pd.DataFrame:
    """Win rate por tramo fino de mae_R, restringido a setups resueltos."""
    checkable = struct_df[
        struct_df['theoretical_dir'].notna() & struct_df['structural_mae'].notna() & struct_df['structural_entry_price'].notna()
    ].copy()
    entry_resolved = checkable.merge(efficiency_df[['id', 'specific_bias_compliance']], on='id', how='left')
    entry_resolved = entry_resolved[entry_resolved['specific_bias_compliance'].notna()].copy()
    entry_resolved['is_valid'] = (entry_resolved['specific_bias_compliance'] == 'Valid').astype(int)

    bin_edges = [-10, -0.01, 0.15, 0.35, 0.6, 0.9, 1.3, 10]
    bin_labels = ['No llega (<0)', '0 a 0.15R', '0.15 a 0.35R', '0.35 a 0.6R', '0.6 a 0.9R', '0.9 a 1.3R', '>1.3R']
    entry_resolved['mae_R_fino'] = pd.cut(entry_resolved['mae_R'], bins=bin_edges, labels=bin_labels)

    out = entry_resolved.groupby('mae_R_fino', observed=False)['is_valid'].agg(N='count', Win_Rate='mean')
    out['Win_Rate_%'] = (out['Win_Rate'] * 100).round(1)
    return out[['N', 'Win_Rate_%']].reset_index()


def structural_reference_table_by_market_bias(struct_df: pd.DataFrame) -> pd.DataFrame:
    return _flatten_agg_cols(struct_df.groupby('market_bias')[['structural_mae', 'structural_mfe', 'mae_R', 'mfe_R']].agg(['count', 'mean', 'median']).round(2))


def structural_reference_table_by_edge_magnitude(struct_df: pd.DataFrame) -> pd.DataFrame:
    edge_mag = struct_df['calc_edge'].abs().rename('edge_mag')
    bins = pd.qcut(edge_mag, q=3, duplicates='drop').rename('abs_calc_edge_tercil')
    return _flatten_agg_cols(struct_df.groupby(bins, observed=False)[['structural_mae', 'structural_mfe', 'mae_R', 'mfe_R']].agg(['count', 'mean', 'median']).round(2))


def structural_reference_table_by_prob_tercile(struct_df: pd.DataFrame) -> pd.DataFrame:
    bins = pd.qcut(struct_df['primary_prob'], q=3, duplicates='drop').rename('primary_prob_tercil')
    return _flatten_agg_cols(struct_df.groupby(bins, observed=False)[['structural_mae', 'structural_mfe', 'mae_R', 'mfe_R']].agg(['count', 'mean', 'median']).round(2))

In [203]:
def _reach_rate_by_bias_icd(df: pd.DataFrame, cond_col: str) -> pd.DataFrame:
    out = df.groupby(['market_bias', 'edge_mag_tercile'], observed=False)[cond_col].agg(N_Verificable='count', Reached='mean')
    out['Reached_%'] = (out['Reached'] * 100).round(1)
    return out[['N_Verificable', 'Reached_%']].reset_index()


def mark_crossed_structural_entry_by_bias_icd(struct_df: pd.DataFrame) -> pd.DataFrame:
    """Sección 14①: mark_price cruzó structural_entry_price, por market_bias x edge_mag_tercile."""
    timing_df = struct_df[
        struct_df['mark_price'].notna() & struct_df['structural_entry_price'].notna() & struct_df['theoretical_dir'].notna()
    ].copy()
    timing_df['crossed_entry'] = np.where(
        timing_df['theoretical_dir'] == 'Long',
        timing_df['mark_price'] <= timing_df['structural_entry_price'],
        timing_df['mark_price'] >= timing_df['structural_entry_price'],
    )
    return _reach_rate_by_bias_icd(timing_df, 'crossed_entry')


def mae_reached_structural_entry_by_bias_icd(struct_df: pd.DataFrame) -> pd.DataFrame:
    """Sección 14②: structural_mae alcanzó structural_entry_price, por market_bias x edge_mag_tercile."""
    checkable = struct_df[
        struct_df['theoretical_dir'].notna() & struct_df['structural_mae'].notna() & struct_df['structural_entry_price'].notna()
    ].copy()
    checkable['reached_entry'] = np.where(
        checkable['theoretical_dir'] == 'Short',
        checkable['structural_mae'] >= checkable['structural_entry_price'],
        checkable['structural_mae'] <= checkable['structural_entry_price'],
    )
    return _reach_rate_by_bias_icd(checkable, 'reached_entry')


def mfe_reached_validation_price_by_bias_icd(struct_df: pd.DataFrame) -> pd.DataFrame:
    """Sección 14③: structural_mfe alcanzó edge_validation_price (TP), por market_bias x edge_mag_tercile."""
    checkable = struct_df[
        struct_df['theoretical_dir'].notna() & struct_df['structural_mfe'].notna() & struct_df['edge_validation_price'].notna()
    ].copy()
    checkable['reached_tp'] = np.where(
        checkable['theoretical_dir'] == 'Long',
        checkable['structural_mfe'] >= checkable['edge_validation_price'],
        checkable['structural_mfe'] <= checkable['edge_validation_price'],
    )
    return _reach_rate_by_bias_icd(checkable, 'reached_tp')


def entry_price_vs_structural_by_bias_icd(struct_df: pd.DataFrame) -> pd.DataFrame:
    """Sección 14④: entry_price real (order_filled==True) vs structural_entry_price."""
    checkable = struct_df[
        (struct_df['order_filled'] == True)
        & struct_df['theoretical_dir'].notna() & struct_df['entry_price'].notna() & struct_df['structural_entry_price'].notna()
    ].copy()
    checkable['entry_worse_than_structural'] = np.where(
        checkable['theoretical_dir'] == 'Long',
        checkable['entry_price'] >= checkable['structural_entry_price'],
        checkable['entry_price'] <= checkable['structural_entry_price'],
    )
    return _reach_rate_by_bias_icd(checkable, 'entry_worse_than_structural').rename(columns={'Reached_%': 'Entro_Peor_%'})


def build_all_entries_df(struct_df: pd.DataFrame) -> pd.DataFrame:
    """Sección 14.B/15.B/16.B/17.B: TODOS los registros con entry_price (incluye order_filled=False),
    excluyendo placeholders legacy (entry_price==closing_price y entry_time==exit_time simultaneos)."""
    with engine.connect() as conn:
        entry_extra_df = _load_tactical_slice(conn, ["entry_time", "exit_time", "closing_price"])
    all_entries_df = struct_df.merge(entry_extra_df, on='id', how='left')
    is_legacy_placeholder = (
        all_entries_df['closing_price'].notna()
        & (all_entries_df['entry_price'] == all_entries_df['closing_price'])
        & (all_entries_df['entry_time'] == all_entries_df['exit_time'])
    )
    # entry_price==0 es un placeholder de DB (NOT NULL), no un precio real -- .notna() no lo detecta.
    is_zero_placeholder = all_entries_df['entry_price'] == 0
    return all_entries_df[
        all_entries_df['entry_price'].notna() & ~is_legacy_placeholder & ~is_zero_placeholder
    ].copy()


def entry_price_vs_structural_all_by_bias_icd(all_entries_df: pd.DataFrame) -> pd.DataFrame:
    """Sección 14④.B: mismo calculo que entry_price_vs_structural_by_bias_icd, sin filtrar order_filled."""
    checkable = all_entries_df[
        all_entries_df['theoretical_dir'].notna() & all_entries_df['entry_price'].notna() & all_entries_df['structural_entry_price'].notna()
    ].copy()
    checkable['entry_worse_than_structural'] = np.where(
        checkable['theoretical_dir'] == 'Long',
        checkable['entry_price'] >= checkable['structural_entry_price'],
        checkable['entry_price'] <= checkable['structural_entry_price'],
    )
    return _reach_rate_by_bias_icd(checkable, 'entry_worse_than_structural').rename(columns={'Reached_%': 'Entro_Peor_%'})


def build_chase_df(df: pd.DataFrame, psych_df: pd.DataFrame) -> pd.DataFrame:
    """Sección 15/15.B: entry_chase_R. El filtro order_filled lo decide el caller
    (pasar struct_df[order_filled==True] para la A, all_entries_df para la B)."""
    chase_df = df[
        df['entry_price'].notna() & df['structural_entry_price'].notna() & df['theoretical_dir'].notna()
    ][['id', 'market_bias', 'theoretical_dir', 'entry_price', 'structural_entry_price', 'one_r']].merge(psych_df, on='id', how='left')
    chase_df['entry_chase_R'] = np.where(
        chase_df['theoretical_dir'] == 'Long',
        (chase_df['entry_price'] - chase_df['structural_entry_price']) / chase_df['one_r'],
        (chase_df['structural_entry_price'] - chase_df['entry_price']) / chase_df['one_r'],
    )
    return chase_df


def entry_chase_by_anxiety(chase_df: pd.DataFrame) -> pd.DataFrame:
    return chase_df.groupby('anxiety_level')['entry_chase_R'].agg(['count', 'mean', 'median']).round(3).reset_index()


def entry_chase_by_impatience(chase_df: pd.DataFrame) -> pd.DataFrame:
    return chase_df.groupby('impatience_level')['entry_chase_R'].agg(['count', 'mean', 'median']).round(3).reset_index()


def entry_chase_by_emotion(chase_df: pd.DataFrame) -> pd.DataFrame:
    return chase_df.groupby('primary_emotion')['entry_chase_R'].agg(['count', 'mean', 'median']).round(3).sort_values('mean', ascending=False).reset_index()


X_VALUES_LLM = [round(x, 1) for x in np.arange(0.0, 1.01, 0.1)]
RATIOS_LLM = [2.0, 2.2]


def _hyp_entry_price(df: pd.DataFrame, version: str, x: float) -> pd.Series:
    if version == 'mark_entry':
        return df['mark_price'] + x * (df['structural_entry_price'] - df['mark_price'])
    if version == 'fractal_entry':
        return df['entry_price'] + x * (df['structural_entry_price'] - df['entry_price'])
    raise ValueError(version)


def _reached_hyp(theoretical_dir: pd.Series, entry_hyp: pd.Series, mae: pd.Series):
    return np.where(theoretical_dir == 'Short', mae >= entry_hyp, mae <= entry_hyp)


def sweep_reach_rate(df: pd.DataFrame, version: str, x_values=X_VALUES_LLM) -> pd.DataFrame:
    """Sección 16/16.B: barrido de x (0=mark_price/entry_price, 1=structural_entry_price), tasa de alcance por MAE.
    El filtro order_filled (si aplica) lo decide el caller."""
    base = df.copy()
    anchor_col = 'mark_price' if version == 'mark_entry' else 'entry_price'
    base = base[
        base['theoretical_dir'].notna() & base['structural_mae'].notna()
        & base['structural_entry_price'].notna() & base[anchor_col].notna()
    ].copy()
    rows = []
    for x in x_values:
        entry_hyp = _hyp_entry_price(base, version, x)
        reached = _reached_hyp(base['theoretical_dir'], entry_hyp, base['structural_mae'])
        tmp = base[['market_bias', 'edge_mag_tercile']].copy()
        tmp['x'] = x
        tmp['reached'] = reached
        rows.append(tmp)
    long_df = pd.concat(rows, ignore_index=True)
    out = long_df.groupby(['x', 'market_bias', 'edge_mag_tercile'], observed=False)['reached'].agg(N_Verificable='count', Reached='mean').reset_index()
    out['Reached_%'] = (out['Reached'] * 100).round(1)
    return out[['x', 'market_bias', 'edge_mag_tercile', 'N_Verificable', 'Reached_%']]


def classify_hypothetical_trade(df: pd.DataFrame, version: str, x: float, ratio: float) -> pd.DataFrame:
    """Sección 17/17.B: TP=edge_validation_price, SL=|TP-entry|/ratio. SL rompe -> Perdido; si no y MFE llega a TP -> Ganado; si no, Inconcluso.
    El filtro order_filled (si aplica) lo decide el caller."""
    base = df.copy()
    anchor_col = 'mark_price' if version == 'mark_entry' else 'entry_price'
    base = base[
        base['theoretical_dir'].notna() & base['structural_mae'].notna() & base['structural_mfe'].notna()
        & base['structural_entry_price'].notna() & base['edge_validation_price'].notna() & base[anchor_col].notna()
    ].copy()
    entry_hyp = _hyp_entry_price(base, version, x)
    reached = _reached_hyp(base['theoretical_dir'], entry_hyp, base['structural_mae'])
    tp = base['edge_validation_price']
    risk_dist = (tp - entry_hyp).abs() / ratio
    sl = np.where(base['theoretical_dir'] == 'Long', entry_hyp - risk_dist, entry_hyp + risk_dist)
    sl_hit = np.where(base['theoretical_dir'] == 'Long', base['structural_mae'] <= sl, base['structural_mae'] >= sl)
    tp_hit = np.where(base['theoretical_dir'] == 'Long', base['structural_mfe'] >= tp, base['structural_mfe'] <= tp)
    outcome = np.select([sl_hit, tp_hit], ['Perdido', 'Ganado'], default='Inconcluso')
    out = base[['id', 'market_bias', 'edge_mag_tercile']].copy()
    out['version'] = version
    out['x'] = x
    out['ratio'] = ratio
    out['reached'] = reached
    out['outcome'] = np.where(reached, outcome, 'No_Activado')
    return out


def build_backtest_summary(pairs, x_values=X_VALUES_LLM, ratios=RATIOS_LLM) -> pd.DataFrame:
    """pairs: lista de (df_ya_filtrado, version) -- el caller decide que filas entran para cada version
    (ej. [(struct_df_llm, 'mark_entry'), (struct_df_llm[order_filled==True], 'fractal_entry')] para la A,
    o [(all_entries_df_llm, 'fractal_entry')] para la B)."""
    detail_rows = []
    for df, version in pairs:
        for x in x_values:
            for ratio in ratios:
                detail_rows.append(classify_hypothetical_trade(df, version, x, ratio))
    backtest_detail = pd.concat(detail_rows, ignore_index=True)
    activated = backtest_detail[backtest_detail['reached']]
    summary = (
        activated.groupby(['version', 'x', 'ratio', 'market_bias', 'edge_mag_tercile'], observed=False)['outcome']
        .value_counts().unstack(fill_value=0).reset_index()
    )
    for col in ('Ganado', 'Perdido', 'Inconcluso'):
        if col not in summary.columns:
            summary[col] = 0
    summary['N_Activado'] = summary['Ganado'] + summary['Perdido'] + summary['Inconcluso']
    summary['Win_Rate_%'] = np.where(
        (summary['Ganado'] + summary['Perdido']) > 0,
        (summary['Ganado'] / (summary['Ganado'] + summary['Perdido']) * 100).round(1),
        np.nan,
    )
    summary = summary.rename(columns={'Ganado': 'N_Ganado', 'Perdido': 'N_Perdido', 'Inconcluso': 'N_Inconcluso'})
    return summary[['version', 'x', 'ratio', 'market_bias', 'edge_mag_tercile', 'N_Activado', 'N_Ganado', 'N_Perdido', 'N_Inconcluso', 'Win_Rate_%']]


def best_x_by_segment(backtest_summary: pd.DataFrame, min_n: int = 5) -> pd.DataFrame:
    qualified = backtest_summary[backtest_summary['N_Activado'] >= min_n].dropna(subset=['Win_Rate_%'])
    if len(qualified) == 0:
        return qualified
    best_idx = qualified.groupby(['version', 'ratio', 'market_bias', 'edge_mag_tercile'], observed=True)['Win_Rate_%'].idxmax()
    return qualified.loc[best_idx].sort_values(['version', 'ratio', 'market_bias', 'edge_mag_tercile']).reset_index(drop=True)


In [204]:
audit_df_llm = _build_audit_df(unified_df, efficiency_df)
struct_df_llm = _build_struct_df(unified_df, efficiency_df)
all_entries_df_llm = build_all_entries_df(struct_df_llm)
px_df_llm = _build_px_df(unified_df, efficiency_df, layer_df)
tf_df_llm, tf_full_llm = _build_tf_df(unified_df, efficiency_df, layer_df)

with engine.connect() as conn:
    # order_filled ya viene en struct_df_llm (merge dentro de _build_struct_df)
    tactical_r_df_llm = _load_tactical_slice(conn, ["r_multiple"])
    psych_df_llm = _load_tactical_slice(conn, ["anxiety_level", "impatience_level", "primary_emotion"])

# anxiety_level/impatience_level son ge=1,le=5 en el schema -- 0 es placeholder de DB, no un valor real.
psych_df_llm['anxiety_level'] = psych_df_llm['anxiety_level'].replace(0, np.nan)
psych_df_llm['impatience_level'] = psych_df_llm['impatience_level'].replace(0, np.nan)
psych_df_llm['primary_emotion'] = psych_df_llm['primary_emotion'].replace('nan', np.nan)

llm_sections = []

llm_sections.append(MarkdownSection(
    title="Sección 2 — Edge estructural: win rate por market_bias y por probabilidad",
    narrative=(
        "Win rate estructural = proporción de setups con specific_bias_compliance=='Valid' "
        "sobre los ya resueltos (Valid+Invalid). primary_prob = long_prob si theoretical_dir=='Long', "
        "si no short_prob. bias_group agrupa market_bias en Choppy vs. Direccional (Bull/Bear)."
    ),
    tables=[
        win_rate_by_market_bias(audit_df_llm),
        win_rate_by_primary_prob_tercile(audit_df_llm),
        win_rate_cross_bias_group_x_prob(audit_df_llm),
    ],
    source_functions=[_build_audit_df, win_rate_by_market_bias, win_rate_by_primary_prob_tercile, win_rate_cross_bias_group_x_prob],
))

llm_sections.append(MarkdownSection(
    title="Sección 3 — structural_resolution por bias/probabilidad, y acierto por bias_a",
    narrative="Distribución de structural_resolution cruzada con market_bias y con tercil de primary_prob; y N_Valid/N_Invalid/Win_Rate por bias_a (el tipo de setup predicho por efficiency_audit).",
    tables=[
        structural_resolution_by_market_bias(audit_df_llm),
        structural_resolution_by_prob_tercile(audit_df_llm),
        accuracy_by_bias_a(audit_df_llm),
    ],
    source_functions=[structural_resolution_by_market_bias, structural_resolution_by_prob_tercile, accuracy_by_bias_a],
))

llm_sections.append(MarkdownSection(
    title="Sección 4 — false_regime_rate: matriz de confusión del detector de régimen",
    narrative=(
        "false_regime_rate lo calcula cli/schemas/audit_efficiency.py:60-79 comparando bias_a "
        "(predicción) vs real_bias_b (realidad), tratando No_Bias(Choppy) como clase negativa. "
        "Precision = TP/(TP+FP) sobre los casos donde se predijo un bias específico; "
        "Recall = TP/(TP+FN)."
    ),
    tables=[
        false_regime_rate_distribution(audit_df_llm),
        dict_to_metrics_frame(false_regime_rate_precision_recall(audit_df_llm)),
        false_regime_rate_by_bias_a(audit_df_llm),
        false_regime_rate_by_market_bias(audit_df_llm),
        false_regime_rate_by_prob_tercile(audit_df_llm),
    ],
    source_functions=[false_regime_rate_distribution, false_regime_rate_precision_recall, false_regime_rate_by_bias_a, false_regime_rate_by_market_bias, false_regime_rate_by_prob_tercile],
))

llm_sections.append(MarkdownSection(
    title="Sección 4.A — Integridad: ¿el structural_mfe registrado confirma los 'Valid'?",
    narrative=(
        "Chequeo de integridad, no forma parte del modelo: de los setups 'Valid', ¿structural_mfe "
        "(efficiency_audit) confirma que el precio llegó a edge_validation_price (unified_department)? "
        "Tesis Long: alcanzado si structural_mfe >= edge_validation_price. Tesis Short: alcanzado si "
        "structural_mfe <= edge_validation_price. Se excluyen los casos sin edge_validation_price o "
        "sin structural_mfe cargado."
    ),
    tables=[
        dict_to_metrics_frame(edge_validation_integrity_summary(unified_df, efficiency_df)),
        edge_validation_not_reached_detail(unified_df, efficiency_df),
    ],
    source_functions=[edge_validation_integrity_summary, edge_validation_not_reached_detail],
))

llm_sections.append(MarkdownSection(
    title="Sección 5 — structural_entry_price vs. MAE/MFE",
    narrative=(
        "structural_entry_price = (TP + RR·SL) / (1+RR), TP=edge_validation_price, "
        "SL=structural_invalidation, RR=STRUCTURAL_TARGET_RR (2.2) — core/math_engine.py::"
        "calculate_structural_entry, precio de entrada teórico (pullback), no el entry_price "
        "real del trade. Todas las tablas usan theoretical_dir (signo de calc_edge, cubre "
        "Choppy), no direction_from_bias."
    ),
    tables=[
        reached_structural_entry_by_bias(struct_df_llm),
        mae_mfe_raw_by_market_bias(struct_df_llm),
        mae_mfe_past_entry_by_market_bias(struct_df_llm),
        mae_mfe_past_entry_by_prob_tercile(struct_df_llm),
        mark_price_distance_to_entry_by_bias(struct_df_llm),
    ],
    source_functions=[_build_struct_df, reached_structural_entry_by_bias, mae_mfe_raw_by_market_bias, mae_mfe_past_entry_by_market_bias, mae_mfe_past_entry_by_prob_tercile, mark_price_distance_to_entry_by_bias],
))

llm_sections.append(MarkdownSection(
    title="Sección 6 — R-múltiplos, mae_R como señal líder, y timing",
    narrative=(
        "Fase 1: 1R = |structural_entry_price - structural_invalidation|; mae_R/mfe_R = "
        "mae_past_entry/mfe_past_entry normalizados por 1R; se compara el potencial "
        "estructural (mfe_R, todos los análisis) contra r_multiple realizado (tactical_audit, "
        "solo order_filled=True con r_multiple no nulo). Fase 2: mae_R cruzado contra "
        "specific_bias_compliance y false_regime_rate. Fase 3: usando mark_price (no "
        "structural_mfe), ¿ya se cruzó structural_entry_price al momento del review?"
    ),
    tables=[
        one_r_distribution(struct_df_llm),
        mae_mfe_R_by_market_bias(struct_df_llm),
        structural_potential_vs_realized(struct_df_llm, tactical_r_df_llm),
        mae_R_by_specific_bias_compliance(audit_df_llm, struct_df_llm),
        win_rate_by_mae_R_tercile(audit_df_llm, struct_df_llm),
        mae_R_by_false_regime_rate(audit_df_llm, struct_df_llm),
        dict_to_metrics_frame(entry_timing_summary(struct_df_llm)),
        entry_timing_by_market_bias(struct_df_llm),
    ],
    source_functions=[one_r_distribution, mae_mfe_R_by_market_bias, structural_potential_vs_realized, mae_R_by_specific_bias_compliance, win_rate_by_mae_R_tercile, mae_R_by_false_regime_rate, entry_timing_summary, entry_timing_by_market_bias],
))

llm_sections.append(MarkdownSection(
    title="Sección 7 — Validación de P0-P4 (core/edge_analysis.py) y métrica 'Rescate'",
    narrative=(
        "Reusa funciones de producción sin modificarlas: analisis_edge (Net Profit/Profit "
        "Factor/Win Rate/Participación/Correlación por P vs. Market_Outcome), "
        "analizar_correlaciones (Spearman entre Ps), profit_factor_by_edge_bins (win rate "
        "por magnitud de |calc_edge|, terciles — asigna Market_Outcome=0 a Choppy). "
        "'Rescate' (métrica nueva): de los fallos del sistema (market_bias Bullish/Bearish, "
        "specific_bias_compliance=='Invalid'), ¿qué Px individual tenía sign(Px)==sign(Market_Outcome)?"
    ),
    tables=[
        analisis_edge(px_df_llm),
        analizar_correlaciones(px_df_llm),
        profit_factor_by_edge_bins(px_df_llm),
        rescue_metric_by_parameter(px_df_llm),
        rescued_cases_detail(px_df_llm),
    ],
    source_functions=[_build_px_df, analisis_edge, analizar_correlaciones, profit_factor_by_edge_bins, get_market_outcome, rescue_metric_by_parameter, rescued_cases_detail],
))

llm_sections.append(MarkdownSection(
    title="Sección 8 — Consenso entre Ps, P0 por bias_a, y simulación de re-ponderación",
    narrative=(
        "consensus_count = cuántos de los 5 Ps (score!=0) coinciden en signo con sign(calc_edge), "
        "cruzado contra is_valid. p0_correct compara sign(p0_score) contra Market_Outcome, "
        "agrupado por bias_a. edge_weight_simulation recalcula el signo de calc_edge con "
        "distintos juegos de pesos P0-P4 (los actuales: core/math_engine.py::calculate_edge_score) "
        "y lo compara contra Market_Outcome, solo en setups Bullish/Bearish resueltos."
    ),
    tables=[
        win_rate_by_consensus_count(px_df_llm),
        p0_accuracy_by_bias_a(px_df_llm, efficiency_df),
        edge_weight_simulation(px_df_llm),
    ],
    source_functions=[win_rate_by_consensus_count, p0_accuracy_by_bias_a, edge_weight_simulation],
))

llm_sections.append(MarkdownSection(
    title="Sección 9 — Temporalidad de la thesis de P0 (regex sobre analysis_layer.thesis)",
    narrative=(
        "El texto de thesis (layer_name=='P0') sigue el patrón '{temporalidad}-B{L|R}' "
        "(L=Bullish, R=Bearish) para 4 temporalidades (1DT/12H/4H/1H). Las variantes de "
        "abreviatura encontradas en el texto real (ej. 12HT/12H/12T) se normalizan a las "
        "4 etiquetas canónicas antes de comparar convergencia con p0_score y win rate contra "
        "Market_Outcome."
    ),
    tables=[
        tf_coverage(tf_df_llm),
        tf_convergence_with_p0(tf_df_llm),
        tf_win_rate_vs_market_outcome(tf_full_llm),
    ],
    source_functions=[_build_tf_df, tf_coverage, tf_convergence_with_p0, tf_win_rate_vs_market_outcome],
))

llm_sections.append(MarkdownSection(
    title="Sección 10 — Profundización en la temporalidad 1H",
    narrative=(
        "htf_majority = signo de la suma de tf_1DT/tf_12H/tf_4H (cuando hay al menos una). "
        "Se compara tf_1H contra esa mayoría, contra bias_a, y cara a cara contra el propio "
        "p0_score (sign(p0_score)), todo evaluado contra Market_Outcome."
    ),
    tables=[
        tf_1h_vs_htf_majority(tf_full_llm),
        tf_1h_win_rate_by_bias_a(tf_full_llm, efficiency_df),
        tf_1h_vs_p0_head_to_head(tf_full_llm),
    ],
    source_functions=[tf_1h_vs_htf_majority, tf_1h_win_rate_by_bias_a, tf_1h_vs_p0_head_to_head],
))

llm_sections.append(MarkdownSection(
    title="Sección 11 — Llegada a structural_entry_price y 'sweet spot' de mae_R",
    narrative=(
        "reached_entry (igual definición que Sección 5①) desglosado en Llega/No_Llega por "
        "market_bias y por tercil de primary_prob. mae_R_sweet_spot corta mae_R en 7 tramos "
        "finos (bin_edges fijos, no cuantiles) y mide win rate por tramo, solo en setups "
        "resueltos."
    ),
    tables=[
        reach_entry_by_market_bias(struct_df_llm),
        reach_entry_by_prob_tercile(struct_df_llm),
        mae_R_sweet_spot(struct_df_llm, efficiency_df),
    ],
    source_functions=[reach_entry_by_market_bias, reach_entry_by_prob_tercile, mae_R_sweet_spot],
))

llm_sections.append(MarkdownSection(
    title="Sección 12 — Tabla de referencia: MAE/MFE esperado por bias, ICD y probabilidad",
    narrative=(
        "structural_mae/structural_mfe crudos (precio) y mae_R/mfe_R (normalizados) agregados "
        "por market_bias, por tercil de |calc_edge|, y por tercil de primary_prob — media y "
        "mediana. Los precios crudos no son comparables entre fechas (XAUUSD se movió de ~4000 "
        "a ~4600 en el período); mae_R/mfe_R sí lo son."
    ),
    tables=[
        structural_reference_table_by_market_bias(struct_df_llm),
        structural_reference_table_by_edge_magnitude(struct_df_llm),
        structural_reference_table_by_prob_tercile(struct_df_llm),
    ],
    source_functions=[structural_reference_table_by_market_bias, structural_reference_table_by_edge_magnitude, structural_reference_table_by_prob_tercile],
))

chase_df_llm = build_chase_df(struct_df_llm[struct_df_llm['order_filled'] == True], psych_df_llm)

llm_sections.append(MarkdownSection(
    title="Sección 14 — Frecuencia de alcance/cruce de cada nivel de precio, por market_bias x ICD",
    narrative=(
        "Cuatro chequeos direccionales (>=/<= segun theoretical_dir), agrupados por market_bias "
        "x tercil de |calc_edge| (edge_mag_tercile): (1) mark_price cruzo structural_entry_price; "
        "(2) structural_mae alcanzo structural_entry_price; (3) structural_mfe alcanzo "
        "edge_validation_price (TP); (4) entry_price real (solo order_filled==True) vs "
        "structural_entry_price -- si el fill quedo peor (Entro_Peor_%) que el pullback ideal. "
        "entry_price puede estar poblado con order_filled=False (intento no llenado, o legacy) -- "
        "(4) filtra explicitamente por order_filled==True, no por entry_price.notna(). "
        "(4.B) mismo calculo sobre all_entries_df -- TODOS los registros con entry_price cargado "
        "(incluye order_filled=False), excluyendo placeholders legacy (entry_price==closing_price "
        "y entry_time==exit_time simultaneos) -- mayor N."
    ),
    tables=[
        mark_crossed_structural_entry_by_bias_icd(struct_df_llm),
        mae_reached_structural_entry_by_bias_icd(struct_df_llm),
        mfe_reached_validation_price_by_bias_icd(struct_df_llm),
        entry_price_vs_structural_by_bias_icd(struct_df_llm),
        entry_price_vs_structural_all_by_bias_icd(all_entries_df_llm),
    ],
    source_functions=[
        _reach_rate_by_bias_icd, mark_crossed_structural_entry_by_bias_icd,
        mae_reached_structural_entry_by_bias_icd, mfe_reached_validation_price_by_bias_icd,
        entry_price_vs_structural_by_bias_icd, build_all_entries_df, entry_price_vs_structural_all_by_bias_icd,
    ],
))

llm_sections.append(MarkdownSection(
    title="Sección 15 — Senal de miedo/FOMO al entrar (entry_chase_R vs estado psicologico)",
    narrative=(
        "entry_chase_R = cuanto peor (positivo) o mejor (negativo/cero) fue entry_price real vs "
        "structural_entry_price, normalizado por one_r, direccional (Long: (entry-structural)/one_r; "
        "Short: (structural-entry)/one_r). Restringido a order_filled==True. Cruzado contra "
        "anxiety_level, impatience_level y primary_emotion (tactical_audit) para ver si mayor "
        "ansiedad/impaciencia se asocia a entradas perseguidas."
    ),
    tables=[
        entry_chase_by_anxiety(chase_df_llm),
        entry_chase_by_impatience(chase_df_llm),
        entry_chase_by_emotion(chase_df_llm),
    ],
    source_functions=[build_chase_df, entry_chase_by_anxiety, entry_chase_by_impatience, entry_chase_by_emotion],
))

chase_df_all_llm = build_chase_df(all_entries_df_llm, psych_df_llm)

llm_sections.append(MarkdownSection(
    title="Sección 15.B - entry_chase_R sobre TODOS los registros (order_filled True y False)",
    narrative=(
        "Mismo entry_chase_R de la Seccion 15, sobre all_entries_df (incluye order_filled=False) -- "
        "mayor N. entry_price es la entrada de precision que el trader busca deliberadamente para "
        "superar structural_entry_price, se carga independientemente de si la orden se llego a "
        "llenar; restringir a order_filled==True descartaba justo los intentos donde apunto a un "
        "nivel y el mercado no se lo dio."
    ),
    tables=[
        entry_chase_by_anxiety(chase_df_all_llm),
        entry_chase_by_impatience(chase_df_all_llm),
        entry_chase_by_emotion(chase_df_all_llm),
    ],
    source_functions=[build_all_entries_df, build_chase_df, entry_chase_by_anxiety, entry_chase_by_impatience, entry_chase_by_emotion],
))

sweep_v1_llm = sweep_reach_rate(struct_df_llm, 'mark_entry')
sweep_v2_llm = sweep_reach_rate(struct_df_llm[struct_df_llm['order_filled'] == True], 'fractal_entry')
sweep_v2_all_llm = sweep_reach_rate(all_entries_df_llm, 'fractal_entry')

llm_sections.append(MarkdownSection(
    title="Sección 16 — Entrada hipotetica a X% de distancia: barrido y tasa de alcance (dos versiones)",
    narrative=(
        "x en [0.0..1.0] paso 0.1. mark_entry interpola mark_price (x=0) <-> structural_entry_price "
        "(x=1) -- toda la muestra. fractal_entry interpola entry_price real (x=0, solo "
        "order_filled==True) <-> structural_entry_price (x=1). reached = structural_mae alcanzo esa "
        "entrada hipotetica (misma logica direccional que Seccion 14/2). En x=1.0 ambas versiones "
        "convergen a structural_entry_price -- debe coincidir con la Seccion 14(2)."
    ),
    tables=[sweep_v1_llm, sweep_v2_llm],
    source_functions=[_hyp_entry_price, _reached_hyp, sweep_reach_rate],
))

llm_sections.append(MarkdownSection(
    title="Sección 16.B - barrido fractal_entry_x sobre TODOS los registros (order_filled True y False)",
    narrative=(
        "Mismo barrido y misma logica de alcance de la Seccion 16 (fractal_entry), sobre "
        "all_entries_df -- mayor N. No hace falta repetir mark_entry, ya usaba toda la muestra."
    ),
    tables=[sweep_v2_all_llm],
    source_functions=[build_all_entries_df, _hyp_entry_price, _reached_hyp, sweep_reach_rate],
))

backtest_summary_llm = build_backtest_summary([
    (struct_df_llm, 'mark_entry'),
    (struct_df_llm[struct_df_llm['order_filled'] == True], 'fractal_entry'),
])
best_x_summary_llm = best_x_by_segment(backtest_summary_llm)

backtest_summary_all_llm = build_backtest_summary([(all_entries_df_llm, 'fractal_entry')])
best_x_summary_all_llm = best_x_by_segment(backtest_summary_all_llm)

llm_sections.append(MarkdownSection(
    title="Sección 17 — Backtest de resultado hipotetico: ratios 1:2 y 1:2.2",
    narrative=(
        "Para cada entrada activada (Seccion 16), TP=edge_validation_price, "
        "SL = entry -/+ |TP-entry|/ratio (Long/Short). Clasificacion (regla confirmada por el "
        "usuario, simplificacion conocida por falta de orden temporal real en structural_mae/mfe): "
        "structural_mae rompe SL -> Perdido; si no y structural_mfe alcanza TP -> Ganado; si no, "
        "Inconcluso; si ambas se cumplen, prevalece Perdido (mas conservador). Tabla larga por "
        "version x x x ratio x market_bias x edge_mag_tercile, mas un resumen con el x optimo "
        "(mayor Win_Rate_%, N_Activado>=5) por segmento -- 'entrada optima aproximada', no perfecta."
    ),
    tables=[backtest_summary_llm, best_x_summary_llm],
    source_functions=[classify_hypothetical_trade, build_backtest_summary, best_x_by_segment],
))

llm_sections.append(MarkdownSection(
    title="Sección 17.B - backtest hipotetico sobre TODOS los registros (order_filled True y False)",
    narrative=(
        "Misma regla de clasificacion de la Seccion 17, corrida solo para fractal_entry sobre "
        "all_entries_df (mayor N que la Seccion 17 original). No hace falta repetir mark_entry -- "
        "ya cubria toda la muestra. Misma advertencia metodologica: proxy sin orden temporal "
        "confirmado, no un veredicto definitivo."
    ),
    tables=[backtest_summary_all_llm, best_x_summary_all_llm],
    source_functions=[build_all_entries_df, classify_hypothetical_trade, build_backtest_summary, best_x_by_segment],
))

llm_trade_columns = [
    'id', 'market_bias', 'bias_a', 'real_bias_b', 'structural_resolution',
    'specific_bias_compliance', 'false_regime_rate', 'resolution_type',
    'calc_edge', 'theoretical_dir', 'primary_prob', 'long_prob', 'short_prob', 'no_trade_prob',
    'p0_score', 'p1_score', 'p2_score', 'p3_score', 'p4_score', 'consensus_count',
    'edge_validation_price', 'structural_invalidation', 'mark_price',
    'structural_mae', 'structural_mfe', 'structural_entry_price', 'one_r', 'mae_R', 'mfe_R',
    'order_filled', 'r_multiple', 'entry_price',
    'anxiety_level', 'impatience_level', 'primary_emotion',
]

llm_trade_df = (
    struct_df_llm
    .merge(px_df_llm[['id', 'p0_score', 'p1_score', 'p2_score', 'p3_score', 'p4_score', 'consensus_count']], on='id', how='left')
    .merge(efficiency_df[['id', 'bias_a', 'real_bias_b', 'structural_resolution', 'specific_bias_compliance', 'false_regime_rate', 'resolution_type']], on='id', how='left')
    .merge(tactical_r_df_llm, on='id', how='left')
    .merge(psych_df_llm, on='id', how='left')
)

llm_trade_records = build_trade_records(llm_trade_df, columns=llm_trade_columns)

llm_markdown = render_llm_markdown_report(
    llm_context,
    llm_sections,
    llm_trade_records,
    title="Blast Master — Reporte para análisis por LLM (Edge Estructural, XAUUSD)",
)
llm_report_path = save_markdown_report(llm_markdown, "custom_analysis_llm_report.md")
print(f"Reporte LLM generado en: {llm_report_path.resolve()}")
print(f"Análisis incluidos en el apéndice JSON: {len(llm_trade_records)}")

Reporte LLM generado en: /home/jorgecg/projects/trading/blast_master/jupyter/custom_analysis_llm_report.md
Análisis incluidos en el apéndice JSON: 70
